# LightGBM and XGBoost Performance Under Multiple Participant Split Ratios

## Objective

This experiment evaluates the performance and stability of LightGBM and XGBoost under different participant independent train, test, and validation split ratios using the Max EEG feature dataset.

The following five participant split configurations are evaluated:

| Experiment | Training | Testing | Validation / CV Holdout |
|---|---:|---:|---:|
| Split 1 | 70% | 20% | 10% |
| Split 2 | 60% | 30% | 10% |
| Split 3 | 75% | 15% | 10% |
| Split 4 | 65% | 25% | 10% |
| Split 5 | 50% | 40% | 10% |

The splitting procedure is participant independent. Therefore, data belonging to the same participant cannot appear in training, validation, and testing sets simultaneously.

The same deterministic random seed is used for every experiment. The 10% validation participant group is kept fixed across all five split configurations to provide a consistent model selection set while the relative proportions of training and testing participants are changed.

## Model Development

Two machine learning algorithms are evaluated:

**LightGBM**

LightGBM uses multiclass gradient boosted decision trees. Hyperparameters are optimized using Optuna with five fold GroupKFold cross validation on the training participants. Balanced sample weights are applied during model training. After hyperparameter optimization, five independent LightGBM runs are performed using different random seeds. The run obtaining the highest validation accuracy is selected for final testing.

**XGBoost**

XGBoost uses the histogram based tree construction method. Hyperparameters including the number of estimators, tree depth, learning rate, subsampling ratio, feature sampling ratio, and regularization parameters are optimized using Optuna with five fold GroupKFold cross validation. Balanced sample weights are used during training. The tuned XGBoost model is subsequently evaluated on the validation and testing participants.

## Evaluation

Each model is evaluated using:

Accuracy, weighted precision, weighted recall, weighted F1 score, macro F1 score, Matthews Correlation Coefficient, Cohen's Kappa, multiclass ROC AUC, classification report, and confusion matrix.

All five split configurations are evaluated independently for LightGBM and XGBoost. This produces ten final model and split results.

## Best Model Selection

Two summaries are produced.

The **Best Single Configuration** represents the individual algorithm and split configuration obtaining the highest Test Accuracy.

The **Best Overall Model** is determined using the mean Test Accuracy obtained by each algorithm across all five split configurations.

The algorithm with the highest mean Test Accuracy is printed as:

`BEST OVERALL MODEL`

This approach allows the comparison to consider not only the strongest individual result but also the consistency of each algorithm across different amounts of training and testing data.

## Restart and Power Interruption Support

The experiment supports continuation after a power interruption.

Optuna studies are stored in SQLite databases. Completed LightGBM final runs are stored independently, XGBoost final models are checkpointed, and every completed algorithm and split result is recorded. When the script is executed again, completed components are automatically detected and reused.

Therefore, completed experiments do not need to be repeated after a restart.

## Output

LightGBM results are stored in:

`/home/rakib/Desktop/ml_apply/Final_Model_Test/LightGBM_Multiple_Split_Results`

XGBoost results are stored in:

`/home/rakib/Desktop/ml_apply/Final_Model_Test/XGBoost_Multiple_Split_Results`

The combined comparison table is stored as:

`XGBoost_LightGBM_Multiple_Split_Results.csv`

The overall algorithm ranking is stored as:

`Overall_Algorithm_Ranking.csv`

The selected model information is stored as:

`Best_Model_Summary.json`

and

`Best_Model_Name.txt`

A graphical comparison of Test Accuracy across the five split ratios is stored as:

`Test_Accuracy_Comparison.png`

In [1]:
# =============================================================================
# MULTIPLE SPLIT COMPARISON
# MAX DATASET
#
# ALGORITHMS
#
# LightGBM
# XGBoost
#
# SPLIT CONFIGURATIONS
#
# Train 70 percent, Test 20 percent, CV Holdout 10 percent
# Train 60 percent, Test 30 percent, CV Holdout 10 percent
# Train 75 percent, Test 15 percent, CV Holdout 10 percent
# Train 65 percent, Test 25 percent, CV Holdout 10 percent
# Train 50 percent, Test 40 percent, CV Holdout 10 percent
#
# Participant independent splitting
#
# The same 10 percent validation participants are retained for all
# five experiments to make the split comparison more controlled.
#
# Within the training participants:
#
# 5 fold GroupKFold
# Optuna tuning
#
# LightGBM
#
# 15 Optuna trials
# 5 final stochastic runs
# Validation based run selection
# CPU execution
#
# XGBoost
#
# 12 Optuna trials
# CPU hist
#
# Final test evaluation for every algorithm and split
#
# POWER CUT SUPPORT
#
# Persistent Optuna SQLite databases
# LightGBM run checkpoints
# XGBoost final model checkpoint
# Completed split result checkpoints
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import gc
import json
import time
import random
import hashlib
import warnings
import traceback

from pathlib import Path

import joblib
import psutil
import optuna

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb

from xgboost import XGBClassifier

from sklearn.model_selection import GroupKFold

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.utils.class_weight import (
    compute_sample_weight
)


# =============================================================================
# WARNINGS
# =============================================================================

warnings.filterwarnings(
    "ignore"
)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


# =============================================================================
# BASIC CONFIGURATION
# =============================================================================

SEED = 42

GROUP_K_FOLDS = 5

LGB_OPTUNA_TRIALS = 15

XGB_OPTUNA_TRIALS = 12

LGB_RUNS = 5


# =============================================================================
# LIGHTGBM
# =============================================================================

LGB_OPTUNA_MAX_ROUNDS = 800

LGB_OPTUNA_EARLY_STOPPING = 30

LGB_FINAL_MAX_ROUNDS = 1500

LGB_FINAL_EARLY_STOPPING = 40


# =============================================================================
# CPU PARALLELISM
#
# Limited to reduce RAM pressure.
# =============================================================================

N_JOBS = max(
    1,
    min(
        8,
        os.cpu_count() or 1
    )
)


# =============================================================================
# INPUT
# =============================================================================

FILE_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "Final_Model_Test/"
    "Cleaned_oN_Selected_Max.csv"
)


# =============================================================================
# OUTPUT ROOT
# =============================================================================

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "Final_Model_Test"
)


# =============================================================================
# ALGORITHM OUTPUT FOLDERS
# =============================================================================

LGB_ROOT = (
    OUTPUT_ROOT
    / "LightGBM_Multiple_Split_Results"
)


XGB_ROOT = (
    OUTPUT_ROOT
    / "XGBoost_Multiple_Split_Results"
)


LGB_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


XGB_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


if not FILE_PATH.exists():

    raise FileNotFoundError(
        f"Input file not found: "
        f"{FILE_PATH}"
    )


# =============================================================================
# SPLIT CONFIGURATIONS
# =============================================================================

SPLIT_CONFIGS = [

    {
        "Split_ID":
            "Split_01",

        "Split_Name":
            "Train70_Test20_CV10",

        "Train_Ratio":
            0.70,

        "Test_Ratio":
            0.20,

        "Validation_Ratio":
            0.10
    },

    {
        "Split_ID":
            "Split_02",

        "Split_Name":
            "Train60_Test30_CV10",

        "Train_Ratio":
            0.60,

        "Test_Ratio":
            0.30,

        "Validation_Ratio":
            0.10
    },

    {
        "Split_ID":
            "Split_03",

        "Split_Name":
            "Train75_Test15_CV10",

        "Train_Ratio":
            0.75,

        "Test_Ratio":
            0.15,

        "Validation_Ratio":
            0.10
    },

    {
        "Split_ID":
            "Split_04",

        "Split_Name":
            "Train65_Test25_CV10",

        "Train_Ratio":
            0.65,

        "Test_Ratio":
            0.25,

        "Validation_Ratio":
            0.10
    },

    {
        "Split_ID":
            "Split_05",

        "Split_Name":
            "Train50_Test40_CV10",

        "Train_Ratio":
            0.50,

        "Test_Ratio":
            0.40,

        "Validation_Ratio":
            0.10
    }
]


# =============================================================================
# RANDOM SEED
# =============================================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)


# =============================================================================
# PROCESS MEMORY
# =============================================================================

PROC = psutil.Process(
    os.getpid()
)


def mem_mb():

    return (
        PROC.memory_info().rss
        / 1024
        / 1024
    )


# =============================================================================
# ATOMIC JSON SAVE
# =============================================================================

def atomic_json(
    obj,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name + ".tmp"
    )


    with open(
        temporary,
        "w",
        encoding="utf8"
    ) as file:

        json.dump(
            obj,
            file,
            indent=4
        )


    os.replace(
        temporary,
        path
    )


# =============================================================================
# ATOMIC CSV SAVE
# =============================================================================

def atomic_csv(
    dataframe,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name + ".tmp"
    )


    dataframe.to_csv(
        temporary,
        index=False
    )


    os.replace(
        temporary,
        path
    )


# =============================================================================
# TIMER
# =============================================================================

TOTAL_START = time.time()


# =============================================================================
# CONFIGURATION INFORMATION
# =============================================================================

print("=" * 90)

print(
    "LIGHTGBM AND XGBOOST MULTIPLE SPLIT EXPERIMENT"
)

print("=" * 90)


print(
    "Input file:"
)

print(
    FILE_PATH
)


print()

print(
    "LightGBM device : CPU"
)

print(
    "XGBoost device  : CPU hist"
)

print(
    "CPU workers     :",
    N_JOBS
)

print(
    "GroupKFold      :",
    GROUP_K_FOLDS
)

print(
    "LightGBM trials :",
    LGB_OPTUNA_TRIALS
)

print(
    "XGBoost trials  :",
    XGB_OPTUNA_TRIALS
)

print(
    "LightGBM runs   :",
    LGB_RUNS
)

print(
    "Data reduction  : NONE"
)

print("=" * 90)


# =============================================================================
# LOAD DATASET
# =============================================================================

print()

print("=" * 90)

print(
    "LOADING MAX DATASET"
)

print("=" * 90)


load_start = time.time()


df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)


print(
    "Raw dataset shape:",
    df.shape
)


print(
    "RAM after loading:",
    f"{mem_mb():.2f} MB"
)


# =============================================================================
# COLUMNS
# =============================================================================

metadata_cols = [

    "PersonSrial",

    "Window",

    "Window_Start",

    "Window_End",

    "Window_Samples"
]


target_col = "Class"


required_columns = [

    "PersonSrial",

    target_col
]


for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"Required column missing: "
            f"{column}"
        )


feature_cols = [

    column

    for column in df.columns

    if column not in metadata_cols

    and column != target_col
]


if len(
    feature_cols
) == 0:

    raise ValueError(
        "No feature columns detected."
    )


print(
    "Feature count:",
    len(
        feature_cols
    )
)


# =============================================================================
# FEATURE MATRIX
# =============================================================================

X_all = (

    df[
        feature_cols
    ]

    .apply(
        pd.to_numeric,
        errors="coerce"
    )

    .to_numpy(
        dtype=np.float32
    )
)


invalid_count = int(
    (
        ~np.isfinite(
            X_all
        )
    ).sum()
)


print(
    "Invalid feature cells:",
    invalid_count
)


if invalid_count > 0:

    raise ValueError(
        "Feature matrix contains NaN or infinite values."
    )


# =============================================================================
# LABELS
# =============================================================================

label_encoder = LabelEncoder()


y_all = label_encoder.fit_transform(

    df[
        target_col
    ].astype(
        str
    )

).astype(
    np.int32
)


class_names = list(
    label_encoder.classes_
)


NUM_CLASSES = len(
    class_names
)


groups_all = (

    df[
        "PersonSrial"
    ]
    .to_numpy()
)


print()

print(
    "Feature matrix:",
    X_all.shape
)

print(
    "Classes:",
    class_names
)

print(
    "Number of classes:",
    NUM_CLASSES
)

print(
    "Participants:",
    len(
        np.unique(
            groups_all
        )
    )
)


# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

class_distribution = []


print()

print(
    "CLASS DISTRIBUTION"
)


for class_index, class_name in enumerate(
    class_names
):

    count = int(
        np.sum(
            y_all
            == class_index
        )
    )


    percentage = (
        count
        / len(
            y_all
        )
        * 100.0
    )


    print(
        f"{class_name}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )


    class_distribution.append(
        {

            "Class":
                class_name,

            "Count":
                count,

            "Percentage":
                percentage
        }
    )


pd.DataFrame(
    class_distribution
).to_csv(
    OUTPUT_ROOT
    / "class_distribution.csv",
    index=False
)


# =============================================================================
# DATASET SIGNATURE
#
# Prevents checkpoints from another version of the CSV being reused.
# =============================================================================

file_stat = FILE_PATH.stat()


dataset_signature_payload = {

    "Path":
        str(
            FILE_PATH
        ),

    "File_Size":
        int(
            file_stat.st_size
        ),

    "Modified_NS":
        int(
            file_stat.st_mtime_ns
        ),

    "Rows":
        int(
            len(
                y_all
            )
        ),

    "Feature_Columns":
        feature_cols,

    "Classes":
        class_names
}


DATASET_SIGNATURE = hashlib.sha256(
    json.dumps(
        dataset_signature_payload,
        sort_keys=True
    ).encode(
        "utf8"
    )
).hexdigest()[
    :16
]


print()

print(
    "Dataset signature:",
    DATASET_SIGNATURE
)


print(
    "Loading time:",
    f"{time.time() - load_start:.2f}",
    "seconds"
)


# =============================================================================
# RELEASE DATAFRAME
# =============================================================================

del df

gc.collect()


# =============================================================================
# PARTICIPANT ORDER
#
# One deterministic participant order is used for every split.
#
# Validation participants are fixed across all five experiments.
# =============================================================================

persons = np.unique(
    groups_all
)


participant_rng = np.random.RandomState(
    SEED
)


participant_rng.shuffle(
    persons
)


NUMBER_PERSONS = len(
    persons
)


NUMBER_VALIDATION_PERSONS = max(
    1,
    int(
        round(
            NUMBER_PERSONS
            * 0.10
        )
    )
)


FIXED_VALIDATION_PERSONS = persons[
    :NUMBER_VALIDATION_PERSONS
]


REMAINING_PERSONS = persons[
    NUMBER_VALIDATION_PERSONS:
]


print()

print("=" * 90)

print(
    "FIXED CROSS VALIDATION HOLDOUT PARTICIPANTS"
)

print("=" * 90)


print(
    "Total participants:",
    NUMBER_PERSONS
)

print(
    "Fixed validation participants:",
    len(
        FIXED_VALIDATION_PERSONS
    )
)


# =============================================================================
# METRIC FUNCTION
# =============================================================================

def calculate_metrics(
    y_true,
    prediction,
    probability
):

    result = {

        "accuracy":

            float(
                accuracy_score(
                    y_true,
                    prediction
                )
            ),

        "precision_weighted":

            float(
                precision_score(
                    y_true,
                    prediction,
                    average="weighted",
                    zero_division=0
                )
            ),

        "recall_weighted":

            float(
                recall_score(
                    y_true,
                    prediction,
                    average="weighted",
                    zero_division=0
                )
            ),

        "f1_weighted":

            float(
                f1_score(
                    y_true,
                    prediction,
                    average="weighted",
                    zero_division=0
                )
            ),

        "f1_macro":

            float(
                f1_score(
                    y_true,
                    prediction,
                    average="macro",
                    zero_division=0
                )
            ),

        "cohen_kappa":

            float(
                cohen_kappa_score(
                    y_true,
                    prediction
                )
            ),

        "mcc":

            float(
                matthews_corrcoef(
                    y_true,
                    prediction
                )
            )
    }


    try:

        result[
            "roc_auc_weighted"
        ] = float(
            roc_auc_score(
                y_true,
                probability,
                multi_class="ovr",
                average="weighted"
            )
        )


    except Exception:

        result[
            "roc_auc_weighted"
        ] = None


    return result


# =============================================================================
# CONFUSION MATRIX
# =============================================================================

def save_confusion_matrix(
    model_name,
    output_directory,
    y_true,
    prediction
):

    matrix = confusion_matrix(
        y_true,
        prediction,
        labels=np.arange(
            NUM_CLASSES
        )
    )


    pd.DataFrame(
        matrix,
        index=
            class_names,
        columns=
            class_names
    ).to_csv(
        output_directory
        / "confusion_matrix.csv"
    )


    fig, ax = plt.subplots(
        figsize=(
            8,
            6
        )
    )


    image = ax.imshow(
        matrix
    )


    ax.set_xticks(
        np.arange(
            NUM_CLASSES
        )
    )


    ax.set_yticks(
        np.arange(
            NUM_CLASSES
        )
    )


    ax.set_xticklabels(
        class_names,
        rotation=45,
        ha="right"
    )


    ax.set_yticklabels(
        class_names
    )


    ax.set_xlabel(
        "Predicted Class"
    )


    ax.set_ylabel(
        "True Class"
    )


    ax.set_title(
        f"{model_name} Confusion Matrix"
    )


    for row in range(
        NUM_CLASSES
    ):

        for column in range(
            NUM_CLASSES
        ):

            ax.text(
                column,
                row,
                str(
                    matrix[
                        row,
                        column
                    ]
                ),
                ha="center",
                va="center"
            )


    fig.colorbar(
        image,
        ax=ax
    )


    fig.tight_layout()


    fig.savefig(
        output_directory
        / "confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # =========================================================================
    # NORMALIZED CONFUSION MATRIX
    # =========================================================================

    row_sums = matrix.sum(
        axis=1,
        keepdims=True
    )


    normalized = np.divide(
        matrix,
        row_sums,
        out=np.zeros_like(
            matrix,
            dtype=np.float64
        ),
        where=(
            row_sums
            != 0
        )
    )


    pd.DataFrame(
        normalized,
        index=
            class_names,
        columns=
            class_names
    ).to_csv(
        output_directory
        / "normalized_confusion_matrix.csv"
    )


    fig, ax = plt.subplots(
        figsize=(
            8,
            6
        )
    )


    image = ax.imshow(
        normalized
    )


    ax.set_xticks(
        np.arange(
            NUM_CLASSES
        )
    )


    ax.set_yticks(
        np.arange(
            NUM_CLASSES
        )
    )


    ax.set_xticklabels(
        class_names,
        rotation=45,
        ha="right"
    )


    ax.set_yticklabels(
        class_names
    )


    ax.set_xlabel(
        "Predicted Class"
    )


    ax.set_ylabel(
        "True Class"
    )


    ax.set_title(
        f"{model_name} Normalized Confusion Matrix"
    )


    for row in range(
        NUM_CLASSES
    ):

        for column in range(
            NUM_CLASSES
        ):

            ax.text(
                column,
                row,
                f"{normalized[row, column]:.2f}",
                ha="center",
                va="center"
            )


    fig.colorbar(
        image,
        ax=ax
    )


    fig.tight_layout()


    fig.savefig(
        output_directory
        / "normalized_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# =============================================================================
# REPORT SAVER
# =============================================================================

def save_test_outputs(
    model_name,
    output_directory,
    y_test,
    g_test,
    prediction,
    probability,
    metrics
):

    report_text = classification_report(
        y_test,
        prediction,
        labels=np.arange(
            NUM_CLASSES
        ),
        target_names=
            class_names,
        digits=4,
        zero_division=0
    )


    with open(
        output_directory
        / "classification_report.txt",
        "w",
        encoding="utf8"
    ) as file:

        file.write(
            report_text
        )


    report_table = pd.DataFrame(
        classification_report(
            y_test,
            prediction,
            labels=np.arange(
                NUM_CLASSES
            ),
            target_names=
                class_names,
            output_dict=True,
            zero_division=0
        )
    ).transpose()


    report_table.to_csv(
        output_directory
        / "classification_report.csv"
    )


    save_confusion_matrix(
        model_name,
        output_directory,
        y_test,
        prediction
    )


    prediction_table = pd.DataFrame(
        {

            "PersonSrial":
                g_test,

            "True_Class":
                label_encoder.inverse_transform(
                    y_test
                ),

            "Predicted_Class":
                label_encoder.inverse_transform(
                    prediction.astype(
                        np.int32
                    )
                )
        }
    )


    for class_index, class_name in enumerate(
        class_names
    ):

        prediction_table[
            f"Probability_{class_name}"
        ] = probability[
            :,
            class_index
        ]


    prediction_table.to_csv(
        output_directory
        / "test_predictions.csv",
        index=False
    )


    atomic_json(
        metrics,
        output_directory
        / "final_metrics.json"
    )


# =============================================================================
# PARTICIPANT SPLIT BUILDER
# =============================================================================

def build_split(
    split_config
):

    train_ratio = split_config[
        "Train_Ratio"
    ]


    test_ratio = split_config[
        "Test_Ratio"
    ]


    validation_ratio = split_config[
        "Validation_Ratio"
    ]


    if not np.isclose(
        train_ratio
        + test_ratio
        + validation_ratio,
        1.0
    ):

        raise ValueError(
            "Split ratios must sum to 1.0."
        )


    number_test = max(
        1,
        int(
            round(
                NUMBER_PERSONS
                * test_ratio
            )
        )
    )


    # =========================================================================
    # All remaining non validation participants become training participants.
    #
    # This avoids losing a participant because of integer rounding.
    # =========================================================================

    number_train = (
        NUMBER_PERSONS
        - NUMBER_VALIDATION_PERSONS
        - number_test
    )


    test_persons = REMAINING_PERSONS[
        :number_test
    ]


    train_persons = REMAINING_PERSONS[
        number_test:
        number_test
        + number_train
    ]


    validation_persons = (
        FIXED_VALIDATION_PERSONS.copy()
    )


    # =========================================================================
    # VERIFY INDEPENDENCE
    # =========================================================================

    assert (
        np.intersect1d(
            train_persons,
            test_persons
        ).size == 0
    )


    assert (
        np.intersect1d(
            train_persons,
            validation_persons
        ).size == 0
    )


    assert (
        np.intersect1d(
            test_persons,
            validation_persons
        ).size == 0
    )


    train_mask = np.isin(
        groups_all,
        train_persons
    )


    validation_mask = np.isin(
        groups_all,
        validation_persons
    )


    test_mask = np.isin(
        groups_all,
        test_persons
    )


    X_train_raw = np.ascontiguousarray(
        X_all[
            train_mask
        ],
        dtype=np.float32
    )


    y_train = np.ascontiguousarray(
        y_all[
            train_mask
        ],
        dtype=np.int32
    )


    g_train = np.asarray(
        groups_all[
            train_mask
        ]
    )


    X_validation_raw = np.ascontiguousarray(
        X_all[
            validation_mask
        ],
        dtype=np.float32
    )


    y_validation = np.ascontiguousarray(
        y_all[
            validation_mask
        ],
        dtype=np.int32
    )


    g_validation = np.asarray(
        groups_all[
            validation_mask
        ]
    )


    X_test_raw = np.ascontiguousarray(
        X_all[
            test_mask
        ],
        dtype=np.float32
    )


    y_test = np.ascontiguousarray(
        y_all[
            test_mask
        ],
        dtype=np.int32
    )


    g_test = np.asarray(
        groups_all[
            test_mask
        ]
    )


    # =========================================================================
    # Make sure training data contain every target class.
    # =========================================================================

    if len(
        np.unique(
            y_train
        )
    ) != NUM_CLASSES:

        raise RuntimeError(
            f"{split_config['Split_Name']} "
            f"training data do not contain all classes."
        )


    # =========================================================================
    # STANDARDIZATION
    #
    # Preserved from the supplied pipeline.
    #
    # The scaler is fitted only using the outer training participants.
    # =============================================================================

    scaler = StandardScaler()


    X_train = scaler.fit_transform(
        X_train_raw
    ).astype(
        np.float32,
        copy=False
    )


    X_validation = scaler.transform(
        X_validation_raw
    ).astype(
        np.float32,
        copy=False
    )


    X_test = scaler.transform(
        X_test_raw
    ).astype(
        np.float32,
        copy=False
    )


    X_train = np.ascontiguousarray(
        X_train
    )


    X_validation = np.ascontiguousarray(
        X_validation
    )


    X_test = np.ascontiguousarray(
        X_test
    )


    return {

        "train_persons":
            train_persons,

        "validation_persons":
            validation_persons,

        "test_persons":
            test_persons,

        "X_train":
            X_train,

        "y_train":
            y_train,

        "g_train":
            g_train,

        "X_validation":
            X_validation,

        "y_validation":
            y_validation,

        "g_validation":
            g_validation,

        "X_test":
            X_test,

        "y_test":
            y_test,

        "g_test":
            g_test,

        "scaler":
            scaler
    }


# =============================================================================
# GROUPKFOLD ITERATOR
# =============================================================================

def group_kfold_iterator(
    X_train,
    y_train,
    g_train
):

    splitter = GroupKFold(
        n_splits=
            GROUP_K_FOLDS
    )


    return splitter.split(
        X_train,
        y_train,
        g_train
    )


# =============================================================================
# LIGHTGBM PARAMETERS
#
# CPU is intentionally forced because the supplied process uses CPU.
# =============================================================================

def make_lgb_params(
    tuned=None,
    seed=SEED
):

    params = {

        "objective":
            "multiclass",

        "num_class":
            NUM_CLASSES,

        "metric":
            "multi_logloss",

        "boosting_type":
            "gbdt",

        "device_type":
            "cpu",

        "force_col_wise":
            True,

        "num_threads":
            N_JOBS,

        "verbosity":
            -1,

        "bagging_freq":
            1,

        "seed":
            seed,

        "bagging_seed":
            seed,

        "feature_fraction_seed":
            seed,

        "data_random_seed":
            seed
    }


    if tuned is not None:

        params.update(
            tuned
        )


    return params


# =============================================================================
# LIGHTGBM EXPERIMENT
# =============================================================================

def run_lightgbm(
    split_config,
    split_data
):

    split_name = split_config[
        "Split_Name"
    ]


    output_dir = (
        LGB_ROOT
        / split_name
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    X_train = split_data[
        "X_train"
    ]


    y_train = split_data[
        "y_train"
    ]


    g_train = split_data[
        "g_train"
    ]


    X_validation = split_data[
        "X_validation"
    ]


    y_validation = split_data[
        "y_validation"
    ]


    X_test = split_data[
        "X_test"
    ]


    y_test = split_data[
        "y_test"
    ]


    g_test = split_data[
        "g_test"
    ]


    experiment_signature_payload = {

        "Algorithm":
            "LightGBM",

        "Dataset":
            DATASET_SIGNATURE,

        "Split":
            split_name,

        "Train_Ratio":
            split_config[
                "Train_Ratio"
            ],

        "Test_Ratio":
            split_config[
                "Test_Ratio"
            ],

        "Validation_Ratio":
            split_config[
                "Validation_Ratio"
            ],

        "Optuna_Trials":
            LGB_OPTUNA_TRIALS,

        "Runs":
            LGB_RUNS,

        "GroupKFold":
            GROUP_K_FOLDS
    }


    experiment_signature = hashlib.sha256(
        json.dumps(
            experiment_signature_payload,
            sort_keys=True
        ).encode(
            "utf8"
        )
    ).hexdigest()[
        :16
    ]


    completion_path = (
        output_dir
        / "completed_result.json"
    )


    # =========================================================================
    # WHOLE SPLIT RESUME
    # =========================================================================

    if completion_path.exists():

        try:

            with open(
                completion_path,
                "r",
                encoding="utf8"
            ) as file:

                completed = json.load(
                    file
                )


            if (
                completed.get(
                    "Experiment_Signature"
                )
                == experiment_signature
            ):

                print(
                    f"LightGBM {split_name} "
                    f"already completed."
                )


                return completed[
                    "Result_Row"
                ]


        except Exception:

            pass


    print()

    print("=" * 90)

    print(
        f"LIGHTGBM | {split_name}"
    )

    print("=" * 90)


    # =========================================================================
    # SAVE SPLIT FILES
    # =========================================================================

    split_summary = {

        "Split_Name":
            split_name,

        "Requested_Train_Ratio":
            split_config[
                "Train_Ratio"
            ],

        "Requested_Test_Ratio":
            split_config[
                "Test_Ratio"
            ],

        "Requested_CV_Ratio":
            split_config[
                "Validation_Ratio"
            ],

        "Training_Participants":
            int(
                len(
                    split_data[
                        "train_persons"
                    ]
                )
            ),

        "Validation_Participants":
            int(
                len(
                    split_data[
                        "validation_persons"
                    ]
                )
            ),

        "Testing_Participants":
            int(
                len(
                    split_data[
                        "test_persons"
                    ]
                )
            ),

        "Training_Rows":
            int(
                len(
                    X_train
                )
            ),

        "Validation_Rows":
            int(
                len(
                    X_validation
                )
            ),

        "Testing_Rows":
            int(
                len(
                    X_test
                )
            )
    }


    atomic_json(
        split_summary,
        output_dir
        / "split_summary.json"
    )


    atomic_json(
        {

            "training":
                [
                    str(
                        person
                    )
                    for person
                    in split_data[
                        "train_persons"
                    ]
                ],

            "validation":
                [
                    str(
                        person
                    )
                    for person
                    in split_data[
                        "validation_persons"
                    ]
                ],

            "testing":
                [
                    str(
                        person
                    )
                    for person
                    in split_data[
                        "test_persons"
                    ]
                ]
        },
        output_dir
        / "participant_split.json"
    )


    joblib.dump(
        split_data[
            "scaler"
        ],
        output_dir
        / "standard_scaler.joblib"
    )


    joblib.dump(
        label_encoder,
        output_dir
        / "label_encoder.joblib"
    )


    # =========================================================================
    # LIGHTGBM OPTUNA
    # =========================================================================

    optuna_database = (
        output_dir
        / "lightgbm_optuna.db"
    )


    storage = (
        f"sqlite:///"
        f"{optuna_database}"
    )


    study_name = (
        f"LGB_"
        f"{split_name}_"
        f"{DATASET_SIGNATURE}_v1"
    )


    def objective(
        trial
    ):

        tuned = {

            "learning_rate":

                trial.suggest_float(
                    "learning_rate",
                    0.02,
                    0.08,
                    log=True
                ),

            "num_leaves":

                trial.suggest_int(
                    "num_leaves",
                    15,
                    63
                ),

            "max_depth":

                trial.suggest_int(
                    "max_depth",
                    3,
                    8
                ),

            "min_data_in_leaf":

                trial.suggest_int(
                    "min_data_in_leaf",
                    40,
                    200
                ),

            "feature_fraction":

                trial.suggest_float(
                    "feature_fraction",
                    0.4,
                    0.8
                ),

            "bagging_fraction":

                trial.suggest_float(
                    "bagging_fraction",
                    0.6,
                    1.0
                ),

            "lambda_l1":

                trial.suggest_float(
                    "lambda_l1",
                    1e-2,
                    20.0,
                    log=True
                ),

            "lambda_l2":

                trial.suggest_float(
                    "lambda_l2",
                    1e-2,
                    20.0,
                    log=True
                )
        }


        params = make_lgb_params(
            tuned,
            SEED
        )


        fold_scores = []


        for fold_number, (
            train_index,
            validation_index

        ) in enumerate(
            group_kfold_iterator(
                X_train,
                y_train,
                g_train
            ),
            start=1
        ):

            fold_start = time.time()


            sample_weight = compute_sample_weight(
                "balanced",
                y_train[
                    train_index
                ]
            )


            training_data = lgb.Dataset(
                X_train[
                    train_index
                ],
                label=
                    y_train[
                        train_index
                    ],
                weight=
                    sample_weight,
                free_raw_data=True
            )


            fold_validation_data = lgb.Dataset(
                X_train[
                    validation_index
                ],
                label=
                    y_train[
                        validation_index
                    ],
                reference=
                    training_data,
                free_raw_data=True
            )


            model = lgb.train(
                params,
                training_data,
                num_boost_round=
                    LGB_OPTUNA_MAX_ROUNDS,
                valid_sets=[
                    fold_validation_data
                ],
                callbacks=[
                    lgb.early_stopping(
                        LGB_OPTUNA_EARLY_STOPPING,
                        verbose=False
                    ),
                    lgb.log_evaluation(
                        0
                    )
                ]
            )


            best_iteration = (
                model.best_iteration

                if model.best_iteration
                and model.best_iteration > 0

                else LGB_OPTUNA_MAX_ROUNDS
            )


            probability = model.predict(
                X_train[
                    validation_index
                ],
                num_iteration=
                    best_iteration
            )


            prediction = np.argmax(
                probability,
                axis=1
            )


            accuracy = accuracy_score(
                y_train[
                    validation_index
                ],
                prediction
            )


            fold_scores.append(
                float(
                    accuracy
                )
            )


            print(
                f"LGB "
                f"{split_name} | "
                f"Trial "
                f"{trial.number + 1} | "
                f"Fold "
                f"{fold_number}/"
                f"{GROUP_K_FOLDS} | "
                f"Accuracy="
                f"{accuracy:.4f} | "
                f"Time="
                f"{time.time() - fold_start:.2f}s"
            )


            del model

            del training_data

            del fold_validation_data

            del sample_weight

            del probability

            del prediction


            gc.collect()


        trial.set_user_attr(
            "fold_std",
            float(
                np.std(
                    fold_scores
                )
            )
        )


        return float(
            np.mean(
                fold_scores
            )
        )


    study = optuna.create_study(
        study_name=
            study_name,
        storage=
            storage,
        direction=
            "maximize",
        sampler=
            optuna.samplers.TPESampler(
                seed=SEED
            ),
        load_if_exists=
            True
    )


    def completed_trials():

        return sum(
            1

            for trial
            in study.trials

            if trial.state
            == optuna.trial.TrialState.COMPLETE
        )


    while completed_trials() < LGB_OPTUNA_TRIALS:

        print(
            f"LightGBM Optuna completed "
            f"{completed_trials()}/"
            f"{LGB_OPTUNA_TRIALS}"
        )


        study.optimize(
            objective,
            n_trials=1,
            show_progress_bar=False,
            gc_after_trial=True
        )


    lgb_best_params = dict(
        study.best_params
    )


    study.trials_dataframe().to_csv(
        output_dir
        / "optuna_trials.csv",
        index=False
    )


    atomic_json(
        lgb_best_params,
        output_dir
        / "best_parameters.json"
    )


    print()

    print(
        "Best LightGBM GroupKFold Accuracy:",
        f"{study.best_value:.4f}"
    )


    print(
        "Best LightGBM Parameters:"
    )


    print(
        lgb_best_params
    )


    # =========================================================================
    # FIVE LIGHTGBM RUNS
    # =========================================================================

    run_records = []


    for run_number in range(
        1,
        LGB_RUNS
        + 1
    ):

        run_record_path = (
            output_dir
            / f"run_{run_number:02d}_record.json"
        )


        run_model_path = (
            output_dir
            / f"run_{run_number:02d}_model.txt"
        )


        run_signature = hashlib.sha256(
            json.dumps(
                {

                    "Experiment":
                        experiment_signature,

                    "Best_Params":
                        lgb_best_params,

                    "Run":
                        run_number

                },
                sort_keys=True
            ).encode(
                "utf8"
            )
        ).hexdigest()[
            :16
        ]


        # =====================================================================
        # RESUME COMPLETED RUN
        # =====================================================================

        if (
            run_record_path.exists()
            and
            run_model_path.exists()
        ):

            try:

                with open(
                    run_record_path,
                    "r",
                    encoding="utf8"
                ) as file:

                    old_record = json.load(
                        file
                    )


                if (
                    old_record.get(
                        "Run_Signature"
                    )
                    == run_signature
                ):

                    run_records.append(
                        old_record
                    )


                    print(
                        f"LightGBM "
                        f"{split_name} "
                        f"run "
                        f"{run_number} "
                        f"already completed."
                    )


                    continue


            except Exception:

                pass


        run_start = time.time()


        run_seed = (
            SEED
            + run_number
            - 1
        )


        params = make_lgb_params(
            lgb_best_params,
            run_seed
        )


        cv_scores = []


        # =====================================================================
        # GROUPKFOLD FOR CURRENT RUN
        # =====================================================================

        for fold_number, (
            train_index,
            validation_index

        ) in enumerate(
            group_kfold_iterator(
                X_train,
                y_train,
                g_train
            ),
            start=1
        ):

            sample_weight = compute_sample_weight(
                "balanced",
                y_train[
                    train_index
                ]
            )


            training_data = lgb.Dataset(
                X_train[
                    train_index
                ],
                label=
                    y_train[
                        train_index
                    ],
                weight=
                    sample_weight
            )


            validation_data = lgb.Dataset(
                X_train[
                    validation_index
                ],
                label=
                    y_train[
                        validation_index
                    ],
                reference=
                    training_data
            )


            fold_model = lgb.train(
                params,
                training_data,
                num_boost_round=
                    LGB_OPTUNA_MAX_ROUNDS,
                valid_sets=[
                    validation_data
                ],
                callbacks=[
                    lgb.early_stopping(
                        LGB_OPTUNA_EARLY_STOPPING,
                        verbose=False
                    ),
                    lgb.log_evaluation(
                        0
                    )
                ]
            )


            best_iteration = (
                fold_model.best_iteration

                if fold_model.best_iteration
                and fold_model.best_iteration > 0

                else LGB_OPTUNA_MAX_ROUNDS
            )


            probability = fold_model.predict(
                X_train[
                    validation_index
                ],
                num_iteration=
                    best_iteration
            )


            prediction = np.argmax(
                probability,
                axis=1
            )


            accuracy = accuracy_score(
                y_train[
                    validation_index
                ],
                prediction
            )


            cv_scores.append(
                float(
                    accuracy
                )
            )


            print(
                f"LGB "
                f"{split_name} | "
                f"Run "
                f"{run_number}/"
                f"{LGB_RUNS} | "
                f"Fold "
                f"{fold_number}/"
                f"{GROUP_K_FOLDS} | "
                f"Accuracy="
                f"{accuracy:.4f}"
            )


            del fold_model

            del training_data

            del validation_data

            del sample_weight

            del probability

            del prediction


            gc.collect()


        # =====================================================================
        # FULL TRAINING
        #
        # Fixed 10 percent validation participants control early stopping.
        # =====================================================================

        full_weight = compute_sample_weight(
            "balanced",
            y_train
        )


        full_training_data = lgb.Dataset(
            X_train,
            label=
                y_train,
            weight=
                full_weight
        )


        holdout_validation_data = lgb.Dataset(
            X_validation,
            label=
                y_validation,
            reference=
                full_training_data
        )


        model = lgb.train(
            params,
            full_training_data,
            num_boost_round=
                LGB_FINAL_MAX_ROUNDS,
            valid_sets=[
                holdout_validation_data
            ],
            callbacks=[
                lgb.early_stopping(
                    LGB_FINAL_EARLY_STOPPING,
                    verbose=False
                ),
                lgb.log_evaluation(
                    0
                )
            ]
        )


        best_iteration = (
            model.best_iteration

            if model.best_iteration
            and model.best_iteration > 0

            else LGB_FINAL_MAX_ROUNDS
        )


        train_probability = model.predict(
            X_train,
            num_iteration=
                best_iteration
        )


        train_prediction = np.argmax(
            train_probability,
            axis=1
        )


        validation_probability = model.predict(
            X_validation,
            num_iteration=
                best_iteration
        )


        validation_prediction = np.argmax(
            validation_probability,
            axis=1
        )


        train_accuracy = accuracy_score(
            y_train,
            train_prediction
        )


        validation_accuracy = accuracy_score(
            y_validation,
            validation_prediction
        )


        run_record = {

            "Run_Signature":
                run_signature,

            "Run":
                run_number,

            "Seed":
                run_seed,

            "Training_Accuracy":
                float(
                    train_accuracy
                ),

            "Validation_Accuracy":
                float(
                    validation_accuracy
                ),

            "GroupKFold_Accuracy":
                float(
                    np.mean(
                        cv_scores
                    )
                ),

            "GroupKFold_STD":
                float(
                    np.std(
                        cv_scores
                    )
                ),

            "Best_Iteration":
                int(
                    best_iteration
                ),

            "Execution_Time_Seconds":
                float(
                    time.time()
                    - run_start
                )
        }


        model.save_model(
            str(
                run_model_path
            )
        )


        atomic_json(
            run_record,
            run_record_path
        )


        run_records.append(
            run_record
        )


        print(
            f"[LGB {split_name} RUN {run_number}] "
            f"Train="
            f"{train_accuracy:.4f}, "
            f"Validation="
            f"{validation_accuracy:.4f}, "
            f"GKF="
            f"{run_record['GroupKFold_Accuracy']:.4f}"
        )


        del model

        del full_training_data

        del holdout_validation_data

        del full_weight

        del train_probability

        del train_prediction

        del validation_probability

        del validation_prediction


        gc.collect()


    run_table = pd.DataFrame(
        run_records
    )


    run_table = (
        run_table
        .sort_values(
            "Run"
        )
        .drop_duplicates(
            subset=[
                "Run"
            ],
            keep="last"
        )
    )


    run_table.to_csv(
        output_dir
        / "five_runs.csv",
        index=False
    )


    # =========================================================================
    # SELECT BEST LIGHTGBM RUN BY VALIDATION ACCURACY
    # =========================================================================

    best_run = (
        run_table
        .sort_values(
            by=[
                "Validation_Accuracy",
                "GroupKFold_Accuracy",
                "Run"
            ],
            ascending=[
                False,
                False,
                True
            ]
        )
        .iloc[
            0
        ]
    )


    best_run_number = int(
        best_run[
            "Run"
        ]
    )


    selected_model_path = (
        output_dir
        / f"run_{best_run_number:02d}_model.txt"
    )


    best_model = lgb.Booster(
        model_file=str(
            selected_model_path
        )
    )


    best_model.save_model(
        str(
            output_dir
            / "best_lightgbm_model.txt"
        )
    )


    # =========================================================================
    # TEST
    # =========================================================================

    test_probability = best_model.predict(
        X_test
    )


    test_prediction = np.argmax(
        test_probability,
        axis=1
    )


    test_metrics = calculate_metrics(
        y_test,
        test_prediction,
        test_probability
    )


    metrics = {

        "Algorithm":
            "LightGBM",

        "Split_Name":
            split_name,

        "Train_Ratio":
            split_config[
                "Train_Ratio"
            ],

        "Test_Ratio":
            split_config[
                "Test_Ratio"
            ],

        "Cross_Validation_Holdout_Ratio":
            split_config[
                "Validation_Ratio"
            ],

        "Training_Participants":
            int(
                len(
                    split_data[
                        "train_persons"
                    ]
                )
            ),

        "Validation_Participants":
            int(
                len(
                    split_data[
                        "validation_persons"
                    ]
                )
            ),

        "Testing_Participants":
            int(
                len(
                    split_data[
                        "test_persons"
                    ]
                )
            ),

        "Training_Rows":
            int(
                len(
                    X_train
                )
            ),

        "Validation_Rows":
            int(
                len(
                    X_validation
                )
            ),

        "Testing_Rows":
            int(
                len(
                    X_test
                )
            ),

        "Training_Accuracy":
            float(
                best_run[
                    "Training_Accuracy"
                ]
            ),

        "Validation_Accuracy":
            float(
                best_run[
                    "Validation_Accuracy"
                ]
            ),

        "GroupKFold_Accuracy":
            float(
                best_run[
                    "GroupKFold_Accuracy"
                ]
            ),

        "GroupKFold_STD":
            float(
                best_run[
                    "GroupKFold_STD"
                ]
            ),

        "Optuna_Best_GroupKFold_Accuracy":
            float(
                study.best_value
            ),

        "Selected_Run":
            best_run_number,

        "Best_Iteration":
            int(
                best_run[
                    "Best_Iteration"
                ]
            ),

        "Test_Accuracy":
            test_metrics[
                "accuracy"
            ],

        "Precision_Weighted":
            test_metrics[
                "precision_weighted"
            ],

        "Recall_Weighted":
            test_metrics[
                "recall_weighted"
            ],

        "F1_Weighted":
            test_metrics[
                "f1_weighted"
            ],

        "F1_Macro":
            test_metrics[
                "f1_macro"
            ],

        "Cohen_Kappa":
            test_metrics[
                "cohen_kappa"
            ],

        "MCC":
            test_metrics[
                "mcc"
            ],

        "ROC_AUC_Weighted":
            test_metrics[
                "roc_auc_weighted"
            ]
    }


    save_test_outputs(
        f"LightGBM {split_name}",
        output_dir,
        y_test,
        g_test,
        test_prediction,
        test_probability,
        metrics
    )


    result_row = {

        "Algorithm":
            "LightGBM",

        "Split_ID":
            split_config[
                "Split_ID"
            ],

        "Split_Name":
            split_name,

        "Train_Ratio":
            split_config[
                "Train_Ratio"
            ],

        "Test_Ratio":
            split_config[
                "Test_Ratio"
            ],

        "CV_Ratio":
            split_config[
                "Validation_Ratio"
            ],

        "Training_Participants":
            len(
                split_data[
                    "train_persons"
                ]
            ),

        "Testing_Participants":
            len(
                split_data[
                    "test_persons"
                ]
            ),

        "CV_Participants":
            len(
                split_data[
                    "validation_persons"
                ]
            ),

        "Training_Accuracy":
            metrics[
                "Training_Accuracy"
            ],

        "Validation_Accuracy":
            metrics[
                "Validation_Accuracy"
            ],

        "GroupKFold_Accuracy":
            metrics[
                "GroupKFold_Accuracy"
            ],

        "GroupKFold_STD":
            metrics[
                "GroupKFold_STD"
            ],

        "Test_Accuracy":
            metrics[
                "Test_Accuracy"
            ],

        "Precision_Weighted":
            metrics[
                "Precision_Weighted"
            ],

        "Recall_Weighted":
            metrics[
                "Recall_Weighted"
            ],

        "F1_Weighted":
            metrics[
                "F1_Weighted"
            ],

        "F1_Macro":
            metrics[
                "F1_Macro"
            ],

        "MCC":
            metrics[
                "MCC"
            ],

        "Cohen_Kappa":
            metrics[
                "Cohen_Kappa"
            ],

        "ROC_AUC_Weighted":
            metrics[
                "ROC_AUC_Weighted"
            ],

        "Result_Directory":
            str(
                output_dir
            )
    }


    atomic_json(
        {

            "Experiment_Signature":
                experiment_signature,

            "Result_Row":
                result_row

        },
        completion_path
    )


    print()

    print(
        f"LightGBM {split_name} "
        f"Test Accuracy = "
        f"{metrics['Test_Accuracy']:.4f}"
    )


    del best_model

    del test_probability

    del test_prediction

    del study


    gc.collect()


    return result_row


# =============================================================================
# XGBOOST EXPERIMENT
# =============================================================================

def run_xgboost(
    split_config,
    split_data
):

    split_name = split_config[
        "Split_Name"
    ]


    output_dir = (
        XGB_ROOT
        / split_name
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    X_train = split_data[
        "X_train"
    ]


    y_train = split_data[
        "y_train"
    ]


    g_train = split_data[
        "g_train"
    ]


    X_validation = split_data[
        "X_validation"
    ]


    y_validation = split_data[
        "y_validation"
    ]


    X_test = split_data[
        "X_test"
    ]


    y_test = split_data[
        "y_test"
    ]


    g_test = split_data[
        "g_test"
    ]


    experiment_signature_payload = {

        "Algorithm":
            "XGBoost",

        "Dataset":
            DATASET_SIGNATURE,

        "Split":
            split_name,

        "Train_Ratio":
            split_config[
                "Train_Ratio"
            ],

        "Test_Ratio":
            split_config[
                "Test_Ratio"
            ],

        "Validation_Ratio":
            split_config[
                "Validation_Ratio"
            ],

        "Optuna_Trials":
            XGB_OPTUNA_TRIALS,

        "GroupKFold":
            GROUP_K_FOLDS
    }


    experiment_signature = hashlib.sha256(
        json.dumps(
            experiment_signature_payload,
            sort_keys=True
        ).encode(
            "utf8"
        )
    ).hexdigest()[
        :16
    ]


    completion_path = (
        output_dir
        / "completed_result.json"
    )


    # =========================================================================
    # WHOLE SPLIT RESUME
    # =========================================================================

    if completion_path.exists():

        try:

            with open(
                completion_path,
                "r",
                encoding="utf8"
            ) as file:

                completed = json.load(
                    file
                )


            if (
                completed.get(
                    "Experiment_Signature"
                )
                == experiment_signature
            ):

                print(
                    f"XGBoost {split_name} "
                    f"already completed."
                )


                return completed[
                    "Result_Row"
                ]


        except Exception:

            pass


    print()

    print("=" * 90)

    print(
        f"XGBOOST | {split_name}"
    )

    print("=" * 90)


    # =========================================================================
    # SAVE SPLIT INFORMATION
    # =========================================================================

    split_summary = {

        "Split_Name":
            split_name,

        "Requested_Train_Ratio":
            split_config[
                "Train_Ratio"
            ],

        "Requested_Test_Ratio":
            split_config[
                "Test_Ratio"
            ],

        "Requested_CV_Ratio":
            split_config[
                "Validation_Ratio"
            ],

        "Training_Participants":
            int(
                len(
                    split_data[
                        "train_persons"
                    ]
                )
            ),

        "Validation_Participants":
            int(
                len(
                    split_data[
                        "validation_persons"
                    ]
                )
            ),

        "Testing_Participants":
            int(
                len(
                    split_data[
                        "test_persons"
                    ]
                )
            ),

        "Training_Rows":
            int(
                len(
                    X_train
                )
            ),

        "Validation_Rows":
            int(
                len(
                    X_validation
                )
            ),

        "Testing_Rows":
            int(
                len(
                    X_test
                )
            )
    }


    atomic_json(
        split_summary,
        output_dir
        / "split_summary.json"
    )


    atomic_json(
        {

            "training":
                [
                    str(
                        person
                    )
                    for person
                    in split_data[
                        "train_persons"
                    ]
                ],

            "validation":
                [
                    str(
                        person
                    )
                    for person
                    in split_data[
                        "validation_persons"
                    ]
                ],

            "testing":
                [
                    str(
                        person
                    )
                    for person
                    in split_data[
                        "test_persons"
                    ]
                ]
        },
        output_dir
        / "participant_split.json"
    )


    joblib.dump(
        split_data[
            "scaler"
        ],
        output_dir
        / "standard_scaler.joblib"
    )


    joblib.dump(
        label_encoder,
        output_dir
        / "label_encoder.joblib"
    )


    # =========================================================================
    # XGBOOST OPTUNA
    # =========================================================================

    optuna_database = (
        output_dir
        / "xgboost_optuna.db"
    )


    storage = (
        f"sqlite:///"
        f"{optuna_database}"
    )


    study_name = (
        f"XGB_"
        f"{split_name}_"
        f"{DATASET_SIGNATURE}_v1"
    )


    def objective(
        trial
    ):

        params = {

            "n_estimators":

                trial.suggest_int(
                    "n_estimators",
                    200,
                    700,
                    step=100
                ),

            "max_depth":

                trial.suggest_int(
                    "max_depth",
                    3,
                    8
                ),

            "learning_rate":

                trial.suggest_float(
                    "learning_rate",
                    0.03,
                    0.15,
                    log=True
                ),

            "subsample":

                trial.suggest_float(
                    "subsample",
                    0.6,
                    1.0
                ),

            "colsample_bytree":

                trial.suggest_float(
                    "colsample_bytree",
                    0.4,
                    0.9
                ),

            "reg_alpha":

                trial.suggest_float(
                    "reg_alpha",
                    1e-2,
                    10.0,
                    log=True
                ),

            "reg_lambda":

                trial.suggest_float(
                    "reg_lambda",
                    1e-2,
                    10.0,
                    log=True
                ),

            "objective":
                "multi:softprob",

            "num_class":
                NUM_CLASSES,

            "eval_metric":
                "mlogloss",

            "tree_method":
                "hist",

            "n_jobs":
                N_JOBS,

            "verbosity":
                0,

            "random_state":
                SEED
        }


        fold_scores = []


        for fold_number, (
            train_index,
            validation_index

        ) in enumerate(
            group_kfold_iterator(
                X_train,
                y_train,
                g_train
            ),
            start=1
        ):

            model = XGBClassifier(
                **params
            )


            sample_weight = compute_sample_weight(
                "balanced",
                y_train[
                    train_index
                ]
            )


            model.fit(
                X_train[
                    train_index
                ],
                y_train[
                    train_index
                ],
                sample_weight=
                    sample_weight
            )


            prediction = model.predict(
                X_train[
                    validation_index
                ]
            )


            accuracy = accuracy_score(
                y_train[
                    validation_index
                ],
                prediction
            )


            fold_scores.append(
                float(
                    accuracy
                )
            )


            print(
                f"XGB "
                f"{split_name} | "
                f"Trial "
                f"{trial.number + 1} | "
                f"Fold "
                f"{fold_number}/"
                f"{GROUP_K_FOLDS} | "
                f"Accuracy="
                f"{accuracy:.4f}"
            )


            del model

            del sample_weight

            del prediction


            gc.collect()


        trial.set_user_attr(
            "fold_std",
            float(
                np.std(
                    fold_scores
                )
            )
        )


        return float(
            np.mean(
                fold_scores
            )
        )


    study = optuna.create_study(
        study_name=
            study_name,
        storage=
            storage,
        direction=
            "maximize",
        sampler=
            optuna.samplers.TPESampler(
                seed=SEED
            ),
        load_if_exists=
            True
    )


    def completed_trials():

        return sum(
            1

            for trial
            in study.trials

            if trial.state
            == optuna.trial.TrialState.COMPLETE
        )


    while completed_trials() < XGB_OPTUNA_TRIALS:

        print(
            f"XGBoost Optuna completed "
            f"{completed_trials()}/"
            f"{XGB_OPTUNA_TRIALS}"
        )


        study.optimize(
            objective,
            n_trials=1,
            show_progress_bar=False,
            gc_after_trial=True
        )


    xgb_best_params = dict(
        study.best_params
    )


    study.trials_dataframe().to_csv(
        output_dir
        / "optuna_trials.csv",
        index=False
    )


    atomic_json(
        xgb_best_params,
        output_dir
        / "best_parameters.json"
    )


    print()

    print(
        "Best XGBoost GroupKFold Accuracy:",
        f"{study.best_value:.4f}"
    )


    print(
        "Best XGBoost Parameters:"
    )


    print(
        xgb_best_params
    )


    # =========================================================================
    # FINAL MODEL
    # =========================================================================

    final_params = {

        **xgb_best_params,

        "objective":
            "multi:softprob",

        "num_class":
            NUM_CLASSES,

        "eval_metric":
            "mlogloss",

        "tree_method":
            "hist",

        "n_jobs":
            N_JOBS,

        "verbosity":
            0,

        "random_state":
            SEED
    }


    model_path = (
        output_dir
        / "xgboost_model.joblib"
    )


    training_record_path = (
        output_dir
        / "training_record.json"
    )


    model_signature = hashlib.sha256(
        json.dumps(
            {

                "Experiment":
                    experiment_signature,

                "Best_Params":
                    xgb_best_params

            },
            sort_keys=True
        ).encode(
            "utf8"
        )
    ).hexdigest()[
        :16
    ]


    reuse_model = False


    if (
        model_path.exists()
        and
        training_record_path.exists()
    ):

        try:

            with open(
                training_record_path,
                "r",
                encoding="utf8"
            ) as file:

                old_training_record = json.load(
                    file
                )


            if (
                old_training_record.get(
                    "Model_Signature"
                )
                == model_signature
            ):

                xgb_model = joblib.load(
                    model_path
                )


                training_record = (
                    old_training_record
                )


                reuse_model = True


                print(
                    f"XGBoost final model for "
                    f"{split_name} already trained."
                )


        except Exception:

            reuse_model = False


    if not reuse_model:

        training_start = time.time()


        sample_weight = compute_sample_weight(
            "balanced",
            y_train
        )


        xgb_model = XGBClassifier(
            **final_params
        )


        xgb_model.fit(
            X_train,
            y_train,
            sample_weight=
                sample_weight
        )


        train_prediction = xgb_model.predict(
            X_train
        )


        train_accuracy = accuracy_score(
            y_train,
            train_prediction
        )


        validation_probability = xgb_model.predict_proba(
            X_validation
        )


        validation_prediction = np.argmax(
            validation_probability,
            axis=1
        )


        validation_accuracy = accuracy_score(
            y_validation,
            validation_prediction
        )


        training_record = {

            "Model_Signature":
                model_signature,

            "Training_Accuracy":
                float(
                    train_accuracy
                ),

            "Validation_Accuracy":
                float(
                    validation_accuracy
                ),

            "Training_Time_Seconds":
                float(
                    time.time()
                    - training_start
                )
        }


        joblib.dump(
            xgb_model,
            model_path
        )


        atomic_json(
            training_record,
            training_record_path
        )


        del sample_weight

        del train_prediction

        del validation_probability

        del validation_prediction


        gc.collect()


    # =========================================================================
    # TEST
    # =========================================================================

    test_probability = xgb_model.predict_proba(
        X_test
    )


    test_prediction = np.argmax(
        test_probability,
        axis=1
    )


    test_metrics = calculate_metrics(
        y_test,
        test_prediction,
        test_probability
    )


    # =========================================================================
    # OPTUNA STD FOR BEST TRIAL
    # =========================================================================

    try:

        best_fold_std = float(
            study.best_trial.user_attrs[
                "fold_std"
            ]
        )


    except Exception:

        best_fold_std = np.nan


    metrics = {

        "Algorithm":
            "XGBoost",

        "Split_Name":
            split_name,

        "Train_Ratio":
            split_config[
                "Train_Ratio"
            ],

        "Test_Ratio":
            split_config[
                "Test_Ratio"
            ],

        "Cross_Validation_Holdout_Ratio":
            split_config[
                "Validation_Ratio"
            ],

        "Training_Participants":
            int(
                len(
                    split_data[
                        "train_persons"
                    ]
                )
            ),

        "Validation_Participants":
            int(
                len(
                    split_data[
                        "validation_persons"
                    ]
                )
            ),

        "Testing_Participants":
            int(
                len(
                    split_data[
                        "test_persons"
                    ]
                )
            ),

        "Training_Rows":
            int(
                len(
                    X_train
                )
            ),

        "Validation_Rows":
            int(
                len(
                    X_validation
                )
            ),

        "Testing_Rows":
            int(
                len(
                    X_test
                )
            ),

        "Training_Accuracy":
            float(
                training_record[
                    "Training_Accuracy"
                ]
            ),

        "Validation_Accuracy":
            float(
                training_record[
                    "Validation_Accuracy"
                ]
            ),

        "GroupKFold_Accuracy":
            float(
                study.best_value
            ),

        "GroupKFold_STD":
            (
                None
                if np.isnan(
                    best_fold_std
                )
                else float(
                    best_fold_std
                )
            ),

        "Test_Accuracy":
            test_metrics[
                "accuracy"
            ],

        "Precision_Weighted":
            test_metrics[
                "precision_weighted"
            ],

        "Recall_Weighted":
            test_metrics[
                "recall_weighted"
            ],

        "F1_Weighted":
            test_metrics[
                "f1_weighted"
            ],

        "F1_Macro":
            test_metrics[
                "f1_macro"
            ],

        "Cohen_Kappa":
            test_metrics[
                "cohen_kappa"
            ],

        "MCC":
            test_metrics[
                "mcc"
            ],

        "ROC_AUC_Weighted":
            test_metrics[
                "roc_auc_weighted"
            ]
    }


    save_test_outputs(
        f"XGBoost {split_name}",
        output_dir,
        y_test,
        g_test,
        test_prediction,
        test_probability,
        metrics
    )


    result_row = {

        "Algorithm":
            "XGBoost",

        "Split_ID":
            split_config[
                "Split_ID"
            ],

        "Split_Name":
            split_name,

        "Train_Ratio":
            split_config[
                "Train_Ratio"
            ],

        "Test_Ratio":
            split_config[
                "Test_Ratio"
            ],

        "CV_Ratio":
            split_config[
                "Validation_Ratio"
            ],

        "Training_Participants":
            len(
                split_data[
                    "train_persons"
                ]
            ),

        "Testing_Participants":
            len(
                split_data[
                    "test_persons"
                ]
            ),

        "CV_Participants":
            len(
                split_data[
                    "validation_persons"
                ]
            ),

        "Training_Accuracy":
            metrics[
                "Training_Accuracy"
            ],

        "Validation_Accuracy":
            metrics[
                "Validation_Accuracy"
            ],

        "GroupKFold_Accuracy":
            metrics[
                "GroupKFold_Accuracy"
            ],

        "GroupKFold_STD":
            metrics[
                "GroupKFold_STD"
            ],

        "Test_Accuracy":
            metrics[
                "Test_Accuracy"
            ],

        "Precision_Weighted":
            metrics[
                "Precision_Weighted"
            ],

        "Recall_Weighted":
            metrics[
                "Recall_Weighted"
            ],

        "F1_Weighted":
            metrics[
                "F1_Weighted"
            ],

        "F1_Macro":
            metrics[
                "F1_Macro"
            ],

        "MCC":
            metrics[
                "MCC"
            ],

        "Cohen_Kappa":
            metrics[
                "Cohen_Kappa"
            ],

        "ROC_AUC_Weighted":
            metrics[
                "ROC_AUC_Weighted"
            ],

        "Result_Directory":
            str(
                output_dir
            )
    }


    atomic_json(
        {

            "Experiment_Signature":
                experiment_signature,

            "Result_Row":
                result_row

        },
        completion_path
    )


    print()

    print(
        f"XGBoost {split_name} "
        f"Test Accuracy = "
        f"{metrics['Test_Accuracy']:.4f}"
    )


    del xgb_model

    del test_probability

    del test_prediction

    del study


    gc.collect()


    return result_row


# =============================================================================
# RUN ALL SPLITS
# =============================================================================

ALL_RESULTS = []


for split_number, split_config in enumerate(
    SPLIT_CONFIGS,
    start=1
):

    print()

    print("#" * 90)

    print(
        f"MULTIPLE SPLIT EXPERIMENT "
        f"{split_number}/"
        f"{len(SPLIT_CONFIGS)}"
    )

    print("#" * 90)


    print(
        "Split:",
        split_config[
            "Split_Name"
        ]
    )


    print(
        "Train:",
        f"{split_config['Train_Ratio'] * 100:.0f}%"
    )


    print(
        "Test:",
        f"{split_config['Test_Ratio'] * 100:.0f}%"
    )


    print(
        "Cross Validation Holdout:",
        f"{split_config['Validation_Ratio'] * 100:.0f}%"
    )


    try:

        split_data = build_split(
            split_config
        )


        print()

        print(
            "Training participants:",
            len(
                split_data[
                    "train_persons"
                ]
            )
        )


        print(
            "Testing participants:",
            len(
                split_data[
                    "test_persons"
                ]
            )
        )


        print(
            "CV holdout participants:",
            len(
                split_data[
                    "validation_persons"
                ]
            )
        )


        print()

        print(
            "Training rows:",
            split_data[
                "X_train"
            ].shape
        )


        print(
            "Testing rows:",
            split_data[
                "X_test"
            ].shape
        )


        print(
            "CV holdout rows:",
            split_data[
                "X_validation"
            ].shape
        )


        print(
            "RAM:",
            f"{mem_mb():.2f} MB"
        )


        # =====================================================================
        # LIGHTGBM
        # =====================================================================

        lgb_result = run_lightgbm(
            split_config,
            split_data
        )


        ALL_RESULTS.append(
            lgb_result
        )


        gc.collect()


        # =====================================================================
        # XGBOOST
        # =====================================================================

        xgb_result = run_xgboost(
            split_config,
            split_data
        )


        ALL_RESULTS.append(
            xgb_result
        )


        # =====================================================================
        # RELEASE CURRENT SPLIT
        # =====================================================================

        del split_data

        gc.collect()


    except KeyboardInterrupt:

        print()

        print(
            "Execution manually interrupted."
        )


        print(
            "Completed Optuna trials, LightGBM runs, "
            "XGBoost models and completed split results remain saved."
        )


        raise


    except Exception:

        error_text = traceback.format_exc()


        error_path = (
            OUTPUT_ROOT
            / f"{split_config['Split_Name']}_last_error.txt"
        )


        with open(
            error_path,
            "w",
            encoding="utf8"
        ) as file:

            file.write(
                error_text
            )


        print()

        print(
            error_text
        )


        print(
            "Error log:"
        )


        print(
            error_path
        )


        raise


# =============================================================================
# COMBINED RESULTS
# =============================================================================

results_table = pd.DataFrame(
    ALL_RESULTS
)


results_table = (
    results_table
    .sort_values(
        by=[
            "Split_ID",
            "Algorithm"
        ]
    )
    .reset_index(
        drop=True
    )
)


results_table.to_csv(
    OUTPUT_ROOT
    / "XGBoost_LightGBM_Multiple_Split_Results.csv",
    index=False
)


# =============================================================================
# TEST ACCURACY PIVOT TABLE
# =============================================================================

test_accuracy_table = results_table.pivot(
    index=
        "Split_Name",
    columns=
        "Algorithm",
    values=
        "Test_Accuracy"
)


test_accuracy_table.to_csv(
    OUTPUT_ROOT
    / "Test_Accuracy_By_Split.csv"
)


# =============================================================================
# OVERALL ALGORITHM RANKING
#
# "Overall Test Accuracy" is defined here as mean Test Accuracy
# over all five requested participant split configurations.
# =============================================================================

algorithm_summary = (

    results_table

    .groupby(
        "Algorithm",
        as_index=False
    )

    .agg(

        Overall_Test_Accuracy=(
            "Test_Accuracy",
            "mean"
        ),

        Test_Accuracy_STD=(
            "Test_Accuracy",
            "std"
        ),

        Best_Test_Accuracy=(
            "Test_Accuracy",
            "max"
        ),

        Mean_Validation_Accuracy=(
            "Validation_Accuracy",
            "mean"
        ),

        Mean_GroupKFold_Accuracy=(
            "GroupKFold_Accuracy",
            "mean"
        ),

        Mean_F1_Weighted=(
            "F1_Weighted",
            "mean"
        ),

        Mean_MCC=(
            "MCC",
            "mean"
        )
    )

    .sort_values(
        by=
            "Overall_Test_Accuracy",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


algorithm_summary.to_csv(
    OUTPUT_ROOT
    / "Overall_Algorithm_Ranking.csv",
    index=False
)


# =============================================================================
# BEST OVERALL ALGORITHM
# =============================================================================

BEST_OVERALL_MODEL = str(
    algorithm_summary.iloc[
        0
    ][
        "Algorithm"
    ]
)


BEST_OVERALL_TEST_ACCURACY = float(
    algorithm_summary.iloc[
        0
    ][
        "Overall_Test_Accuracy"
    ]
)


# =============================================================================
# BEST SINGLE CONFIGURATION
# =============================================================================

best_single_row = (

    results_table

    .sort_values(
        by=[
            "Test_Accuracy",
            "F1_Weighted",
            "MCC"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )

    .iloc[
        0
    ]
)


BEST_SINGLE_MODEL = str(
    best_single_row[
        "Algorithm"
    ]
)


BEST_SINGLE_SPLIT = str(
    best_single_row[
        "Split_Name"
    ]
)


BEST_SINGLE_TEST_ACCURACY = float(
    best_single_row[
        "Test_Accuracy"
    ]
)


# =============================================================================
# SAVE BEST MODEL SUMMARY
# =============================================================================

best_model_summary = {

    "Best_Overall_Model":
        BEST_OVERALL_MODEL,

    "Selection_Basis":
        (
            "Mean Test Accuracy across "
            "all five requested split ratios"
        ),

    "Overall_Test_Accuracy":
        BEST_OVERALL_TEST_ACCURACY,

    "Best_Single_Configuration_Model":
        BEST_SINGLE_MODEL,

    "Best_Single_Configuration_Split":
        BEST_SINGLE_SPLIT,

    "Best_Single_Test_Accuracy":
        BEST_SINGLE_TEST_ACCURACY
}


atomic_json(
    best_model_summary,
    OUTPUT_ROOT
    / "Best_Model_Summary.json"
)


with open(
    OUTPUT_ROOT
    / "Best_Model_Name.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        f"BEST OVERALL MODEL: "
        f"{BEST_OVERALL_MODEL}\n"
    )

    file.write(
        f"OVERALL TEST ACCURACY: "
        f"{BEST_OVERALL_TEST_ACCURACY:.6f}\n"
    )

    file.write(
        f"BEST SINGLE CONFIGURATION: "
        f"{BEST_SINGLE_MODEL}\n"
    )

    file.write(
        f"BEST SPLIT: "
        f"{BEST_SINGLE_SPLIT}\n"
    )

    file.write(
        f"BEST SINGLE TEST ACCURACY: "
        f"{BEST_SINGLE_TEST_ACCURACY:.6f}\n"
    )


# =============================================================================
# TEST ACCURACY COMPARISON FIGURE
# =============================================================================

plot_table = results_table.copy()


plot_table[
    "Split_Label"
] = (
    plot_table[
        "Train_Ratio"
    ]
    .mul(
        100
    )
    .astype(
        int
    )
    .astype(
        str
    )
    + "/"
    +
    plot_table[
        "Test_Ratio"
    ]
    .mul(
        100
    )
    .astype(
        int
    )
    .astype(
        str
    )
    + "/"
    +
    plot_table[
        "CV_Ratio"
    ]
    .mul(
        100
    )
    .astype(
        int
    )
    .astype(
        str
    )
)


pivot_plot = plot_table.pivot(
    index=
        "Split_Label",
    columns=
        "Algorithm",
    values=
        "Test_Accuracy"
)


pivot_plot = pivot_plot.reindex(
    [
        "70/20/10",
        "60/30/10",
        "75/15/10",
        "65/25/10",
        "50/40/10"
    ]
)


ax = pivot_plot.plot(
    kind="bar",
    figsize=(
        10,
        6
    )
)


ax.set_xlabel(
    "Train / Test / CV Participant Ratio"
)


ax.set_ylabel(
    "Test Accuracy"
)


ax.set_title(
    "LightGBM and XGBoost Test Accuracy Across Split Ratios"
)


ax.legend(
    title="Algorithm"
)


ax.grid(
    axis="y",
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    OUTPUT_ROOT
    / "Test_Accuracy_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close()


# =============================================================================
# FINAL SCREEN OUTPUT
# =============================================================================

total_time = (
    time.time()
    - TOTAL_START
)


print()

print("#" * 90)

print(
    "ALL LIGHTGBM AND XGBOOST MULTIPLE SPLIT RESULTS"
)

print("#" * 90)


print()

print(
    results_table[
        [

            "Algorithm",

            "Split_Name",

            "Training_Accuracy",

            "Validation_Accuracy",

            "GroupKFold_Accuracy",

            "Test_Accuracy",

            "F1_Weighted",

            "MCC"

        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print()

print("=" * 90)

print(
    "OVERALL ALGORITHM RANKING"
)

print("=" * 90)


print()

print(
    algorithm_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print()

print("#" * 90)

print(
    "BEST OVERALL MODEL"
)

print("#" * 90)


print(
    "Model Name             :",
    BEST_OVERALL_MODEL
)


print(
    "Overall Test Accuracy  :",
    f"{BEST_OVERALL_TEST_ACCURACY:.4f}"
)


print(
    "Selection Basis        : "
    "Mean Test Accuracy across all 5 split ratios"
)


print()

print(
    "BEST SINGLE CONFIGURATION"
)


print(
    "Model Name             :",
    BEST_SINGLE_MODEL
)


print(
    "Split                  :",
    BEST_SINGLE_SPLIT
)


print(
    "Test Accuracy          :",
    f"{BEST_SINGLE_TEST_ACCURACY:.4f}"
)


print()

print(
    "Total execution time   :",
    f"{total_time / 60:.2f}",
    "minutes"
)


print(
    "Current RAM            :",
    f"{mem_mb():.2f}",
    "MB"
)


print()

print(
    "LIGHTGBM RESULTS:"
)

print(
    LGB_ROOT
)


print()

print(
    "XGBOOST RESULTS:"
)

print(
    XGB_ROOT
)


print()

print(
    "COMBINED RESULT:"
)

print(
    OUTPUT_ROOT
    / "XGBoost_LightGBM_Multiple_Split_Results.csv"
)


print("#" * 90)

/home/rakib/Desktop/ml_apply/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LIGHTGBM AND XGBOOST MULTIPLE SPLIT EXPERIMENT
Input file:
/home/rakib/Desktop/ml_apply/Final_Model_Test/Cleaned_oN_Selected_Max.csv

LightGBM device : CPU
XGBoost device  : CPU hist
CPU workers     : 8
GroupKFold      : 5
LightGBM trials : 15
XGBoost trials  : 12
LightGBM runs   : 5
Data reduction  : NONE

LOADING MAX DATASET
Raw dataset shape: (149350, 238)
RAM after loading: 499.71 MB
Feature count: 232
Invalid feature cells: 0

Feature matrix: (149350, 232)
Classes: ['class_lecture', 'music', 'normal', 'reading']
Number of classes: 4
Participants: 60

CLASS DISTRIBUTION
class_lecture: 39857 (26.69%)
music: 32276 (21.61%)
normal: 39327 (26.33%)
reading: 37890 (25.37%)

Dataset signature: 9f53e766a1b04d74
Loading time: 8.41 seconds

FIXED CROSS VALIDATION HOLDOUT PARTICIPANTS
Total participants: 60
Fixed validation participants: 6

##########################################################################################
MULTIPLE SPLIT EXPERIMENT 1/5
#################################

# Statistical Evaluation of the Final LightGBM and XGBoost Models

## Experimental Setting

The final statistical evaluation is performed on the LightGBM and XGBoost models obtained using the selected participant independent data split:

| Dataset Portion | Ratio |
|---|---:|
| Training | 60% |
| Testing | 30% |
| Validation | 10% |

The same previously trained LightGBM and XGBoost models are reused for this analysis. Therefore, the statistical tests evaluate the exact models obtained from the selected 60%, 30%, and 10% configuration.

Participant independence is maintained in the original data split. Samples from the same participant do not occur simultaneously in the training, validation, and testing partitions.

## Models

The following two classifiers are evaluated:

**LightGBM**

LightGBM uses gradient boosted decision trees with the hyperparameters previously optimized through participant based GroupKFold cross validation and Optuna.

**XGBoost**

XGBoost uses histogram based gradient boosted trees with hyperparameters previously optimized using participant based GroupKFold cross validation and Optuna.

## Statistical and Performance Evaluation

Fifteen evaluation procedures are considered.

### Bootstrap 95% Confidence Interval

A participant level cluster bootstrap is performed rather than independently resampling EEG windows. Test participants are sampled with replacement and the accuracy distribution is calculated for both classifiers.

The analysis reports the 95% confidence interval for LightGBM accuracy, XGBoost accuracy, and the paired difference between their accuracies.

### McNemar Test

McNemar's test compares the paired classification outcomes of LightGBM and XGBoost on the same test samples.

The contingency table contains samples correctly classified by both models, samples correctly classified only by LightGBM, samples correctly classified only by XGBoost, and samples incorrectly classified by both models.

Both the continuity corrected statistic and exact binomial p value are reported.

### Wilcoxon Signed Rank Test

The Wilcoxon signed rank test compares participant level accuracies between LightGBM and XGBoost.

Each test participant therefore contributes one paired LightGBM accuracy and one paired XGBoost accuracy.

This approach avoids treating every EEG window as an independent experimental subject.

### Paired t Test

A paired t test is also applied to participant level accuracies.

The null hypothesis states that the mean participant level accuracy difference between LightGBM and XGBoost is zero.

### Friedman Test

The Friedman test is intended for comparisons involving three or more related algorithms or treatments.

Because the current study compares only LightGBM and XGBoost, a valid Friedman comparison is not performed. The result folder records the method as not applicable.

### Nemenyi Test

The Nemenyi test is normally used as a post hoc procedure following a Friedman test involving three or more algorithms.

Because only two classifiers are being compared, the Nemenyi test is recorded as not applicable.

### DeLong Test

The original DeLong test compares correlated ROC AUC values for binary classification.

Because the current problem contains four classes, a one versus rest analysis is performed separately for each class.

For each class, LightGBM and XGBoost ROC AUC values are compared using the paired DeLong procedure.

Bonferroni adjusted p values are also reported to account for the four classwise comparisons.

### Permutation Test

A participant level paired permutation test evaluates the difference between LightGBM and XGBoost accuracies.

The signs of participant level accuracy differences are randomly permuted 10,000 times to construct the null distribution.

The resulting two sided p value indicates whether the observed participant level performance difference is larger than expected under the null hypothesis.

### Cochran Q Test

Cochran's Q test is the extension of McNemar's test for three or more related binary treatments or classifiers.

Because only two models are evaluated, Cochran's Q is not applicable. McNemar's test provides the appropriate paired comparison for the current experiment.

### Cohen Kappa

Cohen's Kappa is calculated independently for LightGBM and XGBoost.

It measures agreement between the predicted classes and true classes while accounting for agreement expected by chance.

Higher Kappa values indicate stronger agreement.

### Chi Square Test

A Chi Square test of independence is applied separately to the multiclass confusion matrix of each classifier.

The analysis tests the association between true class membership and predicted class membership.

The observed and expected frequency matrices are saved for both algorithms.

### Repeated Stratified Cross Validation

Repeated Stratified Group Cross Validation is performed using the training and validation participant pool while preserving the final 30% test participants.

Five stratified grouped folds are repeated five times, producing 25 evaluations for each model.

Participant grouping is explicitly maintained so that samples belonging to the same participant cannot occur in both the training and validation portions of a fold.

The previously optimized hyperparameters are reused rather than performing hyperparameter optimization again.

Mean accuracy, standard deviation, weighted F1 score, and all fold results are saved.

### Brier Score

The multiclass Brier score evaluates the quality of predicted probabilities rather than only the final class labels.

The score measures the squared difference between predicted class probabilities and the actual one hot encoded class labels.

Lower Brier scores indicate better probabilistic predictions.

Class specific one versus rest Brier scores are also calculated.

### Calibration Curve

Calibration curves compare predicted probabilities with observed class frequencies.

One versus rest calibration curves are generated separately for each class and for both algorithms.

Expected Calibration Error and Maximum Calibration Error are also calculated using the confidence of the predicted class.

Lower calibration error indicates that predicted probabilities more accurately represent empirical prediction reliability.

### Hosmer Lemeshow Test

The classical Hosmer Lemeshow procedure is designed for binary probability models.

For the current four class classification problem, the test is applied separately to each class using a one versus rest formulation.

Predicted probabilities are divided into ten groups and observed positive frequencies are compared with expected positive frequencies.

The Hosmer Lemeshow statistic, degrees of freedom, raw p value, and Bonferroni adjusted p value are reported for every model and class.

## Significance Level

For inferential statistical tests, the conventional significance level can be interpreted as:

`p < 0.05`

A p value below 0.05 indicates evidence against the corresponding null hypothesis.

Statistical significance should not be interpreted as equivalent to practical significance. Accuracy differences, confidence intervals, effect direction, calibration metrics, and participant level consistency should also be considered.

## Important Methodological Note

The 60%, 30%, and 10% data split was selected after comparing test performance across several alternative split ratios.

Therefore, the current statistical evaluation is a post selection analysis rather than an entirely independent confirmatory test.

For a final publication claim, an external dataset or an additional completely untouched participant holdout set would provide stronger evidence of generalization.

## Output Structure

The results are saved under:

`/home/rakib/Desktop/ml_apply/15_Testing apply_FinalModel/`

Separate folders are generated for:

`01_Bootstrap_95_Confidence_Interval`

`02_McNemar_Test`

`03_Wilcoxon_Signed_Rank_Test`

`04_Paired_t_Test`

`05_Friedman_Test`

`06_Nemenyi_Test`

`07_DeLong_Test`

`08_Permutation_Test`

`09_Cochran_Q_Test`

`10_Cohen_Kappa`

`11_Chi_Square_Test`

`12_Repeated_Stratified_Cross_Validation`

`13_Brier_Score`

`14_Calibration_Curve`

`15_Hosmer_Lemeshow_Test`

The root output directory also contains the paired test predictions, participant level accuracy table, base model metrics, overall statistical test summary, and final statistical summary.

In [4]:
# =============================================================================
# 15 STATISTICAL AND PERFORMANCE TESTING METHODS
#
# FINAL MODELS
#
# LightGBM
# XGBoost
#
# SELECTED SPLIT
#
# Training   60 percent
# Testing    30 percent
# Validation 10 percent
#
# INPUT DATASET
#
# /home/rakib/Desktop/ml_apply/Final_Model_Test/
# Cleaned_oN_Selected_Max.csv
#
# PREVIOUS TRAINED MODELS
#
# /home/rakib/Desktop/ml_apply/Final_Model_Test/
# LightGBM_Multiple_Split_Results/Train60_Test30_CV10/
#
# /home/rakib/Desktop/ml_apply/Final_Model_Test/
# XGBoost_Multiple_Split_Results/Train60_Test30_CV10/
#
# OUTPUT
#
# /home/rakib/Desktop/ml_apply/15_Testing apply_FinalModel/
#
# METHODS
#
# 01 Bootstrap 95 Percent Confidence Interval
# 02 McNemar Test
# 03 Wilcoxon Signed Rank Test
# 04 Paired t Test
# 05 Friedman Test
# 06 Nemenyi Test
# 07 DeLong Test
# 08 Permutation Test
# 09 Cochran Q Test
# 10 Cohen Kappa
# 11 Chi Square Test
# 12 Repeated Stratified Cross Validation
# 13 Brier Score
# 14 Calibration Curve
# 15 Hosmer Lemeshow Test
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import gc
import json
import time
import math
import random
import hashlib
import warnings
import traceback

from pathlib import Path

import joblib
import psutil

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import lightgbm as lgb

from xgboost import XGBClassifier

from scipy.stats import (
    binomtest,
    wilcoxon,
    ttest_rel,
    chi2,
    chi2_contingency,
    norm
)

from sklearn.model_selection import (
    StratifiedGroupKFold
)

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.calibration import (
    calibration_curve
)

from sklearn.utils.class_weight import (
    compute_sample_weight
)


# =============================================================================
# WARNINGS
# =============================================================================

warnings.filterwarnings(
    "ignore"
)


# =============================================================================
# GLOBAL CONFIGURATION
# =============================================================================

SEED = 42


random.seed(
    SEED
)


np.random.seed(
    SEED
)


N_JOBS = max(
    1,
    min(
        8,
        os.cpu_count() or 1
    )
)


# =============================================================================
# STATISTICAL TEST CONFIGURATION
# =============================================================================

BOOTSTRAP_ITERATIONS = 5000

PERMUTATION_ITERATIONS = 10000

REPEATED_CV_REPEATS = 5

REPEATED_CV_FOLDS = 5

CALIBRATION_BINS = 10

HL_GROUPS = 10

SIGNIFICANCE_LEVEL = 0.05


# =============================================================================
# PATHS
# =============================================================================

INPUT_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "Final_Model_Test/"
    "Cleaned_oN_Selected_Max.csv"
)


PREVIOUS_ROOT = (
    INPUT_PATH.parent
)


# =============================================================================
# LIGHTGBM SOURCE
# =============================================================================

LGB_SOURCE_DIR = (
    PREVIOUS_ROOT
    / "LightGBM_Multiple_Split_Results"
    / "Train60_Test30_CV10"
)


# =============================================================================
# XGBOOST SOURCE
# =============================================================================

XGB_SOURCE_DIR = (
    PREVIOUS_ROOT
    / "XGBoost_Multiple_Split_Results"
    / "Train60_Test30_CV10"
)


# =============================================================================
# OUTPUT ROOT
# =============================================================================

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "15_Testing apply_FinalModel"
)


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# OUTPUT METHOD FOLDERS
# =============================================================================

METHOD_DIRS = {

    "bootstrap":
        OUTPUT_ROOT
        / "01_Bootstrap_95_Confidence_Interval",

    "mcnemar":
        OUTPUT_ROOT
        / "02_McNemar_Test",

    "wilcoxon":
        OUTPUT_ROOT
        / "03_Wilcoxon_Signed_Rank_Test",

    "paired_t":
        OUTPUT_ROOT
        / "04_Paired_t_Test",

    "friedman":
        OUTPUT_ROOT
        / "05_Friedman_Test",

    "nemenyi":
        OUTPUT_ROOT
        / "06_Nemenyi_Test",

    "delong":
        OUTPUT_ROOT
        / "07_DeLong_Test",

    "permutation":
        OUTPUT_ROOT
        / "08_Permutation_Test",

    "cochran_q":
        OUTPUT_ROOT
        / "09_Cochran_Q_Test",

    "kappa":
        OUTPUT_ROOT
        / "10_Cohen_Kappa",

    "chi_square":
        OUTPUT_ROOT
        / "11_Chi_Square_Test",

    "repeated_cv":
        OUTPUT_ROOT
        / "12_Repeated_Stratified_Cross_Validation",

    "brier":
        OUTPUT_ROOT
        / "13_Brier_Score",

    "calibration":
        OUTPUT_ROOT
        / "14_Calibration_Curve",

    "hosmer":
        OUTPUT_ROOT
        / "15_Hosmer_Lemeshow_Test"
}


for directory in METHOD_DIRS.values():

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# =============================================================================
# MEMORY MONITOR
# =============================================================================

PROCESS = psutil.Process(
    os.getpid()
)


def memory_mb():

    return (
        PROCESS.memory_info().rss
        / 1024
        / 1024
    )


# =============================================================================
# ATOMIC JSON SAVE
# =============================================================================

def save_json(
    data,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name
        + ".tmp"
    )


    with open(
        temporary,
        "w",
        encoding="utf8"
    ) as file:

        json.dump(
            data,
            file,
            indent=4,
            allow_nan=True
        )


    os.replace(
        temporary,
        path
    )


# =============================================================================
# ATOMIC CSV SAVE
# =============================================================================

def save_csv(
    dataframe,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name
        + ".tmp"
    )


    dataframe.to_csv(
        temporary,
        index=False
    )


    os.replace(
        temporary,
        path
    )


# =============================================================================
# PARTICIPANT ID NORMALIZATION
#
# Prevents accidental mismatch such as:
#
# 1
# 1.0
#
# =============================================================================

def normalize_person_id(
    value
):

    text = str(
        value
    ).strip()


    try:

        numeric = float(
            text
        )


        if (
            np.isfinite(
                numeric
            )
            and
            numeric.is_integer()
        ):

            return str(
                int(
                    numeric
                )
            )


    except Exception:

        pass


    return text


# =============================================================================
# P VALUE INTERPRETATION
# =============================================================================

def significance_text(
    p_value
):

    if p_value is None:

        return "Not applicable"


    try:

        if not np.isfinite(
            p_value
        ):

            return "Not available"


    except Exception:

        return "Not available"


    if p_value < SIGNIFICANCE_LEVEL:

        return (
            "Statistically significant at alpha = 0.05"
        )


    return (
        "Not statistically significant at alpha = 0.05"
    )


# =============================================================================
# CHECK REQUIRED FILES
# =============================================================================

REQUIRED_FILES = [

    INPUT_PATH,

    LGB_SOURCE_DIR
    / "best_lightgbm_model.txt",

    LGB_SOURCE_DIR
    / "standard_scaler.joblib",

    LGB_SOURCE_DIR
    / "label_encoder.joblib",

    LGB_SOURCE_DIR
    / "participant_split.json",

    LGB_SOURCE_DIR
    / "best_parameters.json",

    LGB_SOURCE_DIR
    / "final_metrics.json",

    XGB_SOURCE_DIR
    / "xgboost_model.joblib",

    XGB_SOURCE_DIR
    / "standard_scaler.joblib",

    XGB_SOURCE_DIR
    / "label_encoder.joblib",

    XGB_SOURCE_DIR
    / "participant_split.json",

    XGB_SOURCE_DIR
    / "best_parameters.json",

    XGB_SOURCE_DIR
    / "final_metrics.json"
]


missing_files = [

    str(
        path
    )

    for path
    in REQUIRED_FILES

    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(

        "Required files are missing:\n\n"

        + "\n".join(
            missing_files
        )
    )


# =============================================================================
# START
# =============================================================================

TOTAL_START = time.time()


print(
    "=" * 100
)


print(
    "15 TESTING METHODS FOR FINAL LIGHTGBM AND XGBOOST MODELS"
)


print(
    "=" * 100
)


print(
    "Input:"
)


print(
    INPUT_PATH
)


print()


print(
    "Output:"
)


print(
    OUTPUT_ROOT
)


print()


print(
    "CPU workers:",
    N_JOBS
)


print(
    "Bootstrap iterations:",
    BOOTSTRAP_ITERATIONS
)


print(
    "Permutation iterations:",
    PERMUTATION_ITERATIONS
)


print(
    "Repeated grouped stratified CV:",
    f"{REPEATED_CV_REPEATS} repeats x "
    f"{REPEATED_CV_FOLDS} folds"
)


print(
    "=" * 100
)


# =============================================================================
# LOAD PREVIOUS MODEL ARTIFACTS
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "LOADING PREVIOUS 60 / 30 / 10 MODELS"
)


print(
    "=" * 100
)


# =============================================================================
# SCALERS
# =============================================================================

lgb_scaler = joblib.load(
    LGB_SOURCE_DIR
    / "standard_scaler.joblib"
)


xgb_scaler = joblib.load(
    XGB_SOURCE_DIR
    / "standard_scaler.joblib"
)


# =============================================================================
# LABEL ENCODERS
# =============================================================================

label_encoder = joblib.load(
    LGB_SOURCE_DIR
    / "label_encoder.joblib"
)


xgb_label_encoder = joblib.load(
    XGB_SOURCE_DIR
    / "label_encoder.joblib"
)


if not np.array_equal(
    label_encoder.classes_,
    xgb_label_encoder.classes_
):

    raise ValueError(

        "LightGBM and XGBoost label encoders "
        "do not contain the same class order."
    )


CLASS_NAMES = list(
    label_encoder.classes_
)


NUM_CLASSES = len(
    CLASS_NAMES
)


# =============================================================================
# PARTICIPANT SPLITS
# =============================================================================

with open(
    LGB_SOURCE_DIR
    / "participant_split.json",
    "r",
    encoding="utf8"
) as file:

    lgb_split = json.load(
        file
    )


with open(
    XGB_SOURCE_DIR
    / "participant_split.json",
    "r",
    encoding="utf8"
) as file:

    xgb_split = json.load(
        file
    )


def normalized_split(
    split_dict
):

    return {

        key:
            sorted(
                [
                    normalize_person_id(
                        value
                    )

                    for value
                    in values
                ]
            )

        for key, values
        in split_dict.items()
    }


if (
    normalized_split(
        lgb_split
    )
    !=
    normalized_split(
        xgb_split
    )
):

    raise ValueError(

        "LightGBM and XGBoost were not trained "
        "with the same participant split."
    )


# =============================================================================
# BEST PARAMETERS
# =============================================================================

with open(
    LGB_SOURCE_DIR
    / "best_parameters.json",
    "r",
    encoding="utf8"
) as file:

    LGB_BEST_PARAMS = json.load(
        file
    )


with open(
    XGB_SOURCE_DIR
    / "best_parameters.json",
    "r",
    encoding="utf8"
) as file:

    XGB_BEST_PARAMS = json.load(
        file
    )


# =============================================================================
# FINAL METRICS
# =============================================================================

with open(
    LGB_SOURCE_DIR
    / "final_metrics.json",
    "r",
    encoding="utf8"
) as file:

    LGB_FINAL_METRICS = json.load(
        file
    )


with open(
    XGB_SOURCE_DIR
    / "final_metrics.json",
    "r",
    encoding="utf8"
) as file:

    XGB_FINAL_METRICS = json.load(
        file
    )


# =============================================================================
# LIGHTGBM FIXED ROUND COUNT FOR REPEATED CV
# =============================================================================

LGB_FIXED_ROUNDS = max(
    1,
    int(
        LGB_FINAL_METRICS.get(
            "Best_Iteration",
            500
        )
    )
)


print(
    "Classes:",
    CLASS_NAMES
)


print(
    "LightGBM fixed rounds for repeated CV:",
    LGB_FIXED_ROUNDS
)


# =============================================================================
# LOAD DATASET
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "LOADING DATASET"
)


print(
    "=" * 100
)


df = pd.read_csv(
    INPUT_PATH,
    low_memory=False
)


# =============================================================================
# COLUMN DEFINITIONS
# =============================================================================

METADATA_COLUMNS = [

    "PersonSrial",

    "Window",

    "Window_Start",

    "Window_End",

    "Window_Samples"
]


TARGET_COLUMN = "Class"


if "PersonSrial" not in df.columns:

    raise ValueError(
        "PersonSrial column is missing."
    )


if TARGET_COLUMN not in df.columns:

    raise ValueError(
        "Class column is missing."
    )


FEATURE_COLUMNS = [

    column

    for column
    in df.columns

    if column not in METADATA_COLUMNS

    and column != TARGET_COLUMN
]


# =============================================================================
# FEATURE MATRIX
# =============================================================================

X_all = (

    df[
        FEATURE_COLUMNS
    ]

    .apply(
        pd.to_numeric,
        errors="coerce"
    )

    .to_numpy(
        dtype=np.float32
    )
)


if not np.isfinite(
    X_all
).all():

    raise ValueError(

        "Feature matrix contains NaN "
        "or infinite values."
    )


# =============================================================================
# LABELS
# =============================================================================

y_all = label_encoder.transform(

    df[
        TARGET_COLUMN
    ].astype(
        str
    )

).astype(
    np.int32
)


# =============================================================================
# PARTICIPANT IDS
# =============================================================================

groups_all = np.asarray(
    [

        normalize_person_id(
            value
        )

        for value
        in df[
            "PersonSrial"
        ].to_numpy()
    ],
    dtype=str
)


# =============================================================================
# FEATURE COMPATIBILITY
# =============================================================================

if (
    lgb_scaler.n_features_in_
    != len(
        FEATURE_COLUMNS
    )
):

    raise ValueError(

        "Current dataset feature count does not "
        "match saved LightGBM scaler."
    )


if (
    xgb_scaler.n_features_in_
    != len(
        FEATURE_COLUMNS
    )
):

    raise ValueError(

        "Current dataset feature count does not "
        "match saved XGBoost scaler."
    )


print(
    "Dataset shape:",
    X_all.shape
)


print(
    "Feature count:",
    len(
        FEATURE_COLUMNS
    )
)


print(
    "Participants:",
    len(
        np.unique(
            groups_all
        )
    )
)


print(
    "RAM:",
    f"{memory_mb():.2f} MB"
)


del df

gc.collect()


# =============================================================================
# NORMALIZED PARTICIPANT SPLITS
# =============================================================================

TRAIN_PERSONS = np.asarray(
    [

        normalize_person_id(
            value
        )

        for value
        in lgb_split[
            "training"
        ]
    ],
    dtype=str
)


VALIDATION_PERSONS = np.asarray(
    [

        normalize_person_id(
            value
        )

        for value
        in lgb_split[
            "validation"
        ]
    ],
    dtype=str
)


TEST_PERSONS = np.asarray(
    [

        normalize_person_id(
            value
        )

        for value
        in lgb_split[
            "testing"
        ]
    ],
    dtype=str
)


# =============================================================================
# SPLIT MASKS
# =============================================================================

train_mask = np.isin(
    groups_all,
    TRAIN_PERSONS
)


validation_mask = np.isin(
    groups_all,
    VALIDATION_PERSONS
)


test_mask = np.isin(
    groups_all,
    TEST_PERSONS
)


# =============================================================================
# VERIFY NO PARTICIPANT OVERLAP
# =============================================================================

assert not np.any(
    train_mask
    & validation_mask
)


assert not np.any(
    train_mask
    & test_mask
)


assert not np.any(
    validation_mask
    & test_mask
)


# =============================================================================
# VERIFY ALL SAVED PARTICIPANTS WERE FOUND
# =============================================================================

found_train_persons = set(
    np.unique(
        groups_all[
            train_mask
        ]
    )
)


found_validation_persons = set(
    np.unique(
        groups_all[
            validation_mask
        ]
    )
)


found_test_persons = set(
    np.unique(
        groups_all[
            test_mask
        ]
    )
)


if found_train_persons != set(
    TRAIN_PERSONS.tolist()
):

    raise ValueError(
        "Not all saved training participants were found."
    )


if found_validation_persons != set(
    VALIDATION_PERSONS.tolist()
):

    raise ValueError(
        "Not all saved validation participants were found."
    )


if found_test_persons != set(
    TEST_PERSONS.tolist()
):

    raise ValueError(
        "Not all saved testing participants were found."
    )


# =============================================================================
# FINAL TEST SET
# =============================================================================

X_test_raw = np.ascontiguousarray(
    X_all[
        test_mask
    ],
    dtype=np.float32
)


y_test = np.ascontiguousarray(
    y_all[
        test_mask
    ],
    dtype=np.int32
)


g_test = np.asarray(
    groups_all[
        test_mask
    ],
    dtype=str
)


# =============================================================================
# DEVELOPMENT SET FOR REPEATED CROSS VALIDATION
#
# Training plus original 10 percent validation.
#
# The final 30 percent test participants remain completely excluded.
# =============================================================================

development_mask = (
    train_mask
    | validation_mask
)


X_dev_raw = np.ascontiguousarray(
    X_all[
        development_mask
    ],
    dtype=np.float32
)


y_dev = np.ascontiguousarray(
    y_all[
        development_mask
    ],
    dtype=np.int32
)


g_dev = np.asarray(
    groups_all[
        development_mask
    ],
    dtype=str
)


print()


print(
    "Training participants:",
    len(
        TRAIN_PERSONS
    )
)


print(
    "Validation participants:",
    len(
        VALIDATION_PERSONS
    )
)


print(
    "Testing participants:",
    len(
        TEST_PERSONS
    )
)


print()


print(
    "Final test rows:",
    len(
        y_test
    )
)


print(
    "Repeated CV development rows:",
    len(
        y_dev
    )
)


# =============================================================================
# RELEASE FULL MATRICES
# =============================================================================

del X_all

del y_all

del groups_all

del train_mask

del validation_mask

del test_mask

del development_mask

gc.collect()


# =============================================================================
# LOAD FINAL MODELS
# =============================================================================

lgb_model = lgb.Booster(

    model_file=str(
        LGB_SOURCE_DIR
        / "best_lightgbm_model.txt"
    )
)


xgb_model = joblib.load(

    XGB_SOURCE_DIR
    / "xgboost_model.joblib"
)


# =============================================================================
# SCALE FINAL TEST DATA
# =============================================================================

X_test_lgb = lgb_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False
)


X_test_xgb = xgb_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False
)


X_test_lgb = np.ascontiguousarray(
    X_test_lgb
)


X_test_xgb = np.ascontiguousarray(
    X_test_xgb
)


# =============================================================================
# FINAL MODEL PREDICTIONS
# =============================================================================

lgb_probability = np.asarray(
    lgb_model.predict(
        X_test_lgb
    ),
    dtype=np.float64
)


lgb_prediction = np.argmax(
    lgb_probability,
    axis=1
).astype(
    np.int32
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_test_xgb
    ),
    dtype=np.float64
)


xgb_prediction = np.argmax(
    xgb_probability,
    axis=1
).astype(
    np.int32
)


# =============================================================================
# VERIFY PROBABILITY SHAPES
# =============================================================================

expected_shape = (
    len(
        y_test
    ),
    NUM_CLASSES
)


if lgb_probability.shape != expected_shape:

    raise ValueError(

        f"Unexpected LightGBM probability shape: "
        f"{lgb_probability.shape}"
    )


if xgb_probability.shape != expected_shape:

    raise ValueError(

        f"Unexpected XGBoost probability shape: "
        f"{xgb_probability.shape}"
    )


# =============================================================================
# BASIC CLASSIFICATION METRICS
# =============================================================================

def classification_metrics(
    y_true,
    prediction,
    probability
):

    result = {

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    prediction
                )
            ),

        "Precision_Weighted":
            float(
                precision_score(
                    y_true,
                    prediction,
                    average="weighted",
                    zero_division=0
                )
            ),

        "Recall_Weighted":
            float(
                recall_score(
                    y_true,
                    prediction,
                    average="weighted",
                    zero_division=0
                )
            ),

        "F1_Weighted":
            float(
                f1_score(
                    y_true,
                    prediction,
                    average="weighted",
                    zero_division=0
                )
            ),

        "F1_Macro":
            float(
                f1_score(
                    y_true,
                    prediction,
                    average="macro",
                    zero_division=0
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y_true,
                    prediction
                )
            ),

        "Cohen_Kappa":
            float(
                cohen_kappa_score(
                    y_true,
                    prediction
                )
            )
    }


    try:

        result[
            "ROC_AUC_Weighted_OVR"
        ] = float(
            roc_auc_score(
                y_true,
                probability,
                multi_class="ovr",
                average="weighted"
            )
        )


    except Exception:

        result[
            "ROC_AUC_Weighted_OVR"
        ] = None


    return result


LGB_BASE_METRICS = classification_metrics(
    y_test,
    lgb_prediction,
    lgb_probability
)


XGB_BASE_METRICS = classification_metrics(
    y_test,
    xgb_prediction,
    xgb_probability
)


base_metrics_table = pd.DataFrame(
    [

        {
            "Algorithm":
                "LightGBM",

            **LGB_BASE_METRICS
        },

        {
            "Algorithm":
                "XGBoost",

            **XGB_BASE_METRICS
        }
    ]
)


base_metrics_table.to_csv(
    OUTPUT_ROOT
    / "Base_Model_Test_Metrics.csv",
    index=False
)


# =============================================================================
# SAVE CLASSIFICATION REPORTS
# =============================================================================

for model_name, prediction in [

    (
        "LightGBM",
        lgb_prediction
    ),

    (
        "XGBoost",
        xgb_prediction
    )
]:

    report_text = classification_report(
        y_test,
        prediction,
        labels=np.arange(
            NUM_CLASSES
        ),
        target_names=
            CLASS_NAMES,
        digits=6,
        zero_division=0
    )


    with open(
        OUTPUT_ROOT
        / f"{model_name}_Classification_Report.txt",
        "w",
        encoding="utf8"
    ) as file:

        file.write(
            report_text
        )


    pd.DataFrame(
        classification_report(
            y_test,
            prediction,
            labels=np.arange(
                NUM_CLASSES
            ),
            target_names=
                CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
    ).transpose().to_csv(
        OUTPUT_ROOT
        / f"{model_name}_Classification_Report.csv"
    )


# =============================================================================
# PAIRED TEST PREDICTIONS
# =============================================================================

paired_predictions = pd.DataFrame(
    {

        "PersonSrial":
            g_test,

        "True_Class":
            label_encoder.inverse_transform(
                y_test
            ),

        "LightGBM_Prediction":
            label_encoder.inverse_transform(
                lgb_prediction
            ),

        "XGBoost_Prediction":
            label_encoder.inverse_transform(
                xgb_prediction
            ),

        "LightGBM_Correct":
            (
                lgb_prediction
                == y_test
            ).astype(
                np.int32
            ),

        "XGBoost_Correct":
            (
                xgb_prediction
                == y_test
            ).astype(
                np.int32
            )
    }
)


for class_index, class_name in enumerate(
    CLASS_NAMES
):

    paired_predictions[
        f"LightGBM_Probability_{class_name}"
    ] = lgb_probability[
        :,
        class_index
    ]


    paired_predictions[
        f"XGBoost_Probability_{class_name}"
    ] = xgb_probability[
        :,
        class_index
    ]


paired_predictions.to_csv(
    OUTPUT_ROOT
    / "Paired_Test_Predictions.csv",
    index=False
)


# =============================================================================
# PARTICIPANT LEVEL TEST RESULTS
#
# This is important because multiple EEG windows belong to each participant.
# =============================================================================

participant_records = []


for participant in np.unique(
    g_test
):

    participant_index = (
        g_test
        == participant
    )


    participant_y = y_test[
        participant_index
    ]


    participant_lgb = lgb_prediction[
        participant_index
    ]


    participant_xgb = xgb_prediction[
        participant_index
    ]


    participant_records.append(
        {

            "PersonSrial":
                participant,

            "Rows":
                int(
                    np.sum(
                        participant_index
                    )
                ),

            "LightGBM_Correct":
                int(
                    np.sum(
                        participant_lgb
                        == participant_y
                    )
                ),

            "XGBoost_Correct":
                int(
                    np.sum(
                        participant_xgb
                        == participant_y
                    )
                ),

            "LightGBM_Accuracy":
                float(
                    accuracy_score(
                        participant_y,
                        participant_lgb
                    )
                ),

            "XGBoost_Accuracy":
                float(
                    accuracy_score(
                        participant_y,
                        participant_xgb
                    )
                )
        }
    )


participant_table = pd.DataFrame(
    participant_records
)


participant_table[
    "Accuracy_Difference_LGB_minus_XGB"
] = (

    participant_table[
        "LightGBM_Accuracy"
    ]

    -

    participant_table[
        "XGBoost_Accuracy"
    ]
)


participant_table.to_csv(
    OUTPUT_ROOT
    / "Participant_Level_Test_Accuracy.csv",
    index=False
)


# =============================================================================
# MASTER SUMMARY
# =============================================================================

STATISTICAL_SUMMARY = []


def add_summary(
    method,
    comparison,
    statistic=None,
    p_value=None,
    interpretation=None,
    status="Completed"
):

    if (
        p_value is not None
        and
        isinstance(
            p_value,
            (
                float,
                int,
                np.floating,
                np.integer
            )
        )
    ):

        significant = significance_text(
            float(
                p_value
            )
        )


    else:

        significant = (
            "Not applicable"
        )


    STATISTICAL_SUMMARY.append(
        {

            "Method":
                method,

            "Comparison":
                comparison,

            "Statistic":
                statistic,

            "P_Value":
                p_value,

            "Significance":
                significant,

            "Status":
                status,

            "Interpretation":
                interpretation
        }
    )


# =============================================================================
# 01 BOOTSTRAP 95 PERCENT CONFIDENCE INTERVAL
#
# PARTICIPANT CLUSTER BOOTSTRAP
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "01 BOOTSTRAP 95 PERCENT CONFIDENCE INTERVAL"
)


print(
    "=" * 100
)


bootstrap_dir = METHOD_DIRS[
    "bootstrap"
]


bootstrap_rng = np.random.default_rng(
    SEED
)


participant_rows = participant_table[
    "Rows"
].to_numpy(
    dtype=np.float64
)


participant_lgb_correct = participant_table[
    "LightGBM_Correct"
].to_numpy(
    dtype=np.float64
)


participant_xgb_correct = participant_table[
    "XGBoost_Correct"
].to_numpy(
    dtype=np.float64
)


n_participants = len(
    participant_table
)


bootstrap_lgb = np.empty(
    BOOTSTRAP_ITERATIONS,
    dtype=np.float64
)


bootstrap_xgb = np.empty(
    BOOTSTRAP_ITERATIONS,
    dtype=np.float64
)


bootstrap_difference = np.empty(
    BOOTSTRAP_ITERATIONS,
    dtype=np.float64
)


for iteration in range(
    BOOTSTRAP_ITERATIONS
):

    sampled_participants = bootstrap_rng.integers(
        0,
        n_participants,
        size=n_participants
    )


    participant_counts = np.bincount(
        sampled_participants,
        minlength=n_participants
    ).astype(
        np.float64
    )


    total_rows = float(
        np.dot(
            participant_counts,
            participant_rows
        )
    )


    lgb_accuracy = float(
        np.dot(
            participant_counts,
            participant_lgb_correct
        )
        / total_rows
    )


    xgb_accuracy = float(
        np.dot(
            participant_counts,
            participant_xgb_correct
        )
        / total_rows
    )


    bootstrap_lgb[
        iteration
    ] = lgb_accuracy


    bootstrap_xgb[
        iteration
    ] = xgb_accuracy


    bootstrap_difference[
        iteration
    ] = (
        lgb_accuracy
        - xgb_accuracy
    )


def percentile_ci(
    values
):

    return (

        float(
            np.percentile(
                values,
                2.5
            )
        ),

        float(
            np.percentile(
                values,
                97.5
            )
        )
    )


lgb_ci_low, lgb_ci_high = percentile_ci(
    bootstrap_lgb
)


xgb_ci_low, xgb_ci_high = percentile_ci(
    bootstrap_xgb
)


diff_ci_low, diff_ci_high = percentile_ci(
    bootstrap_difference
)


bootstrap_summary = pd.DataFrame(
    [

        {
            "Measure":
                "LightGBM Test Accuracy",

            "Observed":
                LGB_BASE_METRICS[
                    "Accuracy"
                ],

            "CI_95_Lower":
                lgb_ci_low,

            "CI_95_Upper":
                lgb_ci_high
        },

        {
            "Measure":
                "XGBoost Test Accuracy",

            "Observed":
                XGB_BASE_METRICS[
                    "Accuracy"
                ],

            "CI_95_Lower":
                xgb_ci_low,

            "CI_95_Upper":
                xgb_ci_high
        },

        {
            "Measure":
                (
                    "Accuracy Difference "
                    "LightGBM minus XGBoost"
                ),

            "Observed":
                (
                    LGB_BASE_METRICS[
                        "Accuracy"
                    ]

                    -

                    XGB_BASE_METRICS[
                        "Accuracy"
                    ]
                ),

            "CI_95_Lower":
                diff_ci_low,

            "CI_95_Upper":
                diff_ci_high
        }
    ]
)


bootstrap_summary.to_csv(
    bootstrap_dir
    / "bootstrap_95_confidence_interval.csv",
    index=False
)


pd.DataFrame(
    {

        "Bootstrap":
            np.arange(
                1,
                BOOTSTRAP_ITERATIONS
                + 1
            ),

        "LightGBM_Accuracy":
            bootstrap_lgb,

        "XGBoost_Accuracy":
            bootstrap_xgb,

        "Difference_LGB_minus_XGB":
            bootstrap_difference
    }
).to_csv(
    bootstrap_dir
    / "bootstrap_samples.csv",
    index=False
)


fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.hist(
    bootstrap_difference,
    bins=50
)


ax.axvline(
    0.0,
    linewidth=1
)


ax.set_xlabel(
    "Accuracy Difference LightGBM minus XGBoost"
)


ax.set_ylabel(
    "Bootstrap Frequency"
)


ax.set_title(
    "Participant Bootstrap Accuracy Difference"
)


ax.grid(
    axis="y",
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    bootstrap_dir
    / "bootstrap_accuracy_difference.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


add_summary(

    "Bootstrap 95% Confidence Interval",

    "LightGBM minus XGBoost accuracy",

    statistic=
        float(
            np.mean(
                bootstrap_difference
            )
        ),

    p_value=None,

    interpretation=
        (
            f"Participant cluster bootstrap 95% CI: "
            f"[{diff_ci_low:.6f}, {diff_ci_high:.6f}]"
        )
)


# =============================================================================
# 02 MCNEMAR TEST
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "02 MCNEMAR TEST"
)


print(
    "=" * 100
)


mcnemar_dir = METHOD_DIRS[
    "mcnemar"
]


lgb_correct = (
    lgb_prediction
    == y_test
)


xgb_correct = (
    xgb_prediction
    == y_test
)


both_correct = int(
    np.sum(
        lgb_correct
        & xgb_correct
    )
)


lgb_only_correct = int(
    np.sum(
        lgb_correct
        & ~xgb_correct
    )
)


xgb_only_correct = int(
    np.sum(
        ~lgb_correct
        & xgb_correct
    )
)


both_wrong = int(
    np.sum(
        ~lgb_correct
        & ~xgb_correct
    )
)


discordant_total = (
    lgb_only_correct
    + xgb_only_correct
)


if discordant_total > 0:

    exact_result = binomtest(
        lgb_only_correct,
        discordant_total,
        p=0.5,
        alternative="two-sided"
    )


    exact_p = float(
        exact_result.pvalue
    )


    mcnemar_statistic = float(
        (
            (
                abs(
                    lgb_only_correct
                    - xgb_only_correct
                )
                - 1
            )
            ** 2
        )
        / discordant_total
    )


    asymptotic_p = float(
        chi2.sf(
            mcnemar_statistic,
            df=1
        )
    )


else:

    exact_p = 1.0

    mcnemar_statistic = 0.0

    asymptotic_p = 1.0


mcnemar_table = pd.DataFrame(
    [

        {
            "Outcome":
                "Both Correct",

            "Count":
                both_correct
        },

        {
            "Outcome":
                "LightGBM Correct XGBoost Wrong",

            "Count":
                lgb_only_correct
        },

        {
            "Outcome":
                "LightGBM Wrong XGBoost Correct",

            "Count":
                xgb_only_correct
        },

        {
            "Outcome":
                "Both Wrong",

            "Count":
                both_wrong
        }
    ]
)


mcnemar_table.to_csv(
    mcnemar_dir
    / "mcnemar_contingency_table.csv",
    index=False
)


save_json(
    {

        "Statistic_Continuity_Corrected":
            mcnemar_statistic,

        "Asymptotic_P_Value":
            asymptotic_p,

        "Exact_Binomial_P_Value":
            exact_p,

        "Significance":
            significance_text(
                exact_p
            ),

        "LightGBM_Correct_XGBoost_Wrong":
            lgb_only_correct,

        "XGBoost_Correct_LightGBM_Wrong":
            xgb_only_correct
    },
    mcnemar_dir
    / "mcnemar_result.json"
)


add_summary(

    "McNemar Test",

    "LightGBM versus XGBoost",

    statistic=
        mcnemar_statistic,

    p_value=
        exact_p,

    interpretation=
        (
            "Exact two sided paired classification comparison. "
            "The raw EEG windows are correlated within participants, "
            "so participant level tests should also be considered."
        )
)


# =============================================================================
# 03 WILCOXON SIGNED RANK TEST
#
# PARTICIPANT LEVEL
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "03 WILCOXON SIGNED RANK TEST"
)


print(
    "=" * 100
)


wilcoxon_dir = METHOD_DIRS[
    "wilcoxon"
]


participant_lgb_accuracy = participant_table[
    "LightGBM_Accuracy"
].to_numpy(
    dtype=np.float64
)


participant_xgb_accuracy = participant_table[
    "XGBoost_Accuracy"
].to_numpy(
    dtype=np.float64
)


participant_difference = (
    participant_lgb_accuracy
    - participant_xgb_accuracy
)


if np.allclose(
    participant_difference,
    0.0
):

    wilcoxon_statistic = 0.0

    wilcoxon_p = 1.0


else:

    wilcoxon_result = wilcoxon(
        participant_lgb_accuracy,
        participant_xgb_accuracy,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto"
    )


    wilcoxon_statistic = float(
        wilcoxon_result.statistic
    )


    wilcoxon_p = float(
        wilcoxon_result.pvalue
    )


participant_table[
    [
        "PersonSrial",
        "Rows",
        "LightGBM_Accuracy",
        "XGBoost_Accuracy",
        "Accuracy_Difference_LGB_minus_XGB"
    ]
].to_csv(
    wilcoxon_dir
    / "participant_paired_accuracies.csv",
    index=False
)


save_json(
    {

        "Statistic":
            wilcoxon_statistic,

        "P_Value":
            wilcoxon_p,

        "Significance":
            significance_text(
                wilcoxon_p
            ),

        "Participants":
            int(
                len(
                    participant_table
                )
            )
    },
    wilcoxon_dir
    / "wilcoxon_result.json"
)


add_summary(

    "Wilcoxon Signed Rank Test",

    "Participant level accuracy",

    statistic=
        wilcoxon_statistic,

    p_value=
        wilcoxon_p,

    interpretation=
        (
            "Two sided paired nonparametric comparison "
            "of participant level accuracy."
        )
)


# =============================================================================
# 04 PAIRED T TEST
#
# PARTICIPANT LEVEL
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "04 PAIRED T TEST"
)


print(
    "=" * 100
)


paired_t_dir = METHOD_DIRS[
    "paired_t"
]


if np.allclose(
    participant_difference,
    participant_difference[
        0
    ]
):

    if np.isclose(
        participant_difference[
            0
        ],
        0.0
    ):

        paired_t_statistic = 0.0

        paired_t_p = 1.0


    else:

        paired_t_statistic = np.inf

        paired_t_p = 0.0


else:

    t_result = ttest_rel(
        participant_lgb_accuracy,
        participant_xgb_accuracy,
        nan_policy="omit"
    )


    paired_t_statistic = float(
        t_result.statistic
    )


    paired_t_p = float(
        t_result.pvalue
    )


save_json(
    {

        "Statistic":
            paired_t_statistic,

        "P_Value":
            paired_t_p,

        "Significance":
            significance_text(
                paired_t_p
            ),

        "Mean_LightGBM_Participant_Accuracy":
            float(
                np.mean(
                    participant_lgb_accuracy
                )
            ),

        "Mean_XGBoost_Participant_Accuracy":
            float(
                np.mean(
                    participant_xgb_accuracy
                )
            ),

        "Mean_Difference":
            float(
                np.mean(
                    participant_difference
                )
            ),

        "Participants":
            int(
                len(
                    participant_table
                )
            )
    },
    paired_t_dir
    / "paired_t_test_result.json"
)


participant_table[
    [
        "PersonSrial",
        "LightGBM_Accuracy",
        "XGBoost_Accuracy",
        "Accuracy_Difference_LGB_minus_XGB"
    ]
].to_csv(
    paired_t_dir
    / "participant_paired_accuracies.csv",
    index=False
)


add_summary(

    "Paired t Test",

    "Participant level accuracy",

    statistic=
        paired_t_statistic,

    p_value=
        paired_t_p,

    interpretation=
        (
            "Two sided paired t test using one accuracy "
            "value per test participant."
        )
)


# =============================================================================
# HELPER FOR NON APPLICABLE METHODS
# =============================================================================

def save_not_applicable(
    directory,
    method,
    reason
):

    result = {

        "Method":
            method,

        "Status":
            "Not Applicable",

        "Reason":
            reason,

        "Number_of_Models":
            2,

        "Models":
            [
                "LightGBM",
                "XGBoost"
            ]
    }


    save_json(
        result,
        directory
        / "result.json"
    )


    with open(
        directory
        / "status.txt",
        "w",
        encoding="utf8"
    ) as file:

        file.write(
            f"{method}\n\n"
        )

        file.write(
            "Status: Not Applicable\n\n"
        )

        file.write(
            reason
        )

        file.write(
            "\n"
        )


# =============================================================================
# 05 FRIEDMAN TEST
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "05 FRIEDMAN TEST"
)


print(
    "=" * 100
)


friedman_reason = (

    "The Friedman test requires three or more related "
    "algorithms or treatments for a meaningful algorithm "
    "comparison. The current experiment contains only "
    "LightGBM and XGBoost. Therefore the Friedman test "
    "is not performed."
)


save_not_applicable(
    METHOD_DIRS[
        "friedman"
    ],
    "Friedman Test",
    friedman_reason
)


add_summary(

    "Friedman Test",

    "LightGBM versus XGBoost",

    status=
        "Not Applicable",

    interpretation=
        friedman_reason
)


# =============================================================================
# 06 NEMENYI TEST
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "06 NEMENYI TEST"
)


print(
    "=" * 100
)


nemenyi_reason = (

    "The Nemenyi procedure is normally a post hoc test "
    "following a Friedman comparison involving three or "
    "more algorithms. With only LightGBM and XGBoost, "
    "a Nemenyi comparison is not appropriate."
)


save_not_applicable(
    METHOD_DIRS[
        "nemenyi"
    ],
    "Nemenyi Test",
    nemenyi_reason
)


add_summary(

    "Nemenyi Test",

    "LightGBM versus XGBoost",

    status=
        "Not Applicable",

    interpretation=
        nemenyi_reason
)


# =============================================================================
# 07 DELONG TEST
#
# MULTICLASS ONE VERSUS REST
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "07 DELONG TEST"
)


print(
    "=" * 100
)


delong_dir = METHOD_DIRS[
    "delong"
]


# =============================================================================
# MIDRANK
# =============================================================================

def compute_midrank(
    x
):

    x = np.asarray(
        x,
        dtype=np.float64
    )


    order = np.argsort(
        x
    )


    sorted_x = x[
        order
    ]


    n = len(
        x
    )


    temporary = np.zeros(
        n,
        dtype=np.float64
    )


    i = 0


    while i < n:

        j = i + 1


        while (
            j < n
            and
            sorted_x[
                j
            ]
            == sorted_x[
                i
            ]
        ):

            j += 1


        temporary[
            i:j
        ] = (
            0.5
            * (
                i
                + j
                - 1
            )
        )


        i = j


    result = np.empty(
        n,
        dtype=np.float64
    )


    result[
        order
    ] = (
        temporary
        + 1.0
    )


    return result


# =============================================================================
# FAST DELONG
# =============================================================================

def fast_delong(
    predictions_sorted_transposed,
    positive_count
):

    m = int(
        positive_count
    )


    n = (
        predictions_sorted_transposed.shape[
            1
        ]
        - m
    )


    classifiers = (
        predictions_sorted_transposed.shape[
            0
        ]
    )


    positive_examples = (
        predictions_sorted_transposed[
            :,
            :m
        ]
    )


    negative_examples = (
        predictions_sorted_transposed[
            :,
            m:
        ]
    )


    tx = np.empty(
        (
            classifiers,
            m
        ),
        dtype=np.float64
    )


    ty = np.empty(
        (
            classifiers,
            n
        ),
        dtype=np.float64
    )


    tz = np.empty(
        (
            classifiers,
            m + n
        ),
        dtype=np.float64
    )


    for classifier_index in range(
        classifiers
    ):

        tx[
            classifier_index
        ] = compute_midrank(
            positive_examples[
                classifier_index
            ]
        )


        ty[
            classifier_index
        ] = compute_midrank(
            negative_examples[
                classifier_index
            ]
        )


        tz[
            classifier_index
        ] = compute_midrank(
            predictions_sorted_transposed[
                classifier_index
            ]
        )


    aucs = (

        tz[
            :,
            :m
        ].sum(
            axis=1
        )
        / m
        / n

        -

        (
            m
            + 1.0
        )
        / 2.0
        / n
    )


    v01 = (
        tz[
            :,
            :m
        ]
        - tx
    ) / n


    v10 = (

        1.0

        -

        (
            tz[
                :,
                m:
            ]
            - ty
        )
        / m
    )


    sx = np.cov(
        v01,
        bias=False
    )


    sy = np.cov(
        v10,
        bias=False
    )


    sx = np.atleast_2d(
        sx
    )


    sy = np.atleast_2d(
        sy
    )


    covariance = (
        sx / m
        + sy / n
    )


    return (
        aucs,
        covariance
    )


# =============================================================================
# DELONG PAIRWISE TEST
# =============================================================================

def delong_test(
    ground_truth,
    prediction_one,
    prediction_two
):

    ground_truth = np.asarray(
        ground_truth,
        dtype=np.int32
    )


    prediction_one = np.asarray(
        prediction_one,
        dtype=np.float64
    )


    prediction_two = np.asarray(
        prediction_two,
        dtype=np.float64
    )


    positive_count = int(
        np.sum(
            ground_truth == 1
        )
    )


    negative_count = (
        len(
            ground_truth
        )
        - positive_count
    )


    if (
        positive_count < 2
        or
        negative_count < 2
    ):

        return {

            "AUC_LightGBM":
                np.nan,

            "AUC_XGBoost":
                np.nan,

            "AUC_Difference_LGB_minus_XGB":
                np.nan,

            "Z":
                np.nan,

            "P_Value":
                np.nan
        }


    sorting_order = np.argsort(
        -ground_truth
    )


    predictions = np.vstack(
        [
            prediction_one,
            prediction_two
        ]
    )[
        :,
        sorting_order
    ]


    aucs, covariance = fast_delong(
        predictions,
        positive_count
    )


    contrast = np.asarray(
        [
            1.0,
            -1.0
        ],
        dtype=np.float64
    )


    variance = float(
        contrast
        @ covariance
        @ contrast.T
    )


    difference = float(
        aucs[
            0
        ]
        - aucs[
            1
        ]
    )


    if (
        not np.isfinite(
            variance
        )
        or
        variance <= 0
    ):

        z_value = np.nan

        p_value = np.nan


    else:

        z_value = float(
            difference
            / math.sqrt(
                variance
            )
        )


        p_value = float(
            2.0
            * norm.sf(
                abs(
                    z_value
                )
            )
        )


    return {

        "AUC_LightGBM":
            float(
                aucs[
                    0
                ]
            ),

        "AUC_XGBoost":
            float(
                aucs[
                    1
                ]
            ),

        "AUC_Difference_LGB_minus_XGB":
            difference,

        "Z":
            z_value,

        "P_Value":
            p_value
    }


delong_records = []


for class_index, class_name in enumerate(
    CLASS_NAMES
):

    binary_truth = (
        y_test
        == class_index
    ).astype(
        np.int32
    )


    result = delong_test(
        binary_truth,
        lgb_probability[
            :,
            class_index
        ],
        xgb_probability[
            :,
            class_index
        ]
    )


    result[
        "Class"
    ] = class_name


    delong_records.append(
        result
    )


delong_table = pd.DataFrame(
    delong_records
)


valid_delong_p = pd.to_numeric(
    delong_table[
        "P_Value"
    ],
    errors="coerce"
)


delong_table[
    "Bonferroni_Adjusted_P"
] = np.minimum(
    valid_delong_p
    * NUM_CLASSES,
    1.0
)


delong_table[
    "Significance_After_Bonferroni"
] = [

    significance_text(
        value
    )

    if pd.notna(
        value
    )

    else "Not available"

    for value in delong_table[
        "Bonferroni_Adjusted_P"
    ]
]


delong_table.to_csv(
    delong_dir
    / "delong_one_vs_rest_results.csv",
    index=False
)


valid_adjusted = delong_table[
    "Bonferroni_Adjusted_P"
].dropna()


if len(
    valid_adjusted
) > 0:

    minimum_delong_p = float(
        valid_adjusted.min()
    )


else:

    minimum_delong_p = None


add_summary(

    "DeLong Test",

    "Classwise one versus rest ROC AUC",

    statistic=None,

    p_value=
        minimum_delong_p,

    interpretation=
        (
            "DeLong was applied separately to each class "
            "using one versus rest ROC analysis. "
            "Bonferroni adjusted p values are reported."
        )
)


# =============================================================================
# 08 PERMUTATION TEST
#
# PARTICIPANT LEVEL PAIRED SIGN PERMUTATION
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "08 PERMUTATION TEST"
)


print(
    "=" * 100
)


permutation_dir = METHOD_DIRS[
    "permutation"
]


observed_difference = float(
    np.mean(
        participant_difference
    )
)


permutation_rng = np.random.default_rng(
    SEED
    + 1000
)


permutation_distribution = np.empty(
    PERMUTATION_ITERATIONS,
    dtype=np.float64
)


for iteration in range(
    PERMUTATION_ITERATIONS
):

    signs = permutation_rng.choice(
        [
            -1.0,
            1.0
        ],
        size=len(
            participant_difference
        )
    )


    permutation_distribution[
        iteration
    ] = float(
        np.mean(
            participant_difference
            * signs
        )
    )


permutation_p = float(

    (
        1
        +
        np.sum(
            np.abs(
                permutation_distribution
            )
            >= abs(
                observed_difference
            )
        )
    )

    /

    (
        PERMUTATION_ITERATIONS
        + 1
    )
)


save_json(
    {

        "Observed_Mean_Participant_Accuracy_Difference":
            observed_difference,

        "Iterations":
            PERMUTATION_ITERATIONS,

        "P_Value":
            permutation_p,

        "Significance":
            significance_text(
                permutation_p
            ),

        "Direction":
            "LightGBM minus XGBoost"
    },
    permutation_dir
    / "permutation_test_result.json"
)


pd.DataFrame(
    {

        "Permutation":
            np.arange(
                1,
                PERMUTATION_ITERATIONS
                + 1
            ),

        "Null_Mean_Difference":
            permutation_distribution
    }
).to_csv(
    permutation_dir
    / "permutation_distribution.csv",
    index=False
)


fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.hist(
    permutation_distribution,
    bins=50
)


ax.axvline(
    observed_difference,
    linewidth=1
)


ax.axvline(
    -observed_difference,
    linewidth=1
)


ax.set_xlabel(
    "Participant Mean Accuracy Difference"
)


ax.set_ylabel(
    "Permutation Frequency"
)


ax.set_title(
    "Participant Level Permutation Test"
)


ax.grid(
    axis="y",
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    permutation_dir
    / "permutation_distribution.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


add_summary(

    "Permutation Test",

    "Participant level accuracy",

    statistic=
        observed_difference,

    p_value=
        permutation_p,

    interpretation=
        (
            "Two sided paired sign permutation test "
            "using participant level accuracy differences."
        )
)


# =============================================================================
# 09 COCHRAN Q TEST
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "09 COCHRAN Q TEST"
)


print(
    "=" * 100
)


cochran_reason = (

    "Cochran Q is an extension of McNemar testing for "
    "three or more matched classifiers or treatments. "
    "Only two classifiers are included here. Therefore "
    "McNemar is the appropriate corresponding test."
)


save_not_applicable(
    METHOD_DIRS[
        "cochran_q"
    ],
    "Cochran Q Test",
    cochran_reason
)


add_summary(

    "Cochran Q Test",

    "LightGBM versus XGBoost",

    status=
        "Not Applicable",

    interpretation=
        cochran_reason
)


# =============================================================================
# 10 COHEN KAPPA
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "10 COHEN KAPPA"
)


print(
    "=" * 100
)


kappa_dir = METHOD_DIRS[
    "kappa"
]


lgb_kappa = float(
    cohen_kappa_score(
        y_test,
        lgb_prediction
    )
)


xgb_kappa = float(
    cohen_kappa_score(
        y_test,
        xgb_prediction
    )
)


model_agreement_kappa = float(
    cohen_kappa_score(
        lgb_prediction,
        xgb_prediction
    )
)


kappa_table = pd.DataFrame(
    [

        {
            "Comparison":
                "True Labels vs LightGBM",

            "Cohen_Kappa":
                lgb_kappa
        },

        {
            "Comparison":
                "True Labels vs XGBoost",

            "Cohen_Kappa":
                xgb_kappa
        },

        {
            "Comparison":
                "LightGBM vs XGBoost Predictions",

            "Cohen_Kappa":
                model_agreement_kappa
        }
    ]
)


kappa_table.to_csv(
    kappa_dir
    / "cohen_kappa_results.csv",
    index=False
)


add_summary(

    "Cohen Kappa",

    "True labels versus LightGBM",

    statistic=
        lgb_kappa,

    interpretation=
        "Chance corrected agreement between true class and LightGBM prediction."
)


add_summary(

    "Cohen Kappa",

    "True labels versus XGBoost",

    statistic=
        xgb_kappa,

    interpretation=
        "Chance corrected agreement between true class and XGBoost prediction."
)


# =============================================================================
# 11 CHI SQUARE TEST
#
# TRUE CLASS VERSUS PREDICTED CLASS
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "11 CHI SQUARE TEST"
)


print(
    "=" * 100
)


chi_dir = METHOD_DIRS[
    "chi_square"
]


chi_records = []


for model_name, prediction in [

    (
        "LightGBM",
        lgb_prediction
    ),

    (
        "XGBoost",
        xgb_prediction
    )
]:

    matrix = confusion_matrix(
        y_test,
        prediction,
        labels=np.arange(
            NUM_CLASSES
        )
    )


    observed_table = pd.DataFrame(
        matrix,
        index=
            CLASS_NAMES,
        columns=
            CLASS_NAMES
    )


    observed_table.to_csv(
        chi_dir
        / f"{model_name}_observed_confusion_matrix.csv"
    )


    try:

        (
            statistic,
            p_value,
            degrees_freedom,
            expected

        ) = chi2_contingency(
            matrix,
            correction=False
        )


        pd.DataFrame(
            expected,
            index=
                CLASS_NAMES,
            columns=
                CLASS_NAMES
        ).to_csv(
            chi_dir
            / f"{model_name}_expected_frequencies.csv"
        )


        record = {

            "Algorithm":
                model_name,

            "Chi_Square":
                float(
                    statistic
                ),

            "Degrees_of_Freedom":
                int(
                    degrees_freedom
                ),

            "P_Value":
                float(
                    p_value
                ),

            "Significance":
                significance_text(
                    float(
                        p_value
                    )
                )
        }


    except Exception as error:

        record = {

            "Algorithm":
                model_name,

            "Chi_Square":
                np.nan,

            "Degrees_of_Freedom":
                np.nan,

            "P_Value":
                np.nan,

            "Significance":
                "Not available",

            "Error":
                str(
                    error
                )
        }


    chi_records.append(
        record
    )


chi_table = pd.DataFrame(
    chi_records
)


chi_table.to_csv(
    chi_dir
    / "chi_square_results.csv",
    index=False
)


for record in chi_records:

    p_value = record.get(
        "P_Value"
    )


    if (
        p_value is not None
        and
        np.isfinite(
            p_value
        )
    ):

        summary_p = float(
            p_value
        )


    else:

        summary_p = None


    add_summary(

        "Chi Square Test",

        record[
            "Algorithm"
        ],

        statistic=
            record.get(
                "Chi_Square"
            ),

        p_value=
            summary_p,

        interpretation=
            (
                "Association between true multiclass labels "
                "and predicted multiclass labels."
            )
    )


# =============================================================================
# 12 REPEATED STRATIFIED GROUP CROSS VALIDATION
#
# 5 REPEATS X 5 FOLDS
#
# POWER CUT SUPPORT
#
# Every completed fold is immediately stored.
#
# If the script is restarted, already completed folds are skipped.
#
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "12 REPEATED STRATIFIED GROUP CROSS VALIDATION"
)


print(
    "=" * 100
)


cv_dir = METHOD_DIRS[
    "repeated_cv"
]


cv_result_path = (
    cv_dir
    / "repeated_stratified_group_cv_results.csv"
)


# =============================================================================
# CV SIGNATURE
# =============================================================================

cv_signature_payload = {

    "Input":
        str(
            INPUT_PATH
        ),

    "LightGBM_Params":
        LGB_BEST_PARAMS,

    "XGBoost_Params":
        XGB_BEST_PARAMS,

    "LightGBM_Rounds":
        LGB_FIXED_ROUNDS,

    "Repeats":
        REPEATED_CV_REPEATS,

    "Folds":
        REPEATED_CV_FOLDS,

    "Seed":
        SEED,

    "Development_Participants":
        sorted(
            np.unique(
                g_dev
            ).tolist()
        )
}


CV_SIGNATURE = hashlib.sha256(
    json.dumps(
        cv_signature_payload,
        sort_keys=True
    ).encode(
        "utf8"
    )
).hexdigest()[
    :16
]


# =============================================================================
# RESTORE PREVIOUS COMPLETED CV FOLDS
# =============================================================================

cv_records = []


if cv_result_path.exists():

    try:

        previous_cv = pd.read_csv(
            cv_result_path,
            dtype={
                "CV_Signature":
                    str,

                "Fold_Key":
                    str
            }
        )


        previous_cv = previous_cv[
            previous_cv[
                "CV_Signature"
            ].astype(
                str
            )
            == CV_SIGNATURE
        ].copy()


        if len(
            previous_cv
        ) > 0:

            cv_records = previous_cv.to_dict(
                orient="records"
            )


    except Exception as error:

        print(
            "Could not restore previous repeated CV file:"
        )


        print(
            error
        )


        cv_records = []


completed_cv_keys = {

    str(
        record[
            "Fold_Key"
        ]
    )

    for record
    in cv_records
}


print(
    "Previously completed CV folds:",
    len(
        completed_cv_keys
    )
)


# =============================================================================
# LIGHTGBM REPEATED CV PARAMETERS
# =============================================================================

def make_repeated_cv_lgb_params(
    seed
):

    parameters = {

        "objective":
            "multiclass",

        "num_class":
            NUM_CLASSES,

        "metric":
            "multi_logloss",

        "boosting_type":
            "gbdt",

        "device_type":
            "cpu",

        "force_col_wise":
            True,

        "num_threads":
            N_JOBS,

        "verbosity":
            -1,

        "bagging_freq":
            1,

        "seed":
            seed,

        "bagging_seed":
            seed,

        "feature_fraction_seed":
            seed,

        "data_random_seed":
            seed
    }


    parameters.update(
        LGB_BEST_PARAMS
    )


    return parameters


# =============================================================================
# XGBOOST REPEATED CV PARAMETERS
# =============================================================================

def make_repeated_cv_xgb_params(
    seed
):

    return {

        **XGB_BEST_PARAMS,

        "objective":
            "multi:softprob",

        "num_class":
            NUM_CLASSES,

        "eval_metric":
            "mlogloss",

        "tree_method":
            "hist",

        "n_jobs":
            N_JOBS,

        "verbosity":
            0,

        "random_state":
            seed
    }


# =============================================================================
# RUN REPEATED CV
# =============================================================================

for repeat_number in range(
    1,
    REPEATED_CV_REPEATS
    + 1
):

    splitter = StratifiedGroupKFold(

        n_splits=
            REPEATED_CV_FOLDS,

        shuffle=
            True,

        random_state=
            SEED
            + repeat_number
    )


    for fold_number, (
        training_index,
        validation_index

    ) in enumerate(
        splitter.split(
            X_dev_raw,
            y_dev,
            g_dev
        ),
        start=1
    ):

        fold_key = (
            f"Repeat_{repeat_number:02d}_"
            f"Fold_{fold_number:02d}"
        )


        # =====================================================================
        # RESUME
        # =====================================================================

        if fold_key in completed_cv_keys:

            print(
                fold_key,
                "already completed."
            )


            continue


        fold_start = time.time()


        # =====================================================================
        # VERIFY GROUP INDEPENDENCE
        # =====================================================================

        training_groups = set(
            g_dev[
                training_index
            ].tolist()
        )


        validation_groups = set(
            g_dev[
                validation_index
            ].tolist()
        )


        if len(
            training_groups.intersection(
                validation_groups
            )
        ) != 0:

            raise RuntimeError(

                f"Participant leakage detected in "
                f"{fold_key}."
            )


        # =====================================================================
        # FOLD SPECIFIC SCALER
        # =====================================================================

        scaler = StandardScaler()


        X_fold_train = scaler.fit_transform(
            X_dev_raw[
                training_index
            ]
        ).astype(
            np.float32,
            copy=False
        )


        X_fold_validation = scaler.transform(
            X_dev_raw[
                validation_index
            ]
        ).astype(
            np.float32,
            copy=False
        )


        X_fold_train = np.ascontiguousarray(
            X_fold_train
        )


        X_fold_validation = np.ascontiguousarray(
            X_fold_validation
        )


        y_fold_train = np.ascontiguousarray(
            y_dev[
                training_index
            ],
            dtype=np.int32
        )


        y_fold_validation = np.ascontiguousarray(
            y_dev[
                validation_index
            ],
            dtype=np.int32
        )


        # =====================================================================
        # LIGHTGBM
        # =====================================================================

        lgb_params = make_repeated_cv_lgb_params(
            SEED
            + repeat_number
            * 100
            + fold_number
        )


        lgb_weight = compute_sample_weight(
            "balanced",
            y_fold_train
        )


        lgb_train_data = lgb.Dataset(
            X_fold_train,
            label=
                y_fold_train,
            weight=
                lgb_weight,
            free_raw_data=True
        )


        fold_lgb_model = lgb.train(
            lgb_params,
            lgb_train_data,
            num_boost_round=
                LGB_FIXED_ROUNDS,
            callbacks=[
                lgb.log_evaluation(
                    0
                )
            ]
        )


        fold_lgb_probability = np.asarray(
            fold_lgb_model.predict(
                X_fold_validation
            ),
            dtype=np.float64
        )


        fold_lgb_prediction = np.argmax(
            fold_lgb_probability,
            axis=1
        ).astype(
            np.int32
        )


        lgb_accuracy = float(
            accuracy_score(
                y_fold_validation,
                fold_lgb_prediction
            )
        )


        lgb_f1 = float(
            f1_score(
                y_fold_validation,
                fold_lgb_prediction,
                average="weighted",
                zero_division=0
            )
        )


        lgb_mcc = float(
            matthews_corrcoef(
                y_fold_validation,
                fold_lgb_prediction
            )
        )


        # =====================================================================
        # XGBOOST
        # =====================================================================

        xgb_params = make_repeated_cv_xgb_params(
            SEED
            + repeat_number
            * 100
            + fold_number
        )


        xgb_weight = compute_sample_weight(
            "balanced",
            y_fold_train
        )


        fold_xgb_model = XGBClassifier(
            **xgb_params
        )


        fold_xgb_model.fit(
            X_fold_train,
            y_fold_train,
            sample_weight=
                xgb_weight
        )


        fold_xgb_probability = np.asarray(
            fold_xgb_model.predict_proba(
                X_fold_validation
            ),
            dtype=np.float64
        )


        fold_xgb_prediction = np.argmax(
            fold_xgb_probability,
            axis=1
        ).astype(
            np.int32
        )


        xgb_accuracy = float(
            accuracy_score(
                y_fold_validation,
                fold_xgb_prediction
            )
        )


        xgb_f1 = float(
            f1_score(
                y_fold_validation,
                fold_xgb_prediction,
                average="weighted",
                zero_division=0
            )
        )


        xgb_mcc = float(
            matthews_corrcoef(
                y_fold_validation,
                fold_xgb_prediction
            )
        )


        # =====================================================================
        # RECORD
        # =====================================================================

        record = {

            "CV_Signature":
                CV_SIGNATURE,

            "Repeat":
                repeat_number,

            "Fold":
                fold_number,

            "Fold_Key":
                fold_key,

            "Training_Participants":
                int(
                    len(
                        training_groups
                    )
                ),

            "Validation_Participants":
                int(
                    len(
                        validation_groups
                    )
                ),

            "Training_Rows":
                int(
                    len(
                        training_index
                    )
                ),

            "Validation_Rows":
                int(
                    len(
                        validation_index
                    )
                ),

            "LightGBM_Accuracy":
                lgb_accuracy,

            "XGBoost_Accuracy":
                xgb_accuracy,

            "LightGBM_F1_Weighted":
                lgb_f1,

            "XGBoost_F1_Weighted":
                xgb_f1,

            "LightGBM_MCC":
                lgb_mcc,

            "XGBoost_MCC":
                xgb_mcc,

            "Accuracy_Difference_LGB_minus_XGB":
                (
                    lgb_accuracy
                    - xgb_accuracy
                ),

            "Execution_Time_Seconds":
                float(
                    time.time()
                    - fold_start
                )
        }


        cv_records.append(
            record
        )


        completed_cv_keys.add(
            fold_key
        )


        # =====================================================================
        # SAVE AFTER EVERY FOLD
        # =====================================================================

        save_csv(
            pd.DataFrame(
                cv_records
            ),
            cv_result_path
        )


        print(

            f"{fold_key} | "
            f"LGB={lgb_accuracy:.4f} | "
            f"XGB={xgb_accuracy:.4f} | "
            f"Difference="
            f"{lgb_accuracy - xgb_accuracy:.4f}"
        )


        # =====================================================================
        # CLEAN MEMORY
        # =====================================================================

        del scaler

        del X_fold_train

        del X_fold_validation

        del y_fold_train

        del y_fold_validation

        del lgb_weight

        del lgb_train_data

        del fold_lgb_model

        del fold_lgb_probability

        del fold_lgb_prediction

        del xgb_weight

        del fold_xgb_model

        del fold_xgb_probability

        del fold_xgb_prediction


        gc.collect()


# =============================================================================
# RELOAD AND VERIFY COMPLETE CV TABLE
# =============================================================================

cv_table = pd.read_csv(
    cv_result_path,
    dtype={
        "CV_Signature":
            str,

        "Fold_Key":
            str
    }
)


cv_table = cv_table[
    cv_table[
        "CV_Signature"
    ].astype(
        str
    )
    == CV_SIGNATURE
].copy()


cv_table = (

    cv_table

    .sort_values(
        [
            "Repeat",
            "Fold"
        ]
    )

    .drop_duplicates(
        subset=[
            "Fold_Key"
        ],
        keep="last"
    )

    .reset_index(
        drop=True
    )
)


expected_cv_results = (
    REPEATED_CV_REPEATS
    * REPEATED_CV_FOLDS
)


if len(
    cv_table
) != expected_cv_results:

    raise RuntimeError(

        f"Repeated CV incomplete. "
        f"Expected {expected_cv_results} folds, "
        f"but found {len(cv_table)}."
    )


save_csv(
    cv_table,
    cv_result_path
)


# =============================================================================
# REPEATED CV SUMMARY
# =============================================================================

cv_summary = pd.DataFrame(
    [

        {
            "Algorithm":
                "LightGBM",

            "Mean_Accuracy":
                float(
                    cv_table[
                        "LightGBM_Accuracy"
                    ].mean()
                ),

            "STD_Accuracy":
                float(
                    cv_table[
                        "LightGBM_Accuracy"
                    ].std(
                        ddof=0
                    )
                ),

            "Minimum_Accuracy":
                float(
                    cv_table[
                        "LightGBM_Accuracy"
                    ].min()
                ),

            "Maximum_Accuracy":
                float(
                    cv_table[
                        "LightGBM_Accuracy"
                    ].max()
                ),

            "Mean_F1_Weighted":
                float(
                    cv_table[
                        "LightGBM_F1_Weighted"
                    ].mean()
                ),

            "Mean_MCC":
                float(
                    cv_table[
                        "LightGBM_MCC"
                    ].mean()
                ),

            "Number_of_Folds":
                int(
                    len(
                        cv_table
                    )
                )
        },

        {
            "Algorithm":
                "XGBoost",

            "Mean_Accuracy":
                float(
                    cv_table[
                        "XGBoost_Accuracy"
                    ].mean()
                ),

            "STD_Accuracy":
                float(
                    cv_table[
                        "XGBoost_Accuracy"
                    ].std(
                        ddof=0
                    )
                ),

            "Minimum_Accuracy":
                float(
                    cv_table[
                        "XGBoost_Accuracy"
                    ].min()
                ),

            "Maximum_Accuracy":
                float(
                    cv_table[
                        "XGBoost_Accuracy"
                    ].max()
                ),

            "Mean_F1_Weighted":
                float(
                    cv_table[
                        "XGBoost_F1_Weighted"
                    ].mean()
                ),

            "Mean_MCC":
                float(
                    cv_table[
                        "XGBoost_MCC"
                    ].mean()
                ),

            "Number_of_Folds":
                int(
                    len(
                        cv_table
                    )
                )
        }
    ]
)


cv_summary.to_csv(
    cv_dir
    / "repeated_cv_summary.csv",
    index=False
)


# =============================================================================
# PAIRED REPEATED CV DIFFERENCE
# =============================================================================

cv_difference = (
    cv_table[
        "LightGBM_Accuracy"
    ].to_numpy(
        dtype=np.float64
    )

    -

    cv_table[
        "XGBoost_Accuracy"
    ].to_numpy(
        dtype=np.float64
    )
)


pd.DataFrame(
    {

        "Fold_Key":
            cv_table[
                "Fold_Key"
            ],

        "Difference_LGB_minus_XGB":
            cv_difference
    }
).to_csv(
    cv_dir
    / "paired_cv_accuracy_difference.csv",
    index=False
)


# =============================================================================
# CORRECTED MATPLOTLIB BOXPLOT
#
# No labels argument is passed to boxplot.
#
# This avoids:
#
# TypeError:
# Axes.boxplot() got an unexpected keyword argument 'labels'
#
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        5
    )
)


boxplot_data = [

    cv_table[
        "LightGBM_Accuracy"
    ].to_numpy(
        dtype=np.float64
    ),

    cv_table[
        "XGBoost_Accuracy"
    ].to_numpy(
        dtype=np.float64
    )
]


ax.boxplot(
    boxplot_data
)


ax.set_xticks(
    [
        1,
        2
    ]
)


ax.set_xticklabels(
    [
        "LightGBM",
        "XGBoost"
    ]
)


ax.set_ylabel(
    "Accuracy"
)


ax.set_xlabel(
    "Algorithm"
)


ax.set_title(
    "Repeated Stratified Group Cross Validation"
)


ax.grid(
    axis="y",
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    cv_dir
    / "repeated_cv_accuracy_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# =============================================================================
# REPEATED CV FOLD COMPARISON CURVE
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        12,
        5
    )
)


x_positions = np.arange(
    1,
    len(
        cv_table
    )
    + 1
)


ax.plot(
    x_positions,
    cv_table[
        "LightGBM_Accuracy"
    ].to_numpy(),
    marker="o",
    label="LightGBM"
)


ax.plot(
    x_positions,
    cv_table[
        "XGBoost_Accuracy"
    ].to_numpy(),
    marker="o",
    label="XGBoost"
)


ax.set_xlabel(
    "Repeated CV Fold"
)


ax.set_ylabel(
    "Accuracy"
)


ax.set_title(
    "Repeated Stratified Group CV Accuracy by Fold"
)


ax.legend()


ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    cv_dir
    / "repeated_cv_fold_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


add_summary(

    "Repeated Stratified Cross Validation",

    "LightGBM",

    statistic=
        float(
            cv_table[
                "LightGBM_Accuracy"
            ].mean()
        ),

    interpretation=
        (
            f"{REPEATED_CV_REPEATS} repeats x "
            f"{REPEATED_CV_FOLDS} StratifiedGroupKFold."
        )
)


add_summary(

    "Repeated Stratified Cross Validation",

    "XGBoost",

    statistic=
        float(
            cv_table[
                "XGBoost_Accuracy"
            ].mean()
        ),

    interpretation=
        (
            f"{REPEATED_CV_REPEATS} repeats x "
            f"{REPEATED_CV_FOLDS} StratifiedGroupKFold."
        )
)


# =============================================================================
# 13 BRIER SCORE
#
# MULTICLASS BRIER SCORE
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "13 BRIER SCORE"
)


print(
    "=" * 100
)


brier_dir = METHOD_DIRS[
    "brier"
]


y_one_hot = np.eye(
    NUM_CLASSES,
    dtype=np.float64
)[
    y_test
]


def multiclass_brier_score(
    probability
):

    return float(
        np.mean(
            np.sum(
                (
                    probability
                    - y_one_hot
                )
                ** 2,
                axis=1
            )
        )
    )


lgb_brier = multiclass_brier_score(
    lgb_probability
)


xgb_brier = multiclass_brier_score(
    xgb_probability
)


brier_class_records = []


for class_index, class_name in enumerate(
    CLASS_NAMES
):

    binary_truth = (
        y_test
        == class_index
    ).astype(
        np.float64
    )


    lgb_class_brier = float(
        np.mean(
            (
                lgb_probability[
                    :,
                    class_index
                ]
                - binary_truth
            )
            ** 2
        )
    )


    xgb_class_brier = float(
        np.mean(
            (
                xgb_probability[
                    :,
                    class_index
                ]
                - binary_truth
            )
            ** 2
        )
    )


    brier_class_records.append(
        {

            "Class":
                class_name,

            "LightGBM_Brier":
                lgb_class_brier,

            "XGBoost_Brier":
                xgb_class_brier
        }
    )


pd.DataFrame(
    [

        {
            "Algorithm":
                "LightGBM",

            "Multiclass_Brier_Score":
                lgb_brier
        },

        {
            "Algorithm":
                "XGBoost",

            "Multiclass_Brier_Score":
                xgb_brier
        }
    ]
).to_csv(
    brier_dir
    / "multiclass_brier_scores.csv",
    index=False
)


pd.DataFrame(
    brier_class_records
).to_csv(
    brier_dir
    / "classwise_brier_scores.csv",
    index=False
)


add_summary(

    "Brier Score",

    "LightGBM",

    statistic=
        lgb_brier,

    interpretation=
        "Lower multiclass Brier score indicates better probability quality."
)


add_summary(

    "Brier Score",

    "XGBoost",

    statistic=
        xgb_brier,

    interpretation=
        "Lower multiclass Brier score indicates better probability quality."
)


# =============================================================================
# 14 CALIBRATION CURVE
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "14 CALIBRATION CURVE"
)


print(
    "=" * 100
)


calibration_dir = METHOD_DIRS[
    "calibration"
]


calibration_records = []


for class_index, class_name in enumerate(
    CLASS_NAMES
):

    binary_truth = (
        y_test
        == class_index
    ).astype(
        np.int32
    )


    fig, ax = plt.subplots(
        figsize=(
            7,
            6
        )
    )


    for model_name, probability in [

        (
            "LightGBM",
            lgb_probability
        ),

        (
            "XGBoost",
            xgb_probability
        )
    ]:

        try:

            (
                fraction_positive,
                mean_predicted

            ) = calibration_curve(
                binary_truth,
                probability[
                    :,
                    class_index
                ],
                n_bins=
                    CALIBRATION_BINS,
                strategy=
                    "quantile"
            )


            for bin_number, (
                predicted_probability,
                observed_frequency

            ) in enumerate(
                zip(
                    mean_predicted,
                    fraction_positive
                ),
                start=1
            ):

                calibration_records.append(
                    {

                        "Class":
                            class_name,

                        "Algorithm":
                            model_name,

                        "Bin":
                            bin_number,

                        "Mean_Predicted_Probability":
                            float(
                                predicted_probability
                            ),

                        "Observed_Frequency":
                            float(
                                observed_frequency
                            )
                    }
                )


            ax.plot(
                mean_predicted,
                fraction_positive,
                marker="o",
                label=
                    model_name
            )


        except Exception as error:

            calibration_records.append(
                {

                    "Class":
                        class_name,

                    "Algorithm":
                        model_name,

                    "Bin":
                        np.nan,

                    "Mean_Predicted_Probability":
                        np.nan,

                    "Observed_Frequency":
                        np.nan,

                    "Error":
                        str(
                            error
                        )
                }
            )


    ax.plot(
        [
            0,
            1
        ],
        [
            0,
            1
        ],
        linestyle=":"
    )


    ax.set_xlabel(
        "Mean Predicted Probability"
    )


    ax.set_ylabel(
        "Observed Frequency"
    )


    ax.set_title(
        f"Calibration Curve: {class_name}"
    )


    ax.legend()


    ax.grid(
        alpha=0.3
    )


    fig.tight_layout()


    fig.savefig(
        calibration_dir
        / f"Calibration_{class_name}.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


calibration_table = pd.DataFrame(
    calibration_records
)


calibration_table.to_csv(
    calibration_dir
    / "classwise_calibration_curve_data.csv",
    index=False
)


# =============================================================================
# EXPECTED CALIBRATION ERROR
# =============================================================================

def confidence_calibration(
    prediction,
    probability,
    y_true,
    bins=10
):

    confidence = np.max(
        probability,
        axis=1
    )


    correctness = (
        prediction
        == y_true
    ).astype(
        np.float64
    )


    bin_edges = np.linspace(
        0.0,
        1.0,
        bins
        + 1
    )


    records = []


    ece = 0.0

    mce = 0.0


    for bin_number in range(
        bins
    ):

        lower = bin_edges[
            bin_number
        ]


        upper = bin_edges[
            bin_number
            + 1
        ]


        if bin_number == bins - 1:

            index = (
                confidence
                >= lower
            ) & (
                confidence
                <= upper
            )


        else:

            index = (
                confidence
                >= lower
            ) & (
                confidence
                < upper
            )


        count = int(
            np.sum(
                index
            )
        )


        if count == 0:

            continue


        mean_confidence = float(
            np.mean(
                confidence[
                    index
                ]
            )
        )


        mean_accuracy = float(
            np.mean(
                correctness[
                    index
                ]
            )
        )


        calibration_gap = abs(
            mean_accuracy
            - mean_confidence
        )


        ece += (
            count
            / len(
                y_true
            )
            * calibration_gap
        )


        mce = max(
            mce,
            calibration_gap
        )


        records.append(
            {

                "Bin":
                    bin_number
                    + 1,

                "Lower_Bound":
                    lower,

                "Upper_Bound":
                    upper,

                "Count":
                    count,

                "Mean_Confidence":
                    mean_confidence,

                "Accuracy":
                    mean_accuracy,

                "Absolute_Gap":
                    calibration_gap
            }
        )


    return (
        pd.DataFrame(
            records
        ),
        float(
            ece
        ),
        float(
            mce
        )
    )


(
    lgb_conf_calibration,
    lgb_ece,
    lgb_mce

) = confidence_calibration(
    lgb_prediction,
    lgb_probability,
    y_test,
    CALIBRATION_BINS
)


(
    xgb_conf_calibration,
    xgb_ece,
    xgb_mce

) = confidence_calibration(
    xgb_prediction,
    xgb_probability,
    y_test,
    CALIBRATION_BINS
)


lgb_conf_calibration.to_csv(
    calibration_dir
    / "LightGBM_confidence_calibration.csv",
    index=False
)


xgb_conf_calibration.to_csv(
    calibration_dir
    / "XGBoost_confidence_calibration.csv",
    index=False
)


calibration_error_summary = pd.DataFrame(
    [

        {
            "Algorithm":
                "LightGBM",

            "ECE":
                lgb_ece,

            "MCE":
                lgb_mce
        },

        {
            "Algorithm":
                "XGBoost",

            "ECE":
                xgb_ece,

            "MCE":
                xgb_mce
        }
    ]
)


calibration_error_summary.to_csv(
    calibration_dir
    / "calibration_error_summary.csv",
    index=False
)


# =============================================================================
# CONFIDENCE CALIBRATION PLOT
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        7,
        6
    )
)


if len(
    lgb_conf_calibration
) > 0:

    ax.plot(
        lgb_conf_calibration[
            "Mean_Confidence"
        ],
        lgb_conf_calibration[
            "Accuracy"
        ],
        marker="o",
        label="LightGBM"
    )


if len(
    xgb_conf_calibration
) > 0:

    ax.plot(
        xgb_conf_calibration[
            "Mean_Confidence"
        ],
        xgb_conf_calibration[
            "Accuracy"
        ],
        marker="o",
        label="XGBoost"
    )


ax.plot(
    [
        0,
        1
    ],
    [
        0,
        1
    ],
    linestyle=":"
)


ax.set_xlabel(
    "Prediction Confidence"
)


ax.set_ylabel(
    "Observed Accuracy"
)


ax.set_title(
    "Overall Confidence Calibration"
)


ax.legend()


ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    calibration_dir
    / "overall_confidence_calibration.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


add_summary(

    "Calibration Curve",

    "LightGBM ECE",

    statistic=
        lgb_ece,

    interpretation=
        "Lower Expected Calibration Error indicates better calibration."
)


add_summary(

    "Calibration Curve",

    "XGBoost ECE",

    statistic=
        xgb_ece,

    interpretation=
        "Lower Expected Calibration Error indicates better calibration."
)


# =============================================================================
# 15 HOSMER LEMESHOW TEST
#
# ONE VERSUS REST FOR MULTICLASS CLASSIFICATION
# =============================================================================

print()


print(
    "=" * 100
)


print(
    "15 HOSMER LEMESHOW TEST"
)


print(
    "=" * 100
)


hosmer_dir = METHOD_DIRS[
    "hosmer"
]


def hosmer_lemeshow_binary(
    y_binary,
    probability,
    groups=10
):

    y_binary = np.asarray(
        y_binary,
        dtype=np.float64
    )


    probability = np.asarray(
        probability,
        dtype=np.float64
    )


    # =========================================================================
    # Clip probabilities only for numerical safety.
    # =========================================================================

    probability = np.clip(
        probability,
        1e-12,
        1.0
        - 1e-12
    )


    sorting_order = np.argsort(
        probability
    )


    split_indices = np.array_split(
        sorting_order,
        groups
    )


    statistic = 0.0

    records = []


    actual_groups = 0


    for group_number, index in enumerate(
        split_indices,
        start=1
    ):

        if len(
            index
        ) == 0:

            continue


        actual_groups += 1


        observed_positive = float(
            np.sum(
                y_binary[
                    index
                ]
            )
        )


        expected_positive = float(
            np.sum(
                probability[
                    index
                ]
            )
        )


        group_size = int(
            len(
                index
            )
        )


        observed_negative = float(
            group_size
            - observed_positive
        )


        expected_negative = float(
            group_size
            - expected_positive
        )


        positive_term = (

            (
                observed_positive
                - expected_positive
            )
            ** 2

            /

            max(
                expected_positive,
                1e-12
            )
        )


        negative_term = (

            (
                observed_negative
                - expected_negative
            )
            ** 2

            /

            max(
                expected_negative,
                1e-12
            )
        )


        statistic += (
            positive_term
            + negative_term
        )


        records.append(
            {

                "Group":
                    group_number,

                "Rows":
                    group_size,

                "Minimum_Probability":
                    float(
                        np.min(
                            probability[
                                index
                            ]
                        )
                    ),

                "Maximum_Probability":
                    float(
                        np.max(
                            probability[
                                index
                            ]
                        )
                    ),

                "Mean_Probability":
                    float(
                        np.mean(
                            probability[
                                index
                            ]
                        )
                    ),

                "Observed_Positive":
                    observed_positive,

                "Expected_Positive":
                    expected_positive,

                "Observed_Negative":
                    observed_negative,

                "Expected_Negative":
                    expected_negative
            }
        )


    degrees_freedom = max(
        actual_groups
        - 2,
        1
    )


    p_value = float(
        chi2.sf(
            statistic,
            degrees_freedom
        )
    )


    return (

        float(
            statistic
        ),

        int(
            degrees_freedom
        ),

        p_value,

        pd.DataFrame(
            records
        )
    )


hl_records = []


for model_name, probability in [

    (
        "LightGBM",
        lgb_probability
    ),

    (
        "XGBoost",
        xgb_probability
    )
]:

    for class_index, class_name in enumerate(
        CLASS_NAMES
    ):

        binary_truth = (
            y_test
            == class_index
        ).astype(
            np.int32
        )


        (
            hl_statistic,
            hl_df,
            hl_p,
            hl_group_table

        ) = hosmer_lemeshow_binary(

            binary_truth,

            probability[
                :,
                class_index
            ],

            groups=
                HL_GROUPS
        )


        hl_group_table.to_csv(
            hosmer_dir
            / (
                f"{model_name}_"
                f"{class_name}_"
                f"HL_groups.csv"
            ),
            index=False
        )


        hl_records.append(
            {

                "Algorithm":
                    model_name,

                "Class":
                    class_name,

                "HL_Statistic":
                    hl_statistic,

                "Degrees_of_Freedom":
                    hl_df,

                "P_Value":
                    hl_p,

                "Method":
                    (
                        "One versus rest "
                        "multiclass extension"
                    )
            }
        )


hl_table = pd.DataFrame(
    hl_records
)


hl_table[
    "Bonferroni_Adjusted_P"
] = np.minimum(
    hl_table[
        "P_Value"
    ]
    * NUM_CLASSES,
    1.0
)


hl_table[
    "Significance_After_Bonferroni"
] = [

    significance_text(
        value
    )

    for value
    in hl_table[
        "Bonferroni_Adjusted_P"
    ]
]


hl_table.to_csv(
    hosmer_dir
    / "hosmer_lemeshow_one_vs_rest_results.csv",
    index=False
)


for model_name in [

    "LightGBM",

    "XGBoost"
]:

    model_hl = hl_table[
        hl_table[
            "Algorithm"
        ]
        == model_name
    ]


    minimum_adjusted_p = float(
        model_hl[
            "Bonferroni_Adjusted_P"
        ].min()
    )


    add_summary(

        "Hosmer Lemeshow Test",

        (
            f"{model_name} classwise "
            f"one versus rest"
        ),

        statistic=
            float(
                model_hl[
                    "HL_Statistic"
                ].mean()
            ),

        p_value=
            minimum_adjusted_p,

        interpretation=
            (
                "Classwise one versus rest Hosmer Lemeshow "
                "analysis with Bonferroni adjusted p values."
            )
    )


# =============================================================================
# FINAL STATISTICAL SUMMARY
# =============================================================================

summary_table = pd.DataFrame(
    STATISTICAL_SUMMARY
)


summary_table.to_csv(
    OUTPUT_ROOT
    / "Statistical_Test_Summary.csv",
    index=False
)


# =============================================================================
# BEST MODEL BY ORIGINAL 60 / 30 / 10 TEST ACCURACY
# =============================================================================

if (
    LGB_BASE_METRICS[
        "Accuracy"
    ]
    >
    XGB_BASE_METRICS[
        "Accuracy"
    ]
):

    BEST_MODEL = "LightGBM"


elif (
    XGB_BASE_METRICS[
        "Accuracy"
    ]
    >
    LGB_BASE_METRICS[
        "Accuracy"
    ]
):

    BEST_MODEL = "XGBoost"


else:

    BEST_MODEL = "Tie"


BEST_TEST_ACCURACY = max(

    LGB_BASE_METRICS[
        "Accuracy"
    ],

    XGB_BASE_METRICS[
        "Accuracy"
    ]
)


# =============================================================================
# REPEATED CV BEST MODEL
# =============================================================================

LGB_REPEATED_CV_MEAN = float(
    cv_table[
        "LightGBM_Accuracy"
    ].mean()
)


XGB_REPEATED_CV_MEAN = float(
    cv_table[
        "XGBoost_Accuracy"
    ].mean()
)


if (
    LGB_REPEATED_CV_MEAN
    >
    XGB_REPEATED_CV_MEAN
):

    REPEATED_CV_BEST_MODEL = (
        "LightGBM"
    )


elif (
    XGB_REPEATED_CV_MEAN
    >
    LGB_REPEATED_CV_MEAN
):

    REPEATED_CV_BEST_MODEL = (
        "XGBoost"
    )


else:

    REPEATED_CV_BEST_MODEL = (
        "Tie"
    )


# =============================================================================
# CALIBRATION BEST
# =============================================================================

if lgb_brier < xgb_brier:

    BEST_BRIER_MODEL = "LightGBM"


elif xgb_brier < lgb_brier:

    BEST_BRIER_MODEL = "XGBoost"


else:

    BEST_BRIER_MODEL = "Tie"


if lgb_ece < xgb_ece:

    BEST_CALIBRATION_MODEL = (
        "LightGBM"
    )


elif xgb_ece < lgb_ece:

    BEST_CALIBRATION_MODEL = (
        "XGBoost"
    )


else:

    BEST_CALIBRATION_MODEL = (
        "Tie"
    )


# =============================================================================
# OVERALL REPORT
# =============================================================================

overall_report = {

    "Selected_Split":
        (
            "Train 60 percent, "
            "Test 30 percent, "
            "Validation 10 percent"
        ),

    "Testing_Participants":
        int(
            len(
                TEST_PERSONS
            )
        ),

    "Testing_Rows":
        int(
            len(
                y_test
            )
        ),

    "LightGBM_Test_Accuracy":
        LGB_BASE_METRICS[
            "Accuracy"
        ],

    "XGBoost_Test_Accuracy":
        XGB_BASE_METRICS[
            "Accuracy"
        ],

    "Best_Model_By_Test_Accuracy":
        BEST_MODEL,

    "Best_Test_Accuracy":
        BEST_TEST_ACCURACY,

    "Bootstrap_Accuracy_Difference_95_CI":
        [
            diff_ci_low,
            diff_ci_high
        ],

    "McNemar_Exact_P":
        exact_p,

    "Wilcoxon_P":
        wilcoxon_p,

    "Paired_t_P":
        paired_t_p,

    "Permutation_P":
        permutation_p,

    "LightGBM_Cohen_Kappa":
        lgb_kappa,

    "XGBoost_Cohen_Kappa":
        xgb_kappa,

    "LightGBM_Brier":
        lgb_brier,

    "XGBoost_Brier":
        xgb_brier,

    "Best_Brier_Model":
        BEST_BRIER_MODEL,

    "LightGBM_ECE":
        lgb_ece,

    "XGBoost_ECE":
        xgb_ece,

    "Best_Calibration_Model":
        BEST_CALIBRATION_MODEL,

    "Repeated_CV_LightGBM_Mean_Accuracy":
        LGB_REPEATED_CV_MEAN,

    "Repeated_CV_LightGBM_STD":
        float(
            cv_table[
                "LightGBM_Accuracy"
            ].std(
                ddof=0
            )
        ),

    "Repeated_CV_XGBoost_Mean_Accuracy":
        XGB_REPEATED_CV_MEAN,

    "Repeated_CV_XGBoost_STD":
        float(
            cv_table[
                "XGBoost_Accuracy"
            ].std(
                ddof=0
            )
        ),

    "Repeated_CV_Best_Model":
        REPEATED_CV_BEST_MODEL,

    "Friedman_Test":
        "Not Applicable with only two models",

    "Nemenyi_Test":
        "Not Applicable with only two models",

    "Cochran_Q_Test":
        "Not Applicable with only two models",

    "DeLong_Method":
        (
            "One versus rest classwise comparison"
        ),

    "Hosmer_Lemeshow_Method":
        (
            "One versus rest classwise comparison"
        )
}


save_json(
    overall_report,
    OUTPUT_ROOT
    / "Overall_Testing_Report.json"
)


# =============================================================================
# FINAL TEXT SUMMARY
# =============================================================================

with open(
    OUTPUT_ROOT
    / "Final_Statistical_Summary.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        "FINAL MODEL STATISTICAL TESTING\n"
    )


    file.write(
        "========================================\n\n"
    )


    file.write(
        "Split: Train 60%, Test 30%, Validation 10%\n\n"
    )


    file.write(
        f"Test Participants: "
        f"{len(TEST_PERSONS)}\n"
    )


    file.write(
        f"Test Rows: "
        f"{len(y_test)}\n\n"
    )


    file.write(
        f"LightGBM Test Accuracy: "
        f"{LGB_BASE_METRICS['Accuracy']:.6f}\n"
    )


    file.write(
        f"XGBoost Test Accuracy: "
        f"{XGB_BASE_METRICS['Accuracy']:.6f}\n\n"
    )


    file.write(
        f"BEST MODEL BY TEST ACCURACY: "
        f"{BEST_MODEL}\n"
    )


    file.write(
        f"Best Test Accuracy: "
        f"{BEST_TEST_ACCURACY:.6f}\n\n"
    )


    file.write(
        f"Bootstrap Difference 95% CI: "
        f"[{diff_ci_low:.6f}, "
        f"{diff_ci_high:.6f}]\n"
    )


    file.write(
        f"McNemar Exact p: "
        f"{exact_p:.10f}\n"
    )


    file.write(
        f"Wilcoxon p: "
        f"{wilcoxon_p:.10f}\n"
    )


    file.write(
        f"Paired t Test p: "
        f"{paired_t_p:.10f}\n"
    )


    file.write(
        f"Permutation p: "
        f"{permutation_p:.10f}\n\n"
    )


    file.write(
        f"LightGBM Cohen Kappa: "
        f"{lgb_kappa:.6f}\n"
    )


    file.write(
        f"XGBoost Cohen Kappa: "
        f"{xgb_kappa:.6f}\n\n"
    )


    file.write(
        f"LightGBM Brier Score: "
        f"{lgb_brier:.6f}\n"
    )


    file.write(
        f"XGBoost Brier Score: "
        f"{xgb_brier:.6f}\n"
    )


    file.write(
        f"Best Brier Model: "
        f"{BEST_BRIER_MODEL}\n\n"
    )


    file.write(
        f"LightGBM ECE: "
        f"{lgb_ece:.6f}\n"
    )


    file.write(
        f"XGBoost ECE: "
        f"{xgb_ece:.6f}\n"
    )


    file.write(
        f"Best Calibration Model: "
        f"{BEST_CALIBRATION_MODEL}\n\n"
    )


    file.write(
        f"Repeated CV LightGBM: "
        f"{LGB_REPEATED_CV_MEAN:.6f} "
        f"plus or minus "
        f"{cv_table['LightGBM_Accuracy'].std(ddof=0):.6f}\n"
    )


    file.write(
        f"Repeated CV XGBoost: "
        f"{XGB_REPEATED_CV_MEAN:.6f} "
        f"plus or minus "
        f"{cv_table['XGBoost_Accuracy'].std(ddof=0):.6f}\n"
    )


    file.write(
        f"Repeated CV Best Model: "
        f"{REPEATED_CV_BEST_MODEL}\n\n"
    )


    file.write(
        "Friedman Test: Not Applicable\n"
    )


    file.write(
        "Nemenyi Test: Not Applicable\n"
    )


    file.write(
        "Cochran Q Test: Not Applicable\n"
    )


# =============================================================================
# SAVE BEST MODEL NAME
# =============================================================================

with open(
    OUTPUT_ROOT
    / "Best_Model_Name.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        f"BEST MODEL BY TEST ACCURACY: "
        f"{BEST_MODEL}\n"
    )


    file.write(
        f"BEST TEST ACCURACY: "
        f"{BEST_TEST_ACCURACY:.6f}\n"
    )


    file.write(
        f"BEST MODEL BY REPEATED CV: "
        f"{REPEATED_CV_BEST_MODEL}\n"
    )


# =============================================================================
# FINAL EXECUTION TIME
# =============================================================================

TOTAL_TIME = (
    time.time()
    - TOTAL_START
)


# =============================================================================
# FINAL SCREEN OUTPUT
# =============================================================================

print()


print(
    "#" * 100
)


print(
    "FINAL STATISTICAL TEST RESULTS"
)


print(
    "#" * 100
)


print()


print(
    base_metrics_table
    .round(
        6
    )
    .to_string(
        index=False
    )
)


print()


print(
    "=" * 100
)


print(
    "BEST MODEL BY FINAL 60 / 30 / 10 TEST ACCURACY"
)


print(
    "=" * 100
)


print(
    "Model:",
    BEST_MODEL
)


print(
    "Accuracy:",
    f"{BEST_TEST_ACCURACY:.6f}"
)


print()


print(
    "=" * 100
)


print(
    "KEY PAIRED STATISTICAL TESTS"
)


print(
    "=" * 100
)


print(
    "Bootstrap difference 95% CI:",
    f"[{diff_ci_low:.6f}, "
    f"{diff_ci_high:.6f}]"
)


print(
    "McNemar exact p:",
    f"{exact_p:.10f}",
    "|",
    significance_text(
        exact_p
    )
)


print(
    "Wilcoxon p:",
    f"{wilcoxon_p:.10f}",
    "|",
    significance_text(
        wilcoxon_p
    )
)


print(
    "Paired t Test p:",
    f"{paired_t_p:.10f}",
    "|",
    significance_text(
        paired_t_p
    )
)


print(
    "Permutation p:",
    f"{permutation_p:.10f}",
    "|",
    significance_text(
        permutation_p
    )
)


print()


print(
    "=" * 100
)


print(
    "REPEATED STRATIFIED GROUP CROSS VALIDATION"
)


print(
    "=" * 100
)


print(
    "LightGBM:",
    f"{LGB_REPEATED_CV_MEAN:.6f}",
    "plus or minus",
    f"{cv_table['LightGBM_Accuracy'].std(ddof=0):.6f}"
)


print(
    "XGBoost:",
    f"{XGB_REPEATED_CV_MEAN:.6f}",
    "plus or minus",
    f"{cv_table['XGBoost_Accuracy'].std(ddof=0):.6f}"
)


print(
    "Best Repeated CV Model:",
    REPEATED_CV_BEST_MODEL
)


print()


print(
    "=" * 100
)


print(
    "PROBABILITY QUALITY"
)


print(
    "=" * 100
)


print(
    "LightGBM Brier:",
    f"{lgb_brier:.6f}"
)


print(
    "XGBoost Brier:",
    f"{xgb_brier:.6f}"
)


print(
    "Best Brier Model:",
    BEST_BRIER_MODEL
)


print()


print(
    "LightGBM ECE:",
    f"{lgb_ece:.6f}"
)


print(
    "XGBoost ECE:",
    f"{xgb_ece:.6f}"
)


print(
    "Best Calibration Model:",
    BEST_CALIBRATION_MODEL
)


print()


print(
    "=" * 100
)


print(
    "METHOD APPLICABILITY"
)


print(
    "=" * 100
)


print(
    "Friedman Test : Not applicable with only two models"
)


print(
    "Nemenyi Test  : Not applicable with only two models"
)


print(
    "Cochran Q Test: Not applicable with only two models"
)


print(
    "DeLong Test   : Applied classwise using one versus rest"
)


print(
    "Hosmer Lemeshow: Applied classwise using one versus rest"
)


print()


print(
    "Total execution time:",
    f"{TOTAL_TIME / 60:.2f}",
    "minutes"
)


print(
    "Current RAM:",
    f"{memory_mb():.2f}",
    "MB"
)


print()


print(
    "ALL RESULTS SAVED IN:"
)


print(
    OUTPUT_ROOT
)


print(
    "#" * 100
)

15 TESTING METHODS FOR FINAL LIGHTGBM AND XGBOOST MODELS
Input:
/home/rakib/Desktop/ml_apply/Final_Model_Test/Cleaned_oN_Selected_Max.csv

Output:
/home/rakib/Desktop/ml_apply/15_Testing apply_FinalModel

CPU workers: 8
Bootstrap iterations: 5000
Permutation iterations: 10000
Repeated grouped stratified CV: 5 repeats x 5 folds

LOADING PREVIOUS 60 / 30 / 10 MODELS
Classes: ['class_lecture', 'music', 'normal', 'reading']
LightGBM fixed rounds for repeated CV: 588

LOADING DATASET
Dataset shape: (149350, 232)
Feature count: 232
Participants: 60
RAM: 1158.25 MB

Training participants: 36
Validation participants: 6
Testing participants: 18

Final test rows: 45678
Repeated CV development rows: 103672

01 BOOTSTRAP 95 PERCENT CONFIDENCE INTERVAL

02 MCNEMAR TEST

03 WILCOXON SIGNED RANK TEST

04 PAIRED T TEST

05 FRIEDMAN TEST

06 NEMENYI TEST

07 DELONG TEST

08 PERMUTATION TEST

09 COCHRAN Q TEST

10 COHEN KAPPA

11 CHI SQUARE TEST

12 REPEATED STRATIFIED GROUP CROSS VALIDATION
Previously 

# Explainable AI Analysis Using SHAP and LIME

## Overview

This experiment applies Explainable Artificial Intelligence methods to the final selected **LightGBM** and **XGBoost** classification models.

The models were previously trained using the selected participant independent data distribution:

| Dataset Portion | Ratio |
|---|---:|
| Training | 60% |
| Testing | 30% |
| Validation | 10% |

The purpose of this analysis is to identify the EEG features that most strongly influence the predictions of each model and to provide both global and local explanations.

Two XAI methods are applied:

| XAI Method | Purpose |
|---|---|
| SHAP | Global and local feature contribution analysis |
| LIME | Local model explanation and aggregated feature importance |

The same trained LightGBM and XGBoost models from the final 60%, 30%, and 10% experiment are reused. No model retraining is performed during the XAI analysis.


# Dataset

The input dataset is:

```text
/home/rakib/Desktop/ml_apply/Final_Model_Test/Cleaned_oN_Selected_Max.csv

In [7]:
# =============================================================================
# FINAL TWO MODEL EXPLAINABLE AI
#
# Models
# LightGBM
# XGBoost
#
# XAI methods
# SHAP
# LIME
#
# Selected split
# Training   60 percent
# Testing    30 percent
# Validation 10 percent
#
# Input
# /home/rakib/Desktop/ml_apply/Final_Model_Test/
# Cleaned_oN_Selected_Max.csv
#
# Output
# /home/rakib/Desktop/ml_apply/Final_Two_Model_XAI/
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import gc
import json
import time
import math
import random
import hashlib
import warnings
import traceback
import re

from pathlib import Path

import joblib
import psutil
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import lightgbm as lgb

try:
    import shap
except ImportError as error:
    raise ImportError(
        "SHAP is not installed. Install it with: pip install shap"
    ) from error

try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError as error:
    raise ImportError(
        "LIME is not installed. Install it with: pip install lime"
    ) from error

from sklearn.metrics import accuracy_score


# =============================================================================
# WARNINGS
# =============================================================================

warnings.filterwarnings("ignore")


# =============================================================================
# GLOBAL CONFIGURATION
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# =============================================================================
# SHAP CONFIGURATION
#
# Set to None to explain the complete test set.
# =============================================================================

SHAP_MAX_SAMPLES = 6000

SHAP_TOP_FEATURES = 25

SHAP_CLASS_TOP_FEATURES = 20

SHAP_DEPENDENCE_FEATURES = 6

SHAP_LOCAL_MAX_DISPLAY = 18


# =============================================================================
# LIME CONFIGURATION
# =============================================================================

LIME_BACKGROUND_MAX_SAMPLES = 6000

LIME_EXPLANATIONS_PER_CLASS = 5

LIME_NUM_FEATURES = 15

LIME_NUM_SAMPLES = 3000


# =============================================================================
# VISUAL CONFIGURATION
# =============================================================================

FIGURE_DPI = 350

LIGHTGBM_COLOR = "#2563EB"
XGBOOST_COLOR = "#F97316"

POSITIVE_COLOR = "#10B981"
NEGATIVE_COLOR = "#EF4444"

ACCENT_COLOR = "#7C3AED"

NEUTRAL_COLOR = "#64748B"
GRID_COLOR = "#CBD5E1"


plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#CBD5E1",
        "axes.labelcolor": "#0F172A",
        "axes.titleweight": "bold",
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "xtick.color": "#334155",
        "ytick.color": "#334155",
        "font.size": 10,
        "legend.frameon": False,
        "savefig.facecolor": "white",
    }
)


# =============================================================================
# PATHS
# =============================================================================

INPUT_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "Final_Model_Test/"
    "Cleaned_oN_Selected_Max.csv"
)

PREVIOUS_ROOT = INPUT_PATH.parent


# =============================================================================
# LIGHTGBM SOURCE
# =============================================================================

LGB_SOURCE_DIR = (
    PREVIOUS_ROOT
    / "LightGBM_Multiple_Split_Results"
    / "Train60_Test30_CV10"
)


# =============================================================================
# XGBOOST SOURCE
# =============================================================================

XGB_SOURCE_DIR = (
    PREVIOUS_ROOT
    / "XGBoost_Multiple_Split_Results"
    / "Train60_Test30_CV10"
)


# =============================================================================
# OUTPUT ROOT
# =============================================================================

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "Final_Two_Model_XAI"
)

SHAP_ROOT = OUTPUT_ROOT / "SHAP"

LIME_ROOT = OUTPUT_ROOT / "LIME"

COMPARISON_ROOT = OUTPUT_ROOT / "XAI_Comparison"


for directory in [
    OUTPUT_ROOT,
    SHAP_ROOT,
    LIME_ROOT,
    COMPARISON_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# MODEL OUTPUT DIRECTORIES
# =============================================================================

LGB_SHAP_DIR = SHAP_ROOT / "LightGBM"

XGB_SHAP_DIR = SHAP_ROOT / "XGBoost"

LGB_LIME_DIR = LIME_ROOT / "LightGBM"

XGB_LIME_DIR = LIME_ROOT / "XGBoost"


for directory in [
    LGB_SHAP_DIR,
    XGB_SHAP_DIR,
    LGB_LIME_DIR,
    XGB_LIME_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# MEMORY
# =============================================================================

PROCESS = psutil.Process(
    os.getpid()
)


def memory_mb():
    return (
        PROCESS.memory_info().rss
        / 1024
        / 1024
    )


# =============================================================================
# TIMER
# =============================================================================

TOTAL_START = time.time()


# =============================================================================
# ATOMIC JSON SAVE
# =============================================================================

def save_json(
    data,
    path,
):

    path = Path(path)

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf8",
    ) as file:

        json.dump(
            data,
            file,
            indent=4,
            allow_nan=True,
        )

    os.replace(
        temporary_path,
        path,
    )


# =============================================================================
# ATOMIC CSV SAVE
# =============================================================================

def save_csv(
    dataframe,
    path,
):

    path = Path(path)

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


# =============================================================================
# SAVE FIGURE
# =============================================================================

def save_figure(
    figure,
    path_without_extension,
):

    path_without_extension = Path(
        path_without_extension
    )

    figure.savefig(
        str(path_without_extension) + ".png",
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    figure.savefig(
        str(path_without_extension) + ".svg",
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


# =============================================================================
# SAFE FILE NAME
# =============================================================================

def safe_filename(
    text,
):

    text = str(text)

    text = re.sub(
        r"[^A-Za-z0-9_.]+",
        "_",
        text,
    )

    return text.strip("_")


# =============================================================================
# NORMALIZE PARTICIPANT ID
# =============================================================================

def normalize_person_id(
    value,
):

    text = str(value).strip()

    try:

        numeric = float(text)

        if (
            np.isfinite(numeric)
            and numeric.is_integer()
        ):
            return str(
                int(numeric)
            )

    except Exception:
        pass

    return text


# =============================================================================
# SHORT LABEL
# =============================================================================

def short_label(
    text,
    width=48,
):

    text = str(text)

    if len(text) <= width:
        return text

    return (
        text[: width - 3]
        + "..."
    )


# =============================================================================
# FILE FINGERPRINT
# =============================================================================

def file_fingerprint(
    path,
):

    path = Path(path)

    information = path.stat()

    return {
        "path": str(path),
        "size": int(
            information.st_size
        ),
        "mtime_ns": int(
            information.st_mtime_ns
        ),
    }


# =============================================================================
# CHECK REQUIRED FILES
# =============================================================================

REQUIRED_FILES = [
    INPUT_PATH,

    LGB_SOURCE_DIR
    / "best_lightgbm_model.txt",

    LGB_SOURCE_DIR
    / "standard_scaler.joblib",

    LGB_SOURCE_DIR
    / "label_encoder.joblib",

    LGB_SOURCE_DIR
    / "participant_split.json",

    LGB_SOURCE_DIR
    / "final_metrics.json",

    XGB_SOURCE_DIR
    / "xgboost_model.joblib",

    XGB_SOURCE_DIR
    / "standard_scaler.joblib",

    XGB_SOURCE_DIR
    / "label_encoder.joblib",

    XGB_SOURCE_DIR
    / "participant_split.json",

    XGB_SOURCE_DIR
    / "final_metrics.json",
]


missing_files = [
    str(path)
    for path in REQUIRED_FILES
    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required model artifacts are missing:\n\n"
        + "\n".join(
            missing_files
        )
    )


# =============================================================================
# HEADER
# =============================================================================

print(
    "=" * 105
)

print(
    "FINAL TWO MODEL EXPLAINABLE AI"
)

print(
    "=" * 105
)

print(
    "Models:"
)

print(
    "LightGBM"
)

print(
    "XGBoost"
)

print()

print(
    "XAI:"
)

print(
    "SHAP"
)

print(
    "LIME"
)

print()

print(
    "Input:"
)

print(
    INPUT_PATH
)

print()

print(
    "Output:"
)

print(
    OUTPUT_ROOT
)

print()

print(
    "SHAP maximum global explanation samples:",
    SHAP_MAX_SAMPLES,
)

print(
    "LIME background samples:",
    LIME_BACKGROUND_MAX_SAMPLES,
)

print(
    "LIME explanations per class:",
    LIME_EXPLANATIONS_PER_CLASS,
)

print(
    "=" * 105
)


# =============================================================================
# LOAD SAVED ARTIFACTS
# =============================================================================

print()

print(
    "=" * 105
)

print(
    "LOADING FINAL MODEL ARTIFACTS"
)

print(
    "=" * 105
)


lgb_scaler = joblib.load(
    LGB_SOURCE_DIR
    / "standard_scaler.joblib"
)

xgb_scaler = joblib.load(
    XGB_SOURCE_DIR
    / "standard_scaler.joblib"
)


label_encoder = joblib.load(
    LGB_SOURCE_DIR
    / "label_encoder.joblib"
)

xgb_label_encoder = joblib.load(
    XGB_SOURCE_DIR
    / "label_encoder.joblib"
)


if not np.array_equal(
    label_encoder.classes_,
    xgb_label_encoder.classes_,
):

    raise ValueError(
        "The LightGBM and XGBoost class encoders differ."
    )


CLASS_NAMES = list(
    label_encoder.classes_
)

NUM_CLASSES = len(
    CLASS_NAMES
)


# =============================================================================
# LOAD PARTICIPANT SPLITS
# =============================================================================

with open(
    LGB_SOURCE_DIR
    / "participant_split.json",
    "r",
    encoding="utf8",
) as file:

    lgb_split = json.load(
        file
    )


with open(
    XGB_SOURCE_DIR
    / "participant_split.json",
    "r",
    encoding="utf8",
) as file:

    xgb_split = json.load(
        file
    )


def normalized_split(
    split_dictionary,
):

    return {
        key: sorted(
            [
                normalize_person_id(value)
                for value in values
            ]
        )
        for key, values
        in split_dictionary.items()
    }


if (
    normalized_split(lgb_split)
    != normalized_split(xgb_split)
):

    raise ValueError(
        "LightGBM and XGBoost participant splits are different."
    )


# =============================================================================
# LOAD FINAL METRICS
# =============================================================================

with open(
    LGB_SOURCE_DIR
    / "final_metrics.json",
    "r",
    encoding="utf8",
) as file:

    lgb_saved_metrics = json.load(
        file
    )


with open(
    XGB_SOURCE_DIR
    / "final_metrics.json",
    "r",
    encoding="utf8",
) as file:

    xgb_saved_metrics = json.load(
        file
    )


print(
    "Classes:",
    CLASS_NAMES,
)


# =============================================================================
# LOAD DATASET
# =============================================================================

print()

print(
    "=" * 105
)

print(
    "LOADING MAX DATASET"
)

print(
    "=" * 105
)


load_start = time.time()


df = pd.read_csv(
    INPUT_PATH,
    low_memory=False,
)


# =============================================================================
# COLUMN DEFINITIONS
# =============================================================================

METADATA_COLUMNS = [
    "PersonSrial",
    "Window",
    "Window_Start",
    "Window_End",
    "Window_Samples",
]

TARGET_COLUMN = "Class"


if "PersonSrial" not in df.columns:

    raise ValueError(
        "PersonSrial column is missing."
    )


if TARGET_COLUMN not in df.columns:

    raise ValueError(
        "Class column is missing."
    )


FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in METADATA_COLUMNS
    and column != TARGET_COLUMN
]


NUMBER_FEATURES = len(
    FEATURE_COLUMNS
)


# =============================================================================
# FEATURE MATRIX
# =============================================================================

X_all = (
    df[
        FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .to_numpy(
        dtype=np.float32
    )
)


if not np.isfinite(
    X_all
).all():

    raise ValueError(
        "Feature matrix contains NaN or infinite values."
    )


# =============================================================================
# LABELS
# =============================================================================

y_all = label_encoder.transform(
    df[
        TARGET_COLUMN
    ].astype(str)
).astype(
    np.int32
)


# =============================================================================
# PARTICIPANTS
# =============================================================================

groups_all = np.asarray(
    [
        normalize_person_id(value)
        for value
        in df[
            "PersonSrial"
        ].to_numpy()
    ],
    dtype=str,
)


print(
    "Dataset shape:",
    X_all.shape,
)

print(
    "Feature count:",
    NUMBER_FEATURES,
)

print(
    "Classes:",
    CLASS_NAMES,
)

print(
    "Participants:",
    len(
        np.unique(
            groups_all
        )
    ),
)

print(
    "Loading time:",
    f"{time.time() - load_start:.2f}",
    "seconds",
)

print(
    "RAM:",
    f"{memory_mb():.2f} MB",
)


# =============================================================================
# VERIFY FEATURE COUNT
# =============================================================================

if (
    int(
        lgb_scaler.n_features_in_
    )
    != NUMBER_FEATURES
):

    raise ValueError(
        "LightGBM scaler feature dimension does not match dataset."
    )


if (
    int(
        xgb_scaler.n_features_in_
    )
    != NUMBER_FEATURES
):

    raise ValueError(
        "XGBoost scaler feature dimension does not match dataset."
    )


# =============================================================================
# PARTICIPANT SPLITS
# =============================================================================

TRAIN_PERSONS = np.asarray(
    [
        normalize_person_id(person)
        for person
        in lgb_split[
            "training"
        ]
    ],
    dtype=str,
)


VALIDATION_PERSONS = np.asarray(
    [
        normalize_person_id(person)
        for person
        in lgb_split[
            "validation"
        ]
    ],
    dtype=str,
)


TEST_PERSONS = np.asarray(
    [
        normalize_person_id(person)
        for person
        in lgb_split[
            "testing"
        ]
    ],
    dtype=str,
)


train_mask = np.isin(
    groups_all,
    TRAIN_PERSONS,
)

validation_mask = np.isin(
    groups_all,
    VALIDATION_PERSONS,
)

test_mask = np.isin(
    groups_all,
    TEST_PERSONS,
)


# =============================================================================
# VERIFY PARTICIPANT INDEPENDENCE
# =============================================================================

if np.any(
    train_mask
    & validation_mask
):

    raise RuntimeError(
        "Training and validation participant overlap detected."
    )


if np.any(
    train_mask
    & test_mask
):

    raise RuntimeError(
        "Training and testing participant overlap detected."
    )


if np.any(
    validation_mask
    & test_mask
):

    raise RuntimeError(
        "Validation and testing participant overlap detected."
    )


# =============================================================================
# VERIFY ALL PARTICIPANTS FOUND
# =============================================================================

if set(
    np.unique(
        groups_all[
            train_mask
        ]
    )
) != set(
    TRAIN_PERSONS.tolist()
):

    raise RuntimeError(
        "Some saved training participants were not found."
    )


if set(
    np.unique(
        groups_all[
            validation_mask
        ]
    )
) != set(
    VALIDATION_PERSONS.tolist()
):

    raise RuntimeError(
        "Some saved validation participants were not found."
    )


if set(
    np.unique(
        groups_all[
            test_mask
        ]
    )
) != set(
    TEST_PERSONS.tolist()
):

    raise RuntimeError(
        "Some saved testing participants were not found."
    )


# =============================================================================
# TRAINING DATA FOR LIME
# =============================================================================

X_train_raw = np.ascontiguousarray(
    X_all[
        train_mask
    ],
    dtype=np.float32,
)

y_train = np.ascontiguousarray(
    y_all[
        train_mask
    ],
    dtype=np.int32,
)


# =============================================================================
# FINAL TEST DATA
# =============================================================================

X_test_raw = np.ascontiguousarray(
    X_all[
        test_mask
    ],
    dtype=np.float32,
)

y_test = np.ascontiguousarray(
    y_all[
        test_mask
    ],
    dtype=np.int32,
)

g_test = np.asarray(
    groups_all[
        test_mask
    ],
    dtype=str,
)


print()

print(
    "Training participants:",
    len(
        TRAIN_PERSONS
    ),
)

print(
    "Validation participants:",
    len(
        VALIDATION_PERSONS
    ),
)

print(
    "Testing participants:",
    len(
        TEST_PERSONS
    ),
)

print(
    "Training rows:",
    len(
        X_train_raw
    ),
)

print(
    "Final testing rows:",
    len(
        X_test_raw
    ),
)


# =============================================================================
# RELEASE ORIGINAL DATAFRAME AND FULL MATRICES
# =============================================================================

del df
del X_all
del y_all
del groups_all

del train_mask
del validation_mask
del test_mask

gc.collect()


# =============================================================================
# LOAD FINAL MODELS
# =============================================================================

print()

print(
    "=" * 105
)

print(
    "LOADING FINAL LIGHTGBM AND XGBOOST MODELS"
)

print(
    "=" * 105
)


lgb_model = lgb.Booster(
    model_file=str(
        LGB_SOURCE_DIR
        / "best_lightgbm_model.txt"
    )
)


xgb_model = joblib.load(
    XGB_SOURCE_DIR
    / "xgboost_model.joblib"
)


# =============================================================================
# SCALE TEST DATA
# =============================================================================

X_test_lgb = lgb_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False,
)

X_test_xgb = xgb_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False,
)


X_test_lgb = np.ascontiguousarray(
    X_test_lgb
)

X_test_xgb = np.ascontiguousarray(
    X_test_xgb
)


# =============================================================================
# FINAL PREDICTIONS
# =============================================================================

lgb_probability = np.asarray(
    lgb_model.predict(
        X_test_lgb
    ),
    dtype=np.float64,
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_test_xgb
    ),
    dtype=np.float64,
)


if lgb_probability.shape != (
    len(
        y_test
    ),
    NUM_CLASSES,
):

    raise RuntimeError(
        f"Unexpected LightGBM probability shape: "
        f"{lgb_probability.shape}"
    )


if xgb_probability.shape != (
    len(
        y_test
    ),
    NUM_CLASSES,
):

    raise RuntimeError(
        f"Unexpected XGBoost probability shape: "
        f"{xgb_probability.shape}"
    )


lgb_prediction = np.argmax(
    lgb_probability,
    axis=1,
).astype(
    np.int32
)

xgb_prediction = np.argmax(
    xgb_probability,
    axis=1,
).astype(
    np.int32
)


lgb_accuracy = float(
    accuracy_score(
        y_test,
        lgb_prediction,
    )
)

xgb_accuracy = float(
    accuracy_score(
        y_test,
        xgb_prediction,
    )
)


print(
    "LightGBM final test accuracy:",
    f"{lgb_accuracy:.6f}",
)

print(
    "XGBoost final test accuracy:",
    f"{xgb_accuracy:.6f}",
)

print(
    "RAM:",
    f"{memory_mb():.2f} MB",
)


# =============================================================================
# SAVE RUN CONFIGURATION
# =============================================================================

save_json(
    {
        "Input_Dataset":
            str(
                INPUT_PATH
            ),

        "Training_Participants":
            int(
                len(
                    TRAIN_PERSONS
                )
            ),

        "Validation_Participants":
            int(
                len(
                    VALIDATION_PERSONS
                )
            ),

        "Testing_Participants":
            int(
                len(
                    TEST_PERSONS
                )
            ),

        "Training_Rows":
            int(
                len(
                    X_train_raw
                )
            ),

        "Testing_Rows":
            int(
                len(
                    X_test_raw
                )
            ),

        "Features":
            int(
                NUMBER_FEATURES
            ),

        "Classes":
            CLASS_NAMES,

        "LightGBM_Test_Accuracy":
            lgb_accuracy,

        "XGBoost_Test_Accuracy":
            xgb_accuracy,

        "SHAP_Global_Max_Samples":
            SHAP_MAX_SAMPLES,

        "SHAP_Dependence_Features":
            SHAP_DEPENDENCE_FEATURES,

        "LIME_Background_Max_Samples":
            LIME_BACKGROUND_MAX_SAMPLES,

        "LIME_Explanations_Per_Class":
            LIME_EXPLANATIONS_PER_CLASS,

        "LIME_Num_Features":
            LIME_NUM_FEATURES,

        "LIME_Num_Samples":
            LIME_NUM_SAMPLES,
    },

    OUTPUT_ROOT
    / "XAI_Run_Configuration.json",
)


# =============================================================================
# CLASS BALANCED SAMPLING
# =============================================================================

def class_balanced_indices(
    labels,
    maximum_samples,
    seed,
):

    labels = np.asarray(
        labels
    )

    total_rows = len(
        labels
    )

    if (
        maximum_samples is None
        or maximum_samples >= total_rows
    ):

        return np.arange(
            total_rows,
            dtype=np.int64,
        )

    rng = np.random.default_rng(
        seed
    )

    classes = np.unique(
        labels
    )

    base_quota = (
        maximum_samples
        // len(
            classes
        )
    )

    remainder = (
        maximum_samples
        % len(
            classes
        )
    )

    selected = []

    for class_position, class_value in enumerate(
        classes
    ):

        available = np.where(
            labels
            == class_value
        )[0]

        quota = (
            base_quota
            + (
                1
                if class_position < remainder
                else 0
            )
        )

        quota = min(
            quota,
            len(
                available
            ),
        )

        chosen = rng.choice(
            available,
            size=quota,
            replace=False,
        )

        selected.extend(
            chosen.tolist()
        )

    selected = np.asarray(
        selected,
        dtype=np.int64,
    )

    if len(
        selected
    ) < maximum_samples:

        all_indices = np.arange(
            total_rows
        )

        unused = np.setdiff1d(
            all_indices,
            selected,
        )

        additional_needed = min(
            maximum_samples
            - len(
                selected
            ),
            len(
                unused
            ),
        )

        if additional_needed > 0:

            extra = rng.choice(
                unused,
                size=additional_needed,
                replace=False,
            )

            selected = np.concatenate(
                [
                    selected,
                    extra,
                ]
            )

    rng.shuffle(
        selected
    )

    return selected


# =============================================================================
# SELECT REPRESENTATIVE LOCAL CASES
# =============================================================================

def select_local_cases(
    y_true,
    prediction,
    probability,
    model_name,
):

    records = []

    for class_index, class_name in enumerate(
        CLASS_NAMES
    ):

        true_indices = np.where(
            y_true
            == class_index
        )[0]

        # =====================================================================
        # HIGH CONFIDENCE CORRECT CASE
        # =====================================================================

        correct_indices = true_indices[
            prediction[
                true_indices
            ]
            == class_index
        ]

        if len(
            correct_indices
        ) > 0:

            confidence = probability[
                correct_indices,
                class_index,
            ]

            chosen = correct_indices[
                np.argmax(
                    confidence
                )
            ]

            records.append(
                {
                    "Model":
                        model_name,

                    "Sample_Index":
                        int(
                            chosen
                        ),

                    "Case_Type":
                        "Correct_High_Confidence",

                    "True_Class_Index":
                        int(
                            class_index
                        ),

                    "True_Class":
                        class_name,

                    "Predicted_Class_Index":
                        int(
                            prediction[
                                chosen
                            ]
                        ),

                    "Predicted_Class":
                        CLASS_NAMES[
                            prediction[
                                chosen
                            ]
                        ],

                    "Prediction_Confidence":
                        float(
                            probability[
                                chosen,
                                prediction[
                                    chosen
                                ],
                            ]
                        ),

                    "PersonSrial":
                        str(
                            g_test[
                                chosen
                            ]
                        ),
                }
            )

        # =====================================================================
        # HIGH CONFIDENCE MISCLASSIFICATION
        # =====================================================================

        incorrect_indices = true_indices[
            prediction[
                true_indices
            ]
            != class_index
        ]

        if len(
            incorrect_indices
        ) > 0:

            incorrect_predictions = prediction[
                incorrect_indices
            ]

            confidence = probability[
                incorrect_indices,
                incorrect_predictions,
            ]

            chosen = incorrect_indices[
                np.argmax(
                    confidence
                )
            ]

            records.append(
                {
                    "Model":
                        model_name,

                    "Sample_Index":
                        int(
                            chosen
                        ),

                    "Case_Type":
                        "Misclassified_High_Confidence",

                    "True_Class_Index":
                        int(
                            class_index
                        ),

                    "True_Class":
                        class_name,

                    "Predicted_Class_Index":
                        int(
                            prediction[
                                chosen
                            ]
                        ),

                    "Predicted_Class":
                        CLASS_NAMES[
                            prediction[
                                chosen
                            ]
                        ],

                    "Prediction_Confidence":
                        float(
                            probability[
                                chosen,
                                prediction[
                                    chosen
                                ],
                            ]
                        ),

                    "PersonSrial":
                        str(
                            g_test[
                                chosen
                            ]
                        ),
                }
            )

    return pd.DataFrame(
        records
    )


# =============================================================================
# LOCAL CASE TABLES
# =============================================================================

lgb_local_cases = select_local_cases(
    y_test,
    lgb_prediction,
    lgb_probability,
    "LightGBM",
)

xgb_local_cases = select_local_cases(
    y_test,
    xgb_prediction,
    xgb_probability,
    "XGBoost",
)


local_cases_combined = pd.concat(
    [
        lgb_local_cases,
        xgb_local_cases,
    ],
    ignore_index=True,
)


local_cases_combined.to_csv(
    OUTPUT_ROOT
    / "Selected_Local_XAI_Cases.csv",
    index=False,
)


# =============================================================================
# SHAP SAMPLE
#
# Local representative cases are always included.
# =============================================================================

base_shap_indices = class_balanced_indices(
    y_test,
    SHAP_MAX_SAMPLES,
    SEED,
)


forced_local_indices = np.unique(
    local_cases_combined[
        "Sample_Index"
    ].to_numpy(
        dtype=np.int64
    )
)


SHAP_INDICES = np.unique(
    np.concatenate(
        [
            base_shap_indices,
            forced_local_indices,
        ]
    )
)


print()

print(
    "SHAP explanation rows:",
    len(
        SHAP_INDICES
    ),
)


# =============================================================================
# SAVE SHAP SAMPLE INFORMATION
# =============================================================================

shap_sample_table = pd.DataFrame(
    {
        "Sample_Index":
            SHAP_INDICES,

        "PersonSrial":
            g_test[
                SHAP_INDICES
            ],

        "True_Class":
            label_encoder.inverse_transform(
                y_test[
                    SHAP_INDICES
                ]
            ),

        "LightGBM_Prediction":
            label_encoder.inverse_transform(
                lgb_prediction[
                    SHAP_INDICES
                ]
            ),

        "XGBoost_Prediction":
            label_encoder.inverse_transform(
                xgb_prediction[
                    SHAP_INDICES
                ]
            ),
    }
)


shap_sample_table.to_csv(
    SHAP_ROOT
    / "SHAP_Explanation_Sample.csv",
    index=False,
)


# =============================================================================
# NORMALIZE MULTICLASS SHAP OUTPUT
#
# Expected result:
# samples x features x classes
# =============================================================================

def normalize_shap_values(
    raw_values,
    number_samples,
    number_features,
    number_classes,
):

    # =========================================================================
    # Older SHAP output
    #
    # list[class]
    # each entry = samples x features
    # =========================================================================

    if isinstance(
        raw_values,
        list,
    ):

        if len(
            raw_values
        ) != number_classes:

            raise ValueError(
                "Unexpected number of SHAP class arrays."
            )

        values = np.stack(
            [
                np.asarray(
                    item
                )
                for item
                in raw_values
            ],
            axis=2,
        )

        return values

    values = np.asarray(
        raw_values
    )

    # =========================================================================
    # samples x features x classes
    # =========================================================================

    if values.shape == (
        number_samples,
        number_features,
        number_classes,
    ):

        return values

    # =========================================================================
    # classes x samples x features
    # =========================================================================

    if values.shape == (
        number_classes,
        number_samples,
        number_features,
    ):

        return np.transpose(
            values,
            (
                1,
                2,
                0,
            ),
        )

    # =========================================================================
    # samples x classes x features
    # =========================================================================

    if values.shape == (
        number_samples,
        number_classes,
        number_features,
    ):

        return np.transpose(
            values,
            (
                0,
                2,
                1,
            ),
        )

    if (
        values.ndim == 2
        and values.shape == (
            number_samples,
            number_features,
        )
        and number_classes == 1
    ):

        return values[
            :,
            :,
            None,
        ]

    raise ValueError(
        f"Unsupported SHAP value shape: {values.shape}"
    )


# =============================================================================
# NORMALIZE SHAP EXPECTED VALUES
# =============================================================================

def normalize_expected_values(
    expected_value,
    number_classes,
):

    values = np.asarray(
        expected_value,
        dtype=np.float64,
    ).reshape(
        -1
    )

    if len(
        values
    ) == number_classes:

        return values

    if len(
        values
    ) == 1:

        return np.repeat(
            values,
            number_classes,
        )

    raise ValueError(
        "Unexpected SHAP expected value dimension."
    )


# =============================================================================
# BUILD TREE EXPLAINER
# =============================================================================

def build_shap_explainer(
    model_name,
    model,
):

    try:

        return shap.TreeExplainer(
            model
        )

    except Exception as first_error:

        if model_name == "XGBoost":

            try:

                return shap.TreeExplainer(
                    model.get_booster()
                )

            except Exception as second_error:

                raise RuntimeError(
                    "Unable to initialize XGBoost TreeExplainer."
                ) from second_error

        raise first_error


# =============================================================================
# GLOBAL SHAP BAR
# =============================================================================

def plot_global_shap_bar(
    importance_table,
    model_name,
    output_directory,
):

    top = (
        importance_table
        .sort_values(
            "Mean_Absolute_SHAP",
            ascending=False,
        )
        .head(
            SHAP_TOP_FEATURES
        )
        .sort_values(
            "Mean_Absolute_SHAP",
            ascending=True,
        )
    )

    model_color = (
        LIGHTGBM_COLOR
        if model_name == "LightGBM"
        else XGBOOST_COLOR
    )

    fig, ax = plt.subplots(
        figsize=(
            10,
            9,
        )
    )

    labels = [
        short_label(
            feature
        )
        for feature
        in top[
            "Feature"
        ]
    ]

    y_position = np.arange(
        len(
            top
        )
    )

    ax.barh(
        y_position,
        top[
            "Mean_Absolute_SHAP"
        ],
        color=model_color,
        alpha=0.88,
    )

    ax.set_yticks(
        y_position
    )

    ax.set_yticklabels(
        labels
    )

    ax.set_xlabel(
        "Mean absolute SHAP value"
    )

    ax.set_ylabel(
        "EEG feature"
    )

    ax.set_title(
        f"{model_name} Global SHAP Feature Importance"
    )

    ax.grid(
        axis="x",
        color=GRID_COLOR,
        alpha=0.45,
    )

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    fig.tight_layout()

    save_figure(
        fig,
        output_directory
        / "01_Global_SHAP_Feature_Importance",
    )


# =============================================================================
# CLASSWISE SHAP HEATMAP
# =============================================================================

def plot_classwise_shap_heatmap(
    class_importance,
    global_importance,
    model_name,
    output_directory,
):

    top_feature_indices = (
        global_importance
        .sort_values(
            "Mean_Absolute_SHAP",
            ascending=False,
        )
        .head(
            SHAP_CLASS_TOP_FEATURES
        )[
            "Feature_Index"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    matrix = (
        class_importance[
            top_feature_indices,
            :
        ].T
    )

    labels = [
        short_label(
            FEATURE_COLUMNS[
                feature_index
            ],
            width=36,
        )
        for feature_index
        in top_feature_indices
    ]

    fig, ax = plt.subplots(
        figsize=(
            14,
            5.5,
        )
    )

    image = ax.imshow(
        matrix,
        aspect="auto",
        cmap="viridis",
    )

    ax.set_xticks(
        np.arange(
            len(
                labels
            )
        )
    )

    ax.set_xticklabels(
        labels,
        rotation=55,
        ha="right",
    )

    ax.set_yticks(
        np.arange(
            NUM_CLASSES
        )
    )

    ax.set_yticklabels(
        CLASS_NAMES
    )

    ax.set_xlabel(
        "EEG feature"
    )

    ax.set_ylabel(
        "Class"
    )

    ax.set_title(
        f"{model_name} Classwise Mean Absolute SHAP"
    )

    color_bar = fig.colorbar(
        image,
        ax=ax,
        pad=0.02,
    )

    color_bar.set_label(
        "Mean absolute SHAP value"
    )

    fig.tight_layout()

    save_figure(
        fig,
        output_directory
        / "02_Classwise_SHAP_Heatmap",
    )


# =============================================================================
# SHAP DEPENDENCE PANEL
#
# CORRECTED VARIABLE NAME:
# SHAP_DEPENDENCE_FEATURES
# =============================================================================

def plot_shap_dependence_panel(
    shap_values,
    raw_data,
    predictions,
    global_importance,
    model_name,
    output_directory,
):

    number_to_plot = min(
        SHAP_DEPENDENCE_FEATURES,
        len(
            global_importance
        ),
    )

    top_features = (
        global_importance
        .sort_values(
            "Mean_Absolute_SHAP",
            ascending=False,
        )
        .head(
            number_to_plot
        )[
            "Feature_Index"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    # =========================================================================
    # Convert:
    #
    # samples x features x classes
    #
    # into:
    #
    # samples x classes x features
    # =========================================================================

    transposed = np.transpose(
        shap_values,
        (
            0,
            2,
            1,
        ),
    )

    predicted_class_shap = transposed[
        np.arange(
            len(
                predictions
            )
        ),
        predictions,
        :,
    ]

    columns = min(
        3,
        max(
            1,
            number_to_plot,
        ),
    )

    rows = int(
        math.ceil(
            number_to_plot
            / columns
        )
    )

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(
            5.4 * columns,
            4.6 * rows,
        ),
        squeeze=False,
    )

    axes = axes.flatten()

    scatter = None

    for plot_index, feature_index in enumerate(
        top_features
    ):

        ax = axes[
            plot_index
        ]

        scatter = ax.scatter(
            raw_data[
                :,
                feature_index
            ],
            predicted_class_shap[
                :,
                feature_index
            ],
            c=predictions,
            cmap="tab10",
            vmin=-0.5,
            vmax=NUM_CLASSES - 0.5,
            alpha=0.42,
            s=13,
            edgecolors="none",
        )

        ax.axhline(
            0.0,
            linewidth=1,
            color="#94A3B8",
        )

        ax.set_xlabel(
            short_label(
                FEATURE_COLUMNS[
                    feature_index
                ],
                width=32,
            )
        )

        ax.set_ylabel(
            "SHAP contribution"
        )

        ax.set_title(
            f"Rank {plot_index + 1}"
        )

        ax.grid(
            alpha=0.2
        )

        ax.spines[
            "top"
        ].set_visible(
            False
        )

        ax.spines[
            "right"
        ].set_visible(
            False
        )

    for unused_index in range(
        len(
            top_features
        ),
        len(
            axes
        ),
    ):

        axes[
            unused_index
        ].axis(
            "off"
        )

    if scatter is not None:

        color_bar = fig.colorbar(
            scatter,
            ax=axes.tolist(),
            fraction=0.025,
            pad=0.02,
        )

        color_bar.set_ticks(
            np.arange(
                NUM_CLASSES
            )
        )

        color_bar.set_ticklabels(
            CLASS_NAMES
        )

        color_bar.set_label(
            "Predicted class"
        )

    fig.suptitle(
        (
            f"{model_name} SHAP Dependence "
            f"for Top Global Features"
        ),
        fontsize=16,
        fontweight="bold",
        y=0.99,
    )

    fig.subplots_adjust(
        top=0.90,
        bottom=0.08,
        left=0.08,
        right=0.88,
        hspace=0.32,
        wspace=0.30,
    )

    save_figure(
        fig,
        output_directory
        / "03_SHAP_Dependence_Top_Features",
    )


# =============================================================================
# SHAP ANALYSIS ENGINE
# =============================================================================

def run_shap_analysis(
    model_name,
    model,
    model_path,
    scaler_path,
    X_test_scaled,
    prediction,
    probability,
    local_cases,
    output_directory,
):

    print()

    print(
        "=" * 105
    )

    print(
        f"SHAP ANALYSIS: {model_name}"
    )

    print(
        "=" * 105
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_file = (
        output_directory
        / "shap_values_checkpoint.npz"
    )

    metadata_file = (
        output_directory
        / "shap_checkpoint_metadata.json"
    )

    subset_hash = hashlib.sha256(
        SHAP_INDICES.tobytes()
    ).hexdigest()[
        :16
    ]

    # =========================================================================
    # Keep this signature compatible with the previous code.
    # =========================================================================

    signature_payload = {
        "Model":
            model_name,

        "Dataset":
            file_fingerprint(
                INPUT_PATH
            ),

        "Model_File":
            file_fingerprint(
                model_path
            ),

        "Scaler_File":
            file_fingerprint(
                scaler_path
            ),

        "Subset_Hash":
            subset_hash,

        "Samples":
            int(
                len(
                    SHAP_INDICES
                )
            ),

        "Features":
            FEATURE_COLUMNS,

        "Classes":
            CLASS_NAMES,
    }

    shap_signature = hashlib.sha256(
        json.dumps(
            signature_payload,
            sort_keys=True,
        ).encode(
            "utf8"
        )
    ).hexdigest()[
        :20
    ]

    shap_values = None

    base_values = None

    # =========================================================================
    # RESTORE SHAP CHECKPOINT
    # =========================================================================

    if (
        checkpoint_file.exists()
        and metadata_file.exists()
    ):

        try:

            with open(
                metadata_file,
                "r",
                encoding="utf8",
            ) as file:

                metadata = json.load(
                    file
                )

            if (
                metadata.get(
                    "Signature"
                )
                == shap_signature
            ):

                checkpoint = np.load(
                    checkpoint_file
                )

                shap_values = checkpoint[
                    "shap_values"
                ]

                base_values = checkpoint[
                    "base_values"
                ]

                saved_indices = checkpoint[
                    "indices"
                ]

                if not np.array_equal(
                    saved_indices,
                    SHAP_INDICES,
                ):

                    shap_values = None
                    base_values = None

                else:

                    print(
                        "Restored existing SHAP checkpoint."
                    )

        except Exception as error:

            print(
                "Unable to restore SHAP checkpoint:"
            )

            print(
                error
            )

            shap_values = None
            base_values = None

    # =========================================================================
    # COMPUTE SHAP ONLY WHEN CHECKPOINT IS NOT AVAILABLE
    # =========================================================================

    if shap_values is None:

        shap_start = time.time()

        print(
            "Building TreeExplainer..."
        )

        explainer = build_shap_explainer(
            model_name,
            model,
        )

        X_explain_scaled = np.ascontiguousarray(
            X_test_scaled[
                SHAP_INDICES
            ]
        )

        print(
            "Computing SHAP values for",
            len(
                SHAP_INDICES
            ),
            "test observations...",
        )

        try:

            raw_shap_values = explainer.shap_values(
                X_explain_scaled,
                check_additivity=False,
            )

        except TypeError:

            raw_shap_values = explainer.shap_values(
                X_explain_scaled
            )

        shap_values = normalize_shap_values(
            raw_shap_values,
            len(
                SHAP_INDICES
            ),
            NUMBER_FEATURES,
            NUM_CLASSES,
        )

        base_values = normalize_expected_values(
            explainer.expected_value,
            NUM_CLASSES,
        )

        shap_values = np.asarray(
            shap_values,
            dtype=np.float32,
        )

        base_values = np.asarray(
            base_values,
            dtype=np.float64,
        )

        # =====================================================================
        # SAVE CHECKPOINT
        # =====================================================================

        temporary_npz = checkpoint_file.with_name(
            checkpoint_file.name
            + ".tmp.npz"
        )

        np.savez_compressed(
            temporary_npz,
            shap_values=shap_values,
            base_values=base_values,
            indices=SHAP_INDICES,
        )

        os.replace(
            temporary_npz,
            checkpoint_file,
        )

        save_json(
            {
                "Signature":
                    shap_signature,

                "Model":
                    model_name,

                "Samples":
                    int(
                        len(
                            SHAP_INDICES
                        )
                    ),

                "Features":
                    int(
                        NUMBER_FEATURES
                    ),

                "Classes":
                    CLASS_NAMES,

                "SHAP_Output_Scale":
                    "TreeExplainer raw model output scale",

                "Computation_Seconds":
                    float(
                        time.time()
                        - shap_start
                    ),
            },
            metadata_file,
        )

        del raw_shap_values
        del X_explain_scaled
        del explainer

        gc.collect()

        print(
            "SHAP computation completed."
        )

    print(
        "SHAP shape:",
        shap_values.shape,
    )

    # =========================================================================
    # ORIGINAL FEATURE VALUES FOR DISPLAY
    # =========================================================================

    X_display = np.ascontiguousarray(
        X_test_raw[
            SHAP_INDICES
        ]
    )

    subset_prediction = prediction[
        SHAP_INDICES
    ]

    # =========================================================================
    # GLOBAL IMPORTANCE
    # =========================================================================

    global_mean_abs = np.mean(
        np.abs(
            shap_values
        ),
        axis=(
            0,
            2,
        ),
    )

    global_signed_mean = np.mean(
        shap_values,
        axis=(
            0,
            2,
        ),
    )

    global_table = pd.DataFrame(
        {
            "Feature_Index":
                np.arange(
                    NUMBER_FEATURES
                ),

            "Feature":
                FEATURE_COLUMNS,

            "Mean_Absolute_SHAP":
                global_mean_abs,

            "Mean_Signed_SHAP":
                global_signed_mean,
        }
    )

    global_table = (
        global_table
        .sort_values(
            "Mean_Absolute_SHAP",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    global_table[
        "Rank"
    ] = np.arange(
        1,
        len(
            global_table
        )
        + 1,
    )

    global_table.to_csv(
        output_directory
        / "Global_SHAP_Feature_Importance.csv",
        index=False,
    )

    # =========================================================================
    # CLASSWISE IMPORTANCE
    # =========================================================================

    class_importance = np.mean(
        np.abs(
            shap_values
        ),
        axis=0,
    )

    class_records = []

    for feature_index, feature_name in enumerate(
        FEATURE_COLUMNS
    ):

        for class_index, class_name in enumerate(
            CLASS_NAMES
        ):

            class_records.append(
                {
                    "Feature_Index":
                        feature_index,

                    "Feature":
                        feature_name,

                    "Class_Index":
                        class_index,

                    "Class":
                        class_name,

                    "Mean_Absolute_SHAP":
                        float(
                            class_importance[
                                feature_index,
                                class_index,
                            ]
                        ),
                }
            )

    class_table = pd.DataFrame(
        class_records
    )

    class_table.to_csv(
        output_directory
        / "Classwise_SHAP_Feature_Importance.csv",
        index=False,
    )

    # =========================================================================
    # GLOBAL BAR
    # =========================================================================

    plot_global_shap_bar(
        global_table,
        model_name,
        output_directory,
    )

    # =========================================================================
    # CLASSWISE HEATMAP
    # =========================================================================

    plot_classwise_shap_heatmap(
        class_importance,
        global_table,
        model_name,
        output_directory,
    )

    # =========================================================================
    # CORRECTED SHAP DEPENDENCE PANEL
    # =========================================================================

    plot_shap_dependence_panel(
        shap_values,
        X_display,
        subset_prediction,
        global_table,
        model_name,
        output_directory,
    )

    # =========================================================================
    # MULTICLASS SHAP BAR
    # =========================================================================

    try:

        class_shap_list = [
            shap_values[
                :,
                :,
                class_index,
            ]
            for class_index
            in range(
                NUM_CLASSES
            )
        ]

        shap.summary_plot(
            class_shap_list,
            X_display,
            feature_names=FEATURE_COLUMNS,
            class_names=CLASS_NAMES,
            plot_type="bar",
            max_display=SHAP_TOP_FEATURES,
            show=False,
        )

        figure = plt.gcf()

        figure.suptitle(
            (
                f"{model_name} Multiclass SHAP "
                f"Feature Importance"
            ),
            fontsize=15,
            fontweight="bold",
            y=1.01,
        )

        figure.tight_layout()

        save_figure(
            figure,
            output_directory
            / "04_Multiclass_SHAP_Stacked_Bar",
        )

    except Exception:

        with open(
            output_directory
            / "04_Multiclass_SHAP_Stacked_Bar_Error.txt",
            "w",
            encoding="utf8",
        ) as file:

            file.write(
                traceback.format_exc()
            )

        plt.close(
            "all"
        )

    # =========================================================================
    # CLASSWISE BEESWARM
    # =========================================================================

    class_beeswarm_directory = (
        output_directory
        / "Classwise_Beeswarm"
    )

    class_beeswarm_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    for class_index, class_name in enumerate(
        CLASS_NAMES
    ):

        try:

            shap.summary_plot(
                shap_values[
                    :,
                    :,
                    class_index,
                ],
                X_display,
                feature_names=FEATURE_COLUMNS,
                max_display=SHAP_CLASS_TOP_FEATURES,
                show=False,
            )

            figure = plt.gcf()

            figure.suptitle(
                (
                    f"{model_name} SHAP Beeswarm\n"
                    f"Class: {class_name}"
                ),
                fontsize=14,
                fontweight="bold",
                y=1.01,
            )

            figure.tight_layout()

            save_figure(
                figure,
                class_beeswarm_directory
                / (
                    f"SHAP_Beeswarm_"
                    f"{safe_filename(class_name)}"
                ),
            )

        except Exception:

            with open(
                class_beeswarm_directory
                / (
                    f"SHAP_Beeswarm_"
                    f"{safe_filename(class_name)}"
                    f"_Error.txt"
                ),
                "w",
                encoding="utf8",
            ) as file:

                file.write(
                    traceback.format_exc()
                )

            plt.close(
                "all"
            )

    # =========================================================================
    # LOCAL SHAP
    # =========================================================================

    local_directory = (
        output_directory
        / "Local_Explanations"
    )

    local_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    subset_position = {
        int(original_index):
            position
        for position, original_index
        in enumerate(
            SHAP_INDICES
        )
    }

    local_shap_records = []

    for _, case in local_cases.iterrows():

        original_index = int(
            case[
                "Sample_Index"
            ]
        )

        predicted_class = int(
            case[
                "Predicted_Class_Index"
            ]
        )

        if original_index not in subset_position:
            continue

        position = subset_position[
            original_index
        ]

        case_name = safe_filename(
            (
                f"sample_{original_index}_"
                f"{case['Case_Type']}_"
                f"true_{case['True_Class']}_"
                f"pred_{case['Predicted_Class']}"
            )
        )

        case_directory = (
            local_directory
            / case_name
        )

        case_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        shap_vector = shap_values[
            position,
            :,
            predicted_class,
        ]

        raw_vector = X_test_raw[
            original_index
        ]

        # =====================================================================
        # LOCAL CONTRIBUTION TABLE
        # =====================================================================

        local_table = pd.DataFrame(
            {
                "Feature_Index":
                    np.arange(
                        NUMBER_FEATURES
                    ),

                "Feature":
                    FEATURE_COLUMNS,

                "Raw_Feature_Value":
                    raw_vector,

                "SHAP_Value":
                    shap_vector,

                "Absolute_SHAP":
                    np.abs(
                        shap_vector
                    ),
            }
        )

        local_table = (
            local_table
            .sort_values(
                "Absolute_SHAP",
                ascending=False,
            )
            .reset_index(
                drop=True
            )
        )

        local_table[
            "Rank"
        ] = np.arange(
            1,
            len(
                local_table
            )
            + 1,
        )

        local_table.to_csv(
            case_directory
            / "Local_SHAP_Contributions.csv",
            index=False,
        )

        # =====================================================================
        # SHAP WATERFALL
        # =====================================================================

        try:

            explanation = shap.Explanation(
                values=shap_vector,
                base_values=base_values[
                    predicted_class
                ],
                data=raw_vector,
                feature_names=FEATURE_COLUMNS,
            )

            shap.plots.waterfall(
                explanation,
                max_display=SHAP_LOCAL_MAX_DISPLAY,
                show=False,
            )

            figure = plt.gcf()

            figure.suptitle(
                (
                    f"{model_name} Local SHAP Explanation\n"
                    f"True: {case['True_Class']}    "
                    f"Predicted: {case['Predicted_Class']}    "
                    f"Confidence: "
                    f"{case['Prediction_Confidence']:.3f}"
                ),
                fontsize=13,
                fontweight="bold",
                y=1.02,
            )

            figure.tight_layout()

            save_figure(
                figure,
                case_directory
                / "SHAP_Waterfall",
            )

        except Exception:

            with open(
                case_directory
                / "SHAP_Waterfall_Error.txt",
                "w",
                encoding="utf8",
            ) as file:

                file.write(
                    traceback.format_exc()
                )

            plt.close(
                "all"
            )

        # =====================================================================
        # INTERACTIVE FORCE PLOT
        # =====================================================================

        try:

            force_plot = shap.force_plot(
                base_values[
                    predicted_class
                ],
                shap_vector,
                raw_vector,
                feature_names=FEATURE_COLUMNS,
                matplotlib=False,
            )

            shap.save_html(
                str(
                    case_directory
                    / "SHAP_Force_Plot.html"
                ),
                force_plot,
            )

        except Exception:

            with open(
                case_directory
                / "SHAP_Force_Plot_Error.txt",
                "w",
                encoding="utf8",
            ) as file:

                file.write(
                    traceback.format_exc()
                )

        local_shap_records.append(
            {
                "Sample_Index":
                    original_index,

                "PersonSrial":
                    case[
                        "PersonSrial"
                    ],

                "Case_Type":
                    case[
                        "Case_Type"
                    ],

                "True_Class":
                    case[
                        "True_Class"
                    ],

                "Predicted_Class":
                    case[
                        "Predicted_Class"
                    ],

                "Prediction_Confidence":
                    float(
                        case[
                            "Prediction_Confidence"
                        ]
                    ),

                "Explained_Class":
                    CLASS_NAMES[
                        predicted_class
                    ],

                "Base_Value":
                    float(
                        base_values[
                            predicted_class
                        ]
                    ),
            }
        )

    pd.DataFrame(
        local_shap_records
    ).to_csv(
        output_directory
        / "Local_SHAP_Case_Summary.csv",
        index=False,
    )

    print(
        f"{model_name} SHAP completed."
    )

    print(
        "Top 10 SHAP features:"
    )

    print(
        global_table[
            [
                "Rank",
                "Feature",
                "Mean_Absolute_SHAP",
            ]
        ]
        .head(
            10
        )
        .to_string(
            index=False
        )
    )

    gc.collect()

    return {
        "global":
            global_table,

        "classwise":
            class_table,

        "values":
            shap_values,

        "base_values":
            base_values,
    }


# =============================================================================
# RUN LIGHTGBM SHAP
# =============================================================================

lgb_shap_result = run_shap_analysis(
    model_name="LightGBM",

    model=lgb_model,

    model_path=(
        LGB_SOURCE_DIR
        / "best_lightgbm_model.txt"
    ),

    scaler_path=(
        LGB_SOURCE_DIR
        / "standard_scaler.joblib"
    ),

    X_test_scaled=X_test_lgb,

    prediction=lgb_prediction,

    probability=lgb_probability,

    local_cases=lgb_local_cases,

    output_directory=LGB_SHAP_DIR,
)


# =============================================================================
# RUN XGBOOST SHAP
# =============================================================================

xgb_shap_result = run_shap_analysis(
    model_name="XGBoost",

    model=xgb_model,

    model_path=(
        XGB_SOURCE_DIR
        / "xgboost_model.joblib"
    ),

    scaler_path=(
        XGB_SOURCE_DIR
        / "standard_scaler.joblib"
    ),

    X_test_scaled=X_test_xgb,

    prediction=xgb_prediction,

    probability=xgb_probability,

    local_cases=xgb_local_cases,

    output_directory=XGB_SHAP_DIR,
)


# =============================================================================
# PREPARE LIME BACKGROUND
# =============================================================================

print()

print(
    "=" * 105
)

print(
    "PREPARING LIME BACKGROUND DATA"
)

print(
    "=" * 105
)


lime_background_indices = class_balanced_indices(
    y_train,
    LIME_BACKGROUND_MAX_SAMPLES,
    SEED + 500,
)


X_lime_background = np.ascontiguousarray(
    X_train_raw[
        lime_background_indices
    ],
    dtype=np.float32,
)


y_lime_background = y_train[
    lime_background_indices
]


print(
    "LIME background rows:",
    len(
        X_lime_background
    ),
)


# =============================================================================
# RAW FEATURE PREDICTION WRAPPERS
#
# LIME receives original feature values.
# The saved scaler is applied inside each wrapper.
# =============================================================================

def predict_lgb_raw(
    raw_features,
):

    raw_features = np.asarray(
        raw_features,
        dtype=np.float32,
    )

    scaled = lgb_scaler.transform(
        raw_features
    ).astype(
        np.float32,
        copy=False,
    )

    return np.asarray(
        lgb_model.predict(
            scaled
        ),
        dtype=np.float64,
    )


def predict_xgb_raw(
    raw_features,
):

    raw_features = np.asarray(
        raw_features,
        dtype=np.float32,
    )

    scaled = xgb_scaler.transform(
        raw_features
    ).astype(
        np.float32,
        copy=False,
    )

    return np.asarray(
        xgb_model.predict_proba(
            scaled
        ),
        dtype=np.float64,
    )


# =============================================================================
# SELECT LIME EXPLANATION ROWS
# =============================================================================

def select_lime_indices(
    local_cases,
    seed,
):

    maximum = (
        NUM_CLASSES
        * LIME_EXPLANATIONS_PER_CLASS
    )

    selected = class_balanced_indices(
        y_test,
        maximum,
        seed,
    )

    local_indices = local_cases[
        "Sample_Index"
    ].to_numpy(
        dtype=np.int64
    )

    return np.unique(
        np.concatenate(
            [
                selected,
                local_indices,
            ]
        )
    )


# =============================================================================
# MODERN LIME LOCAL PLOT
# =============================================================================

def save_lime_local_plot(
    explanation,
    label_index,
    model_name,
    case,
    output_directory,
):

    contribution_list = explanation.as_list(
        label=label_index
    )

    rules = [
        item[
            0
        ]
        for item
        in contribution_list
    ]

    weights = np.asarray(
        [
            item[
                1
            ]
            for item
            in contribution_list
        ],
        dtype=np.float64,
    )

    rules = rules[
        ::-1
    ]

    weights = weights[
        ::-1
    ]

    colors = [
        POSITIVE_COLOR
        if value >= 0
        else NEGATIVE_COLOR
        for value
        in weights
    ]

    fig, ax = plt.subplots(
        figsize=(
            11,
            7,
        )
    )

    y_position = np.arange(
        len(
            rules
        )
    )

    ax.barh(
        y_position,
        weights,
        color=colors,
        alpha=0.9,
    )

    ax.set_yticks(
        y_position
    )

    ax.set_yticklabels(
        [
            short_label(
                rule,
                width=58,
            )
            for rule
            in rules
        ]
    )

    ax.axvline(
        0.0,
        color="#94A3B8",
        linewidth=1,
    )

    ax.set_xlabel(
        "LIME contribution weight"
    )

    ax.set_ylabel(
        "Feature condition"
    )

    ax.set_title(
        (
            f"{model_name} Local LIME Explanation\n"
            f"True: {case['True_Class']}    "
            f"Predicted: {case['Predicted_Class']}    "
            f"Confidence: "
            f"{case['Prediction_Confidence']:.3f}"
        )
    )

    ax.grid(
        axis="x",
        color=GRID_COLOR,
        alpha=0.4,
    )

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    fig.tight_layout()

    save_figure(
        fig,
        output_directory
        / "LIME_Local_Explanation",
    )


# =============================================================================
# LIME ANALYSIS ENGINE
# =============================================================================

def run_lime_analysis(
    model_name,
    prediction_function,
    prediction,
    probability,
    local_cases,
    model_path,
    scaler_path,
    output_directory,
    seed,
):

    print()

    print(
        "=" * 105
    )

    print(
        f"LIME ANALYSIS: {model_name}"
    )

    print(
        "=" * 105
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_directory = (
        output_directory
        / "Local_Explanations"
    )

    checkpoint_directory = (
        output_directory
        / "Checkpoints"
    )

    local_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    lime_indices = select_lime_indices(
        local_cases,
        seed,
    )

    print(
        "LIME explanation observations:",
        len(
            lime_indices
        ),
    )

    # =========================================================================
    # LIME CONFIGURATION SIGNATURE
    # =========================================================================

    lime_signature_payload = {
        "Model":
            model_name,

        "Dataset":
            file_fingerprint(
                INPUT_PATH
            ),

        "Model_File":
            file_fingerprint(
                model_path
            ),

        "Scaler_File":
            file_fingerprint(
                scaler_path
            ),

        "Background_Indices_Hash":
            hashlib.sha256(
                lime_background_indices.tobytes()
            ).hexdigest()[
                :16
            ],

        "Explanation_Indices_Hash":
            hashlib.sha256(
                lime_indices.tobytes()
            ).hexdigest()[
                :16
            ],

        "Number_Samples":
            LIME_NUM_SAMPLES,

        "Number_Features":
            LIME_NUM_FEATURES,

        "Classes":
            CLASS_NAMES,
    }

    lime_signature = hashlib.sha256(
        json.dumps(
            lime_signature_payload,
            sort_keys=True,
        ).encode(
            "utf8"
        )
    ).hexdigest()[
        :20
    ]

    save_json(
        {
            "Signature":
                lime_signature,

            **lime_signature_payload,
        },

        output_directory
        / "LIME_Configuration.json",
    )

    # =========================================================================
    # LIME EXPLAINER
    # =========================================================================

    explainer = LimeTabularExplainer(
        training_data=X_lime_background,

        feature_names=FEATURE_COLUMNS,

        class_names=CLASS_NAMES,

        mode="classification",

        discretize_continuous=True,

        discretizer="quartile",

        sample_around_instance=True,

        random_state=seed,
    )

    contribution_path = (
        output_directory
        / "LIME_Feature_Contributions_Long.csv"
    )

    # =========================================================================
    # RESTORE PREVIOUS CONTRIBUTIONS
    # =========================================================================

    current_contributions = pd.DataFrame()

    if contribution_path.exists():

        try:

            all_contributions = pd.read_csv(
                contribution_path,
                low_memory=False,
            )

            if (
                "Signature"
                in all_contributions.columns
            ):

                current_contributions = all_contributions[
                    all_contributions[
                        "Signature"
                    ].astype(
                        str
                    )
                    == lime_signature
                ].copy()

        except Exception:

            current_contributions = pd.DataFrame()

    # =========================================================================
    # LOCAL CASE LOOKUP
    # =========================================================================

    local_case_lookup = {
        int(
            row[
                "Sample_Index"
            ]
        ):
            row

        for _, row
        in local_cases.iterrows()
    }

    # =========================================================================
    # EXPLAIN OBSERVATIONS
    # =========================================================================

    for explanation_number, sample_index in enumerate(
        lime_indices,
        start=1,
    ):

        sample_index = int(
            sample_index
        )

        predicted_class = int(
            prediction[
                sample_index
            ]
        )

        checkpoint_path = (
            checkpoint_directory
            / f"sample_{sample_index}.json"
        )

        already_has_contributions = False

        if len(
            current_contributions
        ) > 0:

            already_has_contributions = bool(
                np.any(
                    current_contributions[
                        "Sample_Index"
                    ].astype(
                        int
                    )
                    == sample_index
                )
            )

        checkpoint_valid = False

        if checkpoint_path.exists():

            try:

                with open(
                    checkpoint_path,
                    "r",
                    encoding="utf8",
                ) as file:

                    checkpoint_data = json.load(
                        file
                    )

                checkpoint_valid = (
                    checkpoint_data.get(
                        "Signature"
                    )
                    == lime_signature
                )

            except Exception:

                checkpoint_valid = False

        if (
            already_has_contributions
            and checkpoint_valid
        ):

            print(
                f"LIME {model_name}: "
                f"sample {sample_index} already completed."
            )

            continue

        print(
            f"LIME {model_name}: "
            f"{explanation_number}/"
            f"{len(lime_indices)} | "
            f"sample {sample_index}"
        )

        explanation_start = time.time()

        explanation = explainer.explain_instance(
            X_test_raw[
                sample_index
            ],

            prediction_function,

            labels=[
                predicted_class
            ],

            num_features=LIME_NUM_FEATURES,

            num_samples=LIME_NUM_SAMPLES,
        )

        feature_map = explanation.as_map()

        if predicted_class not in feature_map:

            raise RuntimeError(
                (
                    f"LIME did not return "
                    f"label {predicted_class}."
                )
            )

        # =====================================================================
        # REMOVE PARTIAL RECORDS FOR THIS SAMPLE
        # =====================================================================

        if len(
            current_contributions
        ) > 0:

            current_contributions = current_contributions[
                current_contributions[
                    "Sample_Index"
                ].astype(
                    int
                )
                != sample_index
            ].copy()

        rows = []

        for feature_index, weight in feature_map[
            predicted_class
        ]:

            feature_index = int(
                feature_index
            )

            rows.append(
                {
                    "Signature":
                        lime_signature,

                    "Model":
                        model_name,

                    "Sample_Index":
                        sample_index,

                    "PersonSrial":
                        str(
                            g_test[
                                sample_index
                            ]
                        ),

                    "True_Class_Index":
                        int(
                            y_test[
                                sample_index
                            ]
                        ),

                    "True_Class":
                        CLASS_NAMES[
                            y_test[
                                sample_index
                            ]
                        ],

                    "Predicted_Class_Index":
                        predicted_class,

                    "Predicted_Class":
                        CLASS_NAMES[
                            predicted_class
                        ],

                    "Prediction_Confidence":
                        float(
                            probability[
                                sample_index,
                                predicted_class,
                            ]
                        ),

                    "Feature_Index":
                        feature_index,

                    "Feature":
                        FEATURE_COLUMNS[
                            feature_index
                        ],

                    "Raw_Feature_Value":
                        float(
                            X_test_raw[
                                sample_index,
                                feature_index,
                            ]
                        ),

                    "LIME_Weight":
                        float(
                            weight
                        ),

                    "Absolute_LIME_Weight":
                        float(
                            abs(
                                weight
                            )
                        ),
                }
            )

        sample_contribution = pd.DataFrame(
            rows
        )

        current_contributions = pd.concat(
            [
                current_contributions,
                sample_contribution,
            ],
            ignore_index=True,
        )

        save_csv(
            current_contributions,
            contribution_path,
        )

        # =====================================================================
        # REPRESENTATIVE LOCAL OUTPUT
        # =====================================================================

        if sample_index in local_case_lookup:

            case = local_case_lookup[
                sample_index
            ]

            case_name = safe_filename(
                (
                    f"sample_{sample_index}_"
                    f"{case['Case_Type']}_"
                    f"true_{case['True_Class']}_"
                    f"pred_{case['Predicted_Class']}"
                )
            )

            case_directory = (
                local_directory
                / case_name
            )

            case_directory.mkdir(
                parents=True,
                exist_ok=True,
            )

            # =================================================================
            # INTERACTIVE HTML
            # =================================================================

            explanation.save_to_file(
                str(
                    case_directory
                    / "LIME_Interactive_Explanation.html"
                )
            )

            # =================================================================
            # PNG AND SVG
            # =================================================================

            save_lime_local_plot(
                explanation,
                predicted_class,
                model_name,
                case,
                case_directory,
            )

            # =================================================================
            # LOCAL RULE TABLE
            # =================================================================

            rule_list = explanation.as_list(
                label=predicted_class
            )

            rule_table = pd.DataFrame(
                {
                    "Feature_Condition":
                        [
                            item[
                                0
                            ]
                            for item
                            in rule_list
                        ],

                    "LIME_Weight":
                        [
                            item[
                                1
                            ]
                            for item
                            in rule_list
                        ],
                }
            )

            rule_table[
                "Absolute_LIME_Weight"
            ] = np.abs(
                rule_table[
                    "LIME_Weight"
                ]
            )

            rule_table.to_csv(
                case_directory
                / "LIME_Local_Rules.csv",
                index=False,
            )

            save_json(
                {
                    "Model":
                        model_name,

                    "Sample_Index":
                        sample_index,

                    "PersonSrial":
                        str(
                            case[
                                "PersonSrial"
                            ]
                        ),

                    "Case_Type":
                        str(
                            case[
                                "Case_Type"
                            ]
                        ),

                    "True_Class":
                        str(
                            case[
                                "True_Class"
                            ]
                        ),

                    "Predicted_Class":
                        str(
                            case[
                                "Predicted_Class"
                            ]
                        ),

                    "Prediction_Confidence":
                        float(
                            case[
                                "Prediction_Confidence"
                            ]
                        ),
                },

                case_directory
                / "Case_Metadata.json",
            )

        # =====================================================================
        # SAVE CHECKPOINT AFTER SUCCESS
        # =====================================================================

        save_json(
            {
                "Signature":
                    lime_signature,

                "Sample_Index":
                    sample_index,

                "Completed":
                    True,

                "Execution_Seconds":
                    float(
                        time.time()
                        - explanation_start
                    ),
            },

            checkpoint_path,
        )

        gc.collect()

    # =========================================================================
    # LOAD FINAL CONTRIBUTIONS
    # =========================================================================

    if not contribution_path.exists():

        raise RuntimeError(
            "LIME contribution file was not created."
        )

    current_contributions = pd.read_csv(
        contribution_path,
        low_memory=False,
    )

    current_contributions = current_contributions[
        current_contributions[
            "Signature"
        ].astype(
            str
        )
        == lime_signature
    ].copy()

    total_explanations = int(
        current_contributions[
            "Sample_Index"
        ].nunique()
    )

    if total_explanations == 0:

        raise RuntimeError(
            "No LIME explanations were generated."
        )

    # =========================================================================
    # GLOBAL LIME IMPORTANCE
    # =========================================================================

    global_aggregate = (
        current_contributions

        .groupby(
            [
                "Feature_Index",
                "Feature",
            ],
            as_index=False,
        )

        .agg(
            Sum_Absolute_LIME=(
                "Absolute_LIME_Weight",
                "sum",
            ),

            Sum_Signed_LIME=(
                "LIME_Weight",
                "sum",
            ),

            Number_of_Nonzero_Explanations=(
                "Sample_Index",
                "nunique",
            ),
        )
    )

    global_aggregate[
        "Mean_Absolute_LIME"
    ] = (
        global_aggregate[
            "Sum_Absolute_LIME"
        ]
        / total_explanations
    )

    global_aggregate[
        "Mean_Signed_LIME"
    ] = (
        global_aggregate[
            "Sum_Signed_LIME"
        ]
        / total_explanations
    )

    global_aggregate = (
        global_aggregate
        .sort_values(
            "Mean_Absolute_LIME",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    global_aggregate[
        "Rank"
    ] = np.arange(
        1,
        len(
            global_aggregate
        )
        + 1,
    )

    global_aggregate.to_csv(
        output_directory
        / "Global_LIME_Feature_Importance.csv",
        index=False,
    )

    # =========================================================================
    # GLOBAL MODERN LIME BAR
    # =========================================================================

    top = (
        global_aggregate
        .head(
            SHAP_TOP_FEATURES
        )
        .sort_values(
            "Mean_Absolute_LIME",
            ascending=True,
        )
    )

    model_color = (
        LIGHTGBM_COLOR
        if model_name == "LightGBM"
        else XGBOOST_COLOR
    )

    fig, ax = plt.subplots(
        figsize=(
            10,
            9,
        )
    )

    y_position = np.arange(
        len(
            top
        )
    )

    ax.barh(
        y_position,
        top[
            "Mean_Absolute_LIME"
        ],
        color=model_color,
        alpha=0.88,
    )

    ax.set_yticks(
        y_position
    )

    ax.set_yticklabels(
        [
            short_label(
                feature
            )
            for feature
            in top[
                "Feature"
            ]
        ]
    )

    ax.set_xlabel(
        "Mean absolute LIME weight"
    )

    ax.set_ylabel(
        "EEG feature"
    )

    ax.set_title(
        f"{model_name} Aggregated LIME Feature Importance"
    )

    ax.grid(
        axis="x",
        color=GRID_COLOR,
        alpha=0.45,
    )

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    fig.tight_layout()

    save_figure(
        fig,
        output_directory
        / "01_Global_LIME_Feature_Importance",
    )

    # =========================================================================
    # CLASSWISE LIME
    # =========================================================================

    class_counts = (
        current_contributions[
            [
                "Sample_Index",
                "Predicted_Class_Index",
            ]
        ]
        .drop_duplicates()
        .groupby(
            "Predicted_Class_Index"
        )[
            "Sample_Index"
        ]
        .nunique()
        .to_dict()
    )

    class_aggregate = (
        current_contributions

        .groupby(
            [
                "Predicted_Class_Index",
                "Predicted_Class",
                "Feature_Index",
                "Feature",
            ],
            as_index=False,
        )

        .agg(
            Sum_Absolute_LIME=(
                "Absolute_LIME_Weight",
                "sum",
            ),

            Sum_Signed_LIME=(
                "LIME_Weight",
                "sum",
            ),
        )
    )

    class_aggregate[
        "Number_of_Class_Explanations"
    ] = class_aggregate[
        "Predicted_Class_Index"
    ].map(
        class_counts
    )

    class_aggregate[
        "Mean_Absolute_LIME"
    ] = (
        class_aggregate[
            "Sum_Absolute_LIME"
        ]
        / class_aggregate[
            "Number_of_Class_Explanations"
        ]
    )

    class_aggregate[
        "Mean_Signed_LIME"
    ] = (
        class_aggregate[
            "Sum_Signed_LIME"
        ]
        / class_aggregate[
            "Number_of_Class_Explanations"
        ]
    )

    class_aggregate.to_csv(
        output_directory
        / "Classwise_LIME_Feature_Importance.csv",
        index=False,
    )

    # =========================================================================
    # CLASSWISE LIME HEATMAP
    # =========================================================================

    top_feature_indices = (
        global_aggregate
        .head(
            SHAP_CLASS_TOP_FEATURES
        )[
            "Feature_Index"
        ]
        .astype(
            int
        )
        .to_numpy()
    )

    matrix = np.zeros(
        (
            NUM_CLASSES,
            len(
                top_feature_indices
            ),
        ),
        dtype=np.float64,
    )

    for class_index in range(
        NUM_CLASSES
    ):

        class_rows = class_aggregate[
            class_aggregate[
                "Predicted_Class_Index"
            ]
            == class_index
        ]

        class_map = dict(
            zip(
                class_rows[
                    "Feature_Index"
                ].astype(
                    int
                ),
                class_rows[
                    "Mean_Absolute_LIME"
                ],
            )
        )

        for column_index, feature_index in enumerate(
            top_feature_indices
        ):

            matrix[
                class_index,
                column_index,
            ] = class_map.get(
                int(
                    feature_index
                ),
                0.0,
            )

    fig, ax = plt.subplots(
        figsize=(
            14,
            5.5,
        )
    )

    image = ax.imshow(
        matrix,
        aspect="auto",
        cmap="magma",
    )

    ax.set_xticks(
        np.arange(
            len(
                top_feature_indices
            )
        )
    )

    ax.set_xticklabels(
        [
            short_label(
                FEATURE_COLUMNS[
                    index
                ],
                width=36,
            )
            for index
            in top_feature_indices
        ],
        rotation=55,
        ha="right",
    )

    ax.set_yticks(
        np.arange(
            NUM_CLASSES
        )
    )

    ax.set_yticklabels(
        CLASS_NAMES
    )

    ax.set_xlabel(
        "EEG feature"
    )

    ax.set_ylabel(
        "Predicted class"
    )

    ax.set_title(
        f"{model_name} Classwise Aggregated LIME Importance"
    )

    color_bar = fig.colorbar(
        image,
        ax=ax,
        pad=0.02,
    )

    color_bar.set_label(
        "Mean absolute LIME weight"
    )

    fig.tight_layout()

    save_figure(
        fig,
        output_directory
        / "02_Classwise_LIME_Heatmap",
    )

    print(
        f"{model_name} LIME completed."
    )

    print(
        "Top 10 LIME features:"
    )

    print(
        global_aggregate[
            [
                "Rank",
                "Feature",
                "Mean_Absolute_LIME",
            ]
        ]
        .head(
            10
        )
        .to_string(
            index=False
        )
    )

    gc.collect()

    return {
        "global":
            global_aggregate,

        "classwise":
            class_aggregate,

        "contributions":
            current_contributions,
    }


# =============================================================================
# RUN LIGHTGBM LIME
# =============================================================================

lgb_lime_result = run_lime_analysis(
    model_name="LightGBM",

    prediction_function=predict_lgb_raw,

    prediction=lgb_prediction,

    probability=lgb_probability,

    local_cases=lgb_local_cases,

    model_path=(
        LGB_SOURCE_DIR
        / "best_lightgbm_model.txt"
    ),

    scaler_path=(
        LGB_SOURCE_DIR
        / "standard_scaler.joblib"
    ),

    output_directory=LGB_LIME_DIR,

    seed=SEED + 100,
)


# =============================================================================
# RUN XGBOOST LIME
# =============================================================================

xgb_lime_result = run_lime_analysis(
    model_name="XGBoost",

    prediction_function=predict_xgb_raw,

    prediction=xgb_prediction,

    probability=xgb_probability,

    local_cases=xgb_local_cases,

    model_path=(
        XGB_SOURCE_DIR
        / "xgboost_model.joblib"
    ),

    scaler_path=(
        XGB_SOURCE_DIR
        / "standard_scaler.joblib"
    ),

    output_directory=XGB_LIME_DIR,

    seed=SEED + 200,
)


# =============================================================================
# XAI COMPARISON
# =============================================================================

print()

print(
    "=" * 105
)

print(
    "CREATING LIGHTGBM AND XGBOOST XAI COMPARISON"
)

print(
    "=" * 105
)


# =============================================================================
# NORMALIZE IMPORTANCE
# =============================================================================

def normalize_importance(
    values,
):

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    maximum = np.max(
        values
    )

    if (
        not np.isfinite(
            maximum
        )
        or maximum <= 0
    ):

        return np.zeros_like(
            values
        )

    return (
        values
        / maximum
    )


# =============================================================================
# SHAP MODEL COMPARISON
# =============================================================================

lgb_shap_table = lgb_shap_result[
    "global"
][
    [
        "Feature",
        "Mean_Absolute_SHAP",
    ]
].copy()


xgb_shap_table = xgb_shap_result[
    "global"
][
    [
        "Feature",
        "Mean_Absolute_SHAP",
    ]
].copy()


lgb_shap_table = lgb_shap_table.rename(
    columns={
        "Mean_Absolute_SHAP":
            "LightGBM_SHAP"
    }
)

xgb_shap_table = xgb_shap_table.rename(
    columns={
        "Mean_Absolute_SHAP":
            "XGBoost_SHAP"
    }
)


shap_comparison = pd.merge(
    lgb_shap_table,
    xgb_shap_table,
    on="Feature",
    how="outer",
).fillna(
    0.0
)


shap_comparison[
    "LightGBM_Normalized"
] = normalize_importance(
    shap_comparison[
        "LightGBM_SHAP"
    ]
)


shap_comparison[
    "XGBoost_Normalized"
] = normalize_importance(
    shap_comparison[
        "XGBoost_SHAP"
    ]
)


shap_comparison[
    "Combined_SHAP_Score"
] = (
    shap_comparison[
        "LightGBM_Normalized"
    ]
    +
    shap_comparison[
        "XGBoost_Normalized"
    ]
) / 2.0


shap_comparison = (
    shap_comparison
    .sort_values(
        "Combined_SHAP_Score",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


shap_comparison[
    "Combined_Rank"
] = np.arange(
    1,
    len(
        shap_comparison
    )
    + 1,
)


shap_comparison.to_csv(
    COMPARISON_ROOT
    / "SHAP_LightGBM_vs_XGBoost.csv",
    index=False,
)


# =============================================================================
# SHAP COMPARISON PLOT
# =============================================================================

top_shap_comparison = (
    shap_comparison
    .head(
        20
    )
    .sort_values(
        "Combined_SHAP_Score",
        ascending=True,
    )
)


y_position = np.arange(
    len(
        top_shap_comparison
    )
)


bar_height = 0.38


fig, ax = plt.subplots(
    figsize=(
        12,
        9,
    )
)


ax.barh(
    y_position
    - bar_height / 2,
    top_shap_comparison[
        "LightGBM_Normalized"
    ],
    height=bar_height,
    color=LIGHTGBM_COLOR,
    label="LightGBM",
)


ax.barh(
    y_position
    + bar_height / 2,
    top_shap_comparison[
        "XGBoost_Normalized"
    ],
    height=bar_height,
    color=XGBOOST_COLOR,
    label="XGBoost",
)


ax.set_yticks(
    y_position
)


ax.set_yticklabels(
    [
        short_label(
            feature
        )
        for feature
        in top_shap_comparison[
            "Feature"
        ]
    ]
)


ax.set_xlabel(
    "Normalized mean absolute SHAP importance"
)


ax.set_ylabel(
    "EEG feature"
)


ax.set_title(
    "Global SHAP Importance Comparison"
)


ax.legend()


ax.grid(
    axis="x",
    alpha=0.3,
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


fig.tight_layout()


save_figure(
    fig,
    COMPARISON_ROOT
    / "01_SHAP_LightGBM_vs_XGBoost",
)


# =============================================================================
# LIME MODEL COMPARISON
# =============================================================================

lgb_lime_table = lgb_lime_result[
    "global"
][
    [
        "Feature",
        "Mean_Absolute_LIME",
    ]
].copy()


xgb_lime_table = xgb_lime_result[
    "global"
][
    [
        "Feature",
        "Mean_Absolute_LIME",
    ]
].copy()


lgb_lime_table = lgb_lime_table.rename(
    columns={
        "Mean_Absolute_LIME":
            "LightGBM_LIME"
    }
)


xgb_lime_table = xgb_lime_table.rename(
    columns={
        "Mean_Absolute_LIME":
            "XGBoost_LIME"
    }
)


lime_comparison = pd.merge(
    lgb_lime_table,
    xgb_lime_table,
    on="Feature",
    how="outer",
).fillna(
    0.0
)


lime_comparison[
    "LightGBM_Normalized"
] = normalize_importance(
    lime_comparison[
        "LightGBM_LIME"
    ]
)


lime_comparison[
    "XGBoost_Normalized"
] = normalize_importance(
    lime_comparison[
        "XGBoost_LIME"
    ]
)


lime_comparison[
    "Combined_LIME_Score"
] = (
    lime_comparison[
        "LightGBM_Normalized"
    ]
    +
    lime_comparison[
        "XGBoost_Normalized"
    ]
) / 2.0


lime_comparison = (
    lime_comparison
    .sort_values(
        "Combined_LIME_Score",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


lime_comparison[
    "Combined_Rank"
] = np.arange(
    1,
    len(
        lime_comparison
    )
    + 1,
)


lime_comparison.to_csv(
    COMPARISON_ROOT
    / "LIME_LightGBM_vs_XGBoost.csv",
    index=False,
)


# =============================================================================
# LIME COMPARISON PLOT
# =============================================================================

top_lime_comparison = (
    lime_comparison
    .head(
        20
    )
    .sort_values(
        "Combined_LIME_Score",
        ascending=True,
    )
)


y_position = np.arange(
    len(
        top_lime_comparison
    )
)


fig, ax = plt.subplots(
    figsize=(
        12,
        9,
    )
)


ax.barh(
    y_position
    - bar_height / 2,
    top_lime_comparison[
        "LightGBM_Normalized"
    ],
    height=bar_height,
    color=LIGHTGBM_COLOR,
    label="LightGBM",
)


ax.barh(
    y_position
    + bar_height / 2,
    top_lime_comparison[
        "XGBoost_Normalized"
    ],
    height=bar_height,
    color=XGBOOST_COLOR,
    label="XGBoost",
)


ax.set_yticks(
    y_position
)


ax.set_yticklabels(
    [
        short_label(
            feature
        )
        for feature
        in top_lime_comparison[
            "Feature"
        ]
    ]
)


ax.set_xlabel(
    "Normalized aggregated LIME importance"
)


ax.set_ylabel(
    "EEG feature"
)


ax.set_title(
    "Aggregated LIME Importance Comparison"
)


ax.legend()


ax.grid(
    axis="x",
    alpha=0.3,
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


fig.tight_layout()


save_figure(
    fig,
    COMPARISON_ROOT
    / "02_LIME_LightGBM_vs_XGBoost",
)


# =============================================================================
# SHAP AND LIME CONSENSUS
#
# Descriptive only.
# Not a statistical significance test.
# =============================================================================

def create_xai_consensus(
    model_name,
    shap_table,
    lime_table,
):

    shap_part = shap_table[
        [
            "Feature",
            "Mean_Absolute_SHAP",
        ]
    ].copy()

    lime_part = lime_table[
        [
            "Feature",
            "Mean_Absolute_LIME",
        ]
    ].copy()

    merged = pd.merge(
        shap_part,
        lime_part,
        on="Feature",
        how="outer",
    ).fillna(
        0.0
    )

    merged[
        "Normalized_SHAP"
    ] = normalize_importance(
        merged[
            "Mean_Absolute_SHAP"
        ]
    )

    merged[
        "Normalized_LIME"
    ] = normalize_importance(
        merged[
            "Mean_Absolute_LIME"
        ]
    )

    merged[
        "Descriptive_Consensus_Score"
    ] = (
        merged[
            "Normalized_SHAP"
        ]
        +
        merged[
            "Normalized_LIME"
        ]
    ) / 2.0

    merged = (
        merged
        .sort_values(
            "Descriptive_Consensus_Score",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    merged[
        "Consensus_Rank"
    ] = np.arange(
        1,
        len(
            merged
        )
        + 1,
    )

    merged.to_csv(
        COMPARISON_ROOT
        / (
            f"{model_name}_"
            f"SHAP_LIME_Consensus.csv"
        ),
        index=False,
    )

    top = (
        merged
        .head(
            20
        )
        .sort_values(
            "Descriptive_Consensus_Score",
            ascending=True,
        )
    )

    fig, ax = plt.subplots(
        figsize=(
            11,
            8.5,
        )
    )

    y_position = np.arange(
        len(
            top
        )
    )

    ax.barh(
        y_position,
        top[
            "Descriptive_Consensus_Score"
        ],
        color=ACCENT_COLOR,
        alpha=0.88,
    )

    ax.set_yticks(
        y_position
    )

    ax.set_yticklabels(
        [
            short_label(
                feature
            )
            for feature
            in top[
                "Feature"
            ]
        ]
    )

    ax.set_xlabel(
        "Normalized SHAP and LIME consensus score"
    )

    ax.set_ylabel(
        "EEG feature"
    )

    ax.set_title(
        (
            f"{model_name} Descriptive "
            f"SHAP and LIME Consensus"
        )
    )

    ax.grid(
        axis="x",
        alpha=0.3,
    )

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    fig.tight_layout()

    save_figure(
        fig,
        COMPARISON_ROOT
        / (
            f"03_{model_name}_"
            f"SHAP_LIME_Consensus"
        ),
    )

    return merged


# =============================================================================
# CONSENSUS
# =============================================================================

lgb_consensus = create_xai_consensus(
    "LightGBM",
    lgb_shap_result[
        "global"
    ],
    lgb_lime_result[
        "global"
    ],
)


xgb_consensus = create_xai_consensus(
    "XGBoost",
    xgb_shap_result[
        "global"
    ],
    xgb_lime_result[
        "global"
    ],
)


# =============================================================================
# TOP FEATURE AGREEMENT
# =============================================================================

TOP_OVERLAP_COUNT = 20


lgb_shap_top = set(
    lgb_shap_result[
        "global"
    ]
    .head(
        TOP_OVERLAP_COUNT
    )[
        "Feature"
    ]
)


xgb_shap_top = set(
    xgb_shap_result[
        "global"
    ]
    .head(
        TOP_OVERLAP_COUNT
    )[
        "Feature"
    ]
)


lgb_lime_top = set(
    lgb_lime_result[
        "global"
    ]
    .head(
        TOP_OVERLAP_COUNT
    )[
        "Feature"
    ]
)


xgb_lime_top = set(
    xgb_lime_result[
        "global"
    ]
    .head(
        TOP_OVERLAP_COUNT
    )[
        "Feature"
    ]
)


shap_overlap = (
    lgb_shap_top
    & xgb_shap_top
)


lime_overlap = (
    lgb_lime_top
    & xgb_lime_top
)


shap_union = (
    lgb_shap_top
    | xgb_shap_top
)


lime_union = (
    lgb_lime_top
    | xgb_lime_top
)


SHAP_JACCARD = (
    len(
        shap_overlap
    )
    / len(
        shap_union
    )
    if len(
        shap_union
    ) > 0
    else 0.0
)


LIME_JACCARD = (
    len(
        lime_overlap
    )
    / len(
        lime_union
    )
    if len(
        lime_union
    ) > 0
    else 0.0
)


overlap_summary = {
    "Top_Features_Compared":
        TOP_OVERLAP_COUNT,

    "SHAP_Common_Features_Count":
        len(
            shap_overlap
        ),

    "SHAP_Jaccard_Similarity":
        SHAP_JACCARD,

    "SHAP_Common_Features":
        sorted(
            shap_overlap
        ),

    "LIME_Common_Features_Count":
        len(
            lime_overlap
        ),

    "LIME_Jaccard_Similarity":
        LIME_JACCARD,

    "LIME_Common_Features":
        sorted(
            lime_overlap
        ),
}


save_json(
    overlap_summary,
    COMPARISON_ROOT
    / "Top_Feature_Overlap.json",
)


# =============================================================================
# OVERVIEW DASHBOARD
# =============================================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        16,
        11,
    ),
)


# =============================================================================
# PANEL 1
# MODEL PERFORMANCE
# =============================================================================

ax = axes[
    0,
    0
]


ax.bar(
    [
        "LightGBM",
        "XGBoost",
    ],
    [
        lgb_accuracy,
        xgb_accuracy,
    ],
    color=[
        LIGHTGBM_COLOR,
        XGBOOST_COLOR,
    ],
)


ax.set_ylim(
    max(
        0.0,
        min(
            lgb_accuracy,
            xgb_accuracy,
        )
        - 0.08,
    ),
    1.0,
)


ax.set_ylabel(
    "Final test accuracy"
)


ax.set_title(
    "Final Model Performance"
)


ax.grid(
    axis="y",
    alpha=0.25,
)


for index, value in enumerate(
    [
        lgb_accuracy,
        xgb_accuracy,
    ]
):

    ax.text(
        index,
        value + 0.003,
        f"{value:.4f}",
        ha="center",
        fontweight="bold",
    )


# =============================================================================
# PANEL 2
# TOP SHAP FEATURES
# =============================================================================

ax = axes[
    0,
    1
]


dashboard_shap = (
    shap_comparison
    .head(
        8
    )
    .sort_values(
        "Combined_SHAP_Score",
        ascending=True,
    )
)


y_position = np.arange(
    len(
        dashboard_shap
    )
)


ax.barh(
    y_position - 0.18,
    dashboard_shap[
        "LightGBM_Normalized"
    ],
    height=0.35,
    color=LIGHTGBM_COLOR,
    label="LightGBM",
)


ax.barh(
    y_position + 0.18,
    dashboard_shap[
        "XGBoost_Normalized"
    ],
    height=0.35,
    color=XGBOOST_COLOR,
    label="XGBoost",
)


ax.set_yticks(
    y_position
)


ax.set_yticklabels(
    [
        short_label(
            feature,
            width=30,
        )
        for feature
        in dashboard_shap[
            "Feature"
        ]
    ]
)


ax.set_title(
    "Top SHAP Features"
)


ax.set_xlabel(
    "Normalized importance"
)


ax.legend()


ax.grid(
    axis="x",
    alpha=0.25,
)


# =============================================================================
# PANEL 3
# TOP LIME FEATURES
# =============================================================================

ax = axes[
    1,
    0
]


dashboard_lime = (
    lime_comparison
    .head(
        8
    )
    .sort_values(
        "Combined_LIME_Score",
        ascending=True,
    )
)


y_position = np.arange(
    len(
        dashboard_lime
    )
)


ax.barh(
    y_position - 0.18,
    dashboard_lime[
        "LightGBM_Normalized"
    ],
    height=0.35,
    color=LIGHTGBM_COLOR,
    label="LightGBM",
)


ax.barh(
    y_position + 0.18,
    dashboard_lime[
        "XGBoost_Normalized"
    ],
    height=0.35,
    color=XGBOOST_COLOR,
    label="XGBoost",
)


ax.set_yticks(
    y_position
)


ax.set_yticklabels(
    [
        short_label(
            feature,
            width=30,
        )
        for feature
        in dashboard_lime[
            "Feature"
        ]
    ]
)


ax.set_title(
    "Top Aggregated LIME Features"
)


ax.set_xlabel(
    "Normalized importance"
)


ax.legend()


ax.grid(
    axis="x",
    alpha=0.25,
)


# =============================================================================
# PANEL 4
# FEATURE AGREEMENT
# =============================================================================

ax = axes[
    1,
    1
]


ax.bar(
    [
        "SHAP",
        "LIME",
    ],
    [
        SHAP_JACCARD,
        LIME_JACCARD,
    ],
    color=[
        ACCENT_COLOR,
        POSITIVE_COLOR,
    ],
)


ax.set_ylim(
    0,
    1,
)


ax.set_ylabel(
    "Jaccard similarity"
)


ax.set_title(
    "LightGBM and XGBoost Top Feature Agreement"
)


ax.grid(
    axis="y",
    alpha=0.25,
)


for index, value in enumerate(
    [
        SHAP_JACCARD,
        LIME_JACCARD,
    ]
):

    ax.text(
        index,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        fontweight="bold",
    )


fig.suptitle(
    (
        "Explainable AI Overview: "
        "Final LightGBM and XGBoost Models"
    ),
    fontsize=18,
    fontweight="bold",
    y=0.99,
)


fig.tight_layout(
    rect=[
        0,
        0,
        1,
        0.97,
    ]
)


save_figure(
    fig,
    COMPARISON_ROOT
    / "04_XAI_Overview_Dashboard",
)


# =============================================================================
# SAFE TOP FEATURE EXTRACTION
# =============================================================================

def top_feature_list(
    dataframe,
    column_name,
    count=10,
):

    values = (
        dataframe[
            column_name
        ]
        .head(
            count
        )
        .astype(
            str
        )
        .tolist()
    )

    while len(
        values
    ) < count:

        values.append(
            ""
        )

    return values


# =============================================================================
# FINAL TOP FEATURE TABLE
# =============================================================================

top_feature_summary = pd.DataFrame(
    {
        "Rank":
            np.arange(
                1,
                11,
            ),

        "LightGBM_SHAP":
            top_feature_list(
                lgb_shap_result[
                    "global"
                ],
                "Feature",
                10,
            ),

        "XGBoost_SHAP":
            top_feature_list(
                xgb_shap_result[
                    "global"
                ],
                "Feature",
                10,
            ),

        "LightGBM_LIME":
            top_feature_list(
                lgb_lime_result[
                    "global"
                ],
                "Feature",
                10,
            ),

        "XGBoost_LIME":
            top_feature_list(
                xgb_lime_result[
                    "global"
                ],
                "Feature",
                10,
            ),

        "LightGBM_Consensus":
            top_feature_list(
                lgb_consensus,
                "Feature",
                10,
            ),

        "XGBoost_Consensus":
            top_feature_list(
                xgb_consensus,
                "Feature",
                10,
            ),
    }
)


top_feature_summary.to_csv(
    COMPARISON_ROOT
    / "Top_10_XAI_Features.csv",
    index=False,
)


# =============================================================================
# FINAL SUMMARY
# =============================================================================

TOTAL_TIME = (
    time.time()
    - TOTAL_START
)


xai_summary = {
    "Models":
        [
            "LightGBM",
            "XGBoost",
        ],

    "Selected_Data_Split":
        (
            "Training 60 percent, "
            "Testing 30 percent, "
            "Validation 10 percent"
        ),

    "LightGBM_Test_Accuracy":
        lgb_accuracy,

    "XGBoost_Test_Accuracy":
        xgb_accuracy,

    "SHAP_Explained_Test_Rows":
        int(
            len(
                SHAP_INDICES
            )
        ),

    "LIME_Background_Training_Rows":
        int(
            len(
                X_lime_background
            )
        ),

    "LightGBM_LIME_Explained_Test_Rows":
        int(
            lgb_lime_result[
                "contributions"
            ][
                "Sample_Index"
            ].nunique()
        ),

    "XGBoost_LIME_Explained_Test_Rows":
        int(
            xgb_lime_result[
                "contributions"
            ][
                "Sample_Index"
            ].nunique()
        ),

    "SHAP_Top_Feature_LightGBM":
        str(
            lgb_shap_result[
                "global"
            ].iloc[
                0
            ][
                "Feature"
            ]
        ),

    "SHAP_Top_Feature_XGBoost":
        str(
            xgb_shap_result[
                "global"
            ].iloc[
                0
            ][
                "Feature"
            ]
        ),

    "LIME_Top_Feature_LightGBM":
        str(
            lgb_lime_result[
                "global"
            ].iloc[
                0
            ][
                "Feature"
            ]
        ),

    "LIME_Top_Feature_XGBoost":
        str(
            xgb_lime_result[
                "global"
            ].iloc[
                0
            ][
                "Feature"
            ]
        ),

    "SHAP_Top20_Model_Jaccard":
        float(
            SHAP_JACCARD
        ),

    "LIME_Top20_Model_Jaccard":
        float(
            LIME_JACCARD
        ),

    "SHAP_Output_Interpretation":
        (
            "Tree SHAP values are calculated on the "
            "tree model output scale. Original feature "
            "values are used for visual display."
        ),

    "LIME_Output_Interpretation":
        (
            "LIME operates on original feature values "
            "while the prediction wrapper applies the "
            "saved model scaler before prediction."
        ),

    "Descriptive_Consensus_Note":
        (
            "The SHAP and LIME consensus score is a "
            "descriptive normalized ranking and is not "
            "a statistical significance test."
        ),

    "Execution_Time_Minutes":
        float(
            TOTAL_TIME
            / 60.0
        ),
}


save_json(
    xai_summary,
    OUTPUT_ROOT
    / "Final_XAI_Summary.json",
)


# =============================================================================
# TEXT SUMMARY
# =============================================================================

with open(
    OUTPUT_ROOT
    / "Final_XAI_Summary.txt",
    "w",
    encoding="utf8",
) as file:

    file.write(
        "FINAL TWO MODEL XAI SUMMARY\n"
    )

    file.write(
        "========================================\n\n"
    )

    file.write(
        "Models: LightGBM and XGBoost\n"
    )

    file.write(
        "Split: Train 60%, Test 30%, Validation 10%\n\n"
    )

    file.write(
        f"LightGBM Test Accuracy: "
        f"{lgb_accuracy:.6f}\n"
    )

    file.write(
        f"XGBoost Test Accuracy: "
        f"{xgb_accuracy:.6f}\n\n"
    )

    file.write(
        f"SHAP Explained Test Rows: "
        f"{len(SHAP_INDICES)}\n"
    )

    file.write(
        f"LIME Background Training Rows: "
        f"{len(X_lime_background)}\n\n"
    )

    file.write(
        "Top LightGBM SHAP Feature:\n"
    )

    file.write(
        f"{lgb_shap_result['global'].iloc[0]['Feature']}\n\n"
    )

    file.write(
        "Top XGBoost SHAP Feature:\n"
    )

    file.write(
        f"{xgb_shap_result['global'].iloc[0]['Feature']}\n\n"
    )

    file.write(
        "Top LightGBM LIME Feature:\n"
    )

    file.write(
        f"{lgb_lime_result['global'].iloc[0]['Feature']}\n\n"
    )

    file.write(
        "Top XGBoost LIME Feature:\n"
    )

    file.write(
        f"{xgb_lime_result['global'].iloc[0]['Feature']}\n\n"
    )

    file.write(
        f"SHAP Top 20 Model Similarity: "
        f"{SHAP_JACCARD:.6f}\n"
    )

    file.write(
        f"LIME Top 20 Model Similarity: "
        f"{LIME_JACCARD:.6f}\n\n"
    )

    file.write(
        f"Execution Time: "
        f"{TOTAL_TIME / 60:.2f} minutes\n"
    )


# =============================================================================
# FINAL SCREEN OUTPUT
# =============================================================================

print()

print(
    "#" * 105
)

print(
    "FINAL SHAP AND LIME ANALYSIS COMPLETED"
)

print(
    "#" * 105
)

print()

print(
    "LightGBM Test Accuracy:",
    f"{lgb_accuracy:.6f}",
)

print(
    "XGBoost Test Accuracy:",
    f"{xgb_accuracy:.6f}",
)

print()

print(
    "SHAP explained rows:",
    len(
        SHAP_INDICES
    ),
)

print(
    "LIME background training rows:",
    len(
        X_lime_background
    ),
)

print()

print(
    "TOP 10 XAI FEATURES"
)

print()

print(
    top_feature_summary.to_string(
        index=False
    )
)

print()

print(
    "SHAP Top 20 LightGBM vs XGBoost similarity:",
    f"{SHAP_JACCARD:.4f}",
)

print(
    "LIME Top 20 LightGBM vs XGBoost similarity:",
    f"{LIME_JACCARD:.4f}",
)

print()

print(
    "Execution time:",
    f"{TOTAL_TIME / 60:.2f}",
    "minutes",
)

print(
    "RAM:",
    f"{memory_mb():.2f}",
    "MB",
)

print()

print(
    "SHAP results:"
)

print(
    SHAP_ROOT
)

print()

print(
    "LIME results:"
)

print(
    LIME_ROOT
)

print()

print(
    "Comparison results:"
)

print(
    COMPARISON_ROOT
)

print()

print(
    "ALL XAI RESULTS SAVED IN:"
)

print(
    OUTPUT_ROOT
)

print(
    "#" * 105
)

FINAL TWO MODEL EXPLAINABLE AI
Models:
LightGBM
XGBoost

XAI:
SHAP
LIME

Input:
/home/rakib/Desktop/ml_apply/Final_Model_Test/Cleaned_oN_Selected_Max.csv

Output:
/home/rakib/Desktop/ml_apply/Final_Two_Model_XAI

SHAP maximum global explanation samples: 6000
LIME background samples: 6000
LIME explanations per class: 5

LOADING FINAL MODEL ARTIFACTS
Classes: ['class_lecture', 'music', 'normal', 'reading']

LOADING MAX DATASET
Dataset shape: (149350, 232)
Feature count: 232
Classes: ['class_lecture', 'music', 'normal', 'reading']
Participants: 60
Loading time: 8.39 seconds
RAM: 1291.63 MB

Training participants: 36
Validation participants: 6
Testing participants: 18
Training rows: 90280
Final testing rows: 45678

LOADING FINAL LIGHTGBM AND XGBOOST MODELS
LightGBM final test accuracy: 0.941482
XGBoost final test accuracy: 0.933819
RAM: 1159.45 MB

SHAP explanation rows: 6013

SHAP ANALYSIS: LightGBM
Building TreeExplainer...
Computing SHAP values for 6013 test observations...
SHAP computa

**XAI Apply END**

**SHAP and LIME Waterfall Diagram Generate**

In [1]:
# =============================================================================
# SHAP AND LIME WATERFALL DIAGRAMS
#
# Models
# LightGBM
# XGBoost
#
# XAI
# SHAP
# LIME
#
# Uses existing completed XAI results
#
# Input
# /home/rakib/Desktop/ml_apply/Final_Model_Test/
# Cleaned_oN_Selected_Max.csv
#
# Existing XAI root
# /home/rakib/Desktop/ml_apply/Final_Two_Model_XAI/
#
# Output
# /home/rakib/Desktop/ml_apply/Final_Two_Model_XAI/
# Waterfall_Diagrams/
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import gc
import json
import random
import warnings
import re

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import matplotlib

matplotlib.use(
    "Agg"
)

import matplotlib.pyplot as plt

import lightgbm as lgb

import shap

from lime.lime_tabular import (
    LimeTabularExplainer
)


# =============================================================================
# CONFIGURATION
# =============================================================================

warnings.filterwarnings(
    "ignore"
)


SEED = 42


random.seed(
    SEED
)


np.random.seed(
    SEED
)


# =============================================================================
# NUMBER OF FEATURES SHOWN
# =============================================================================

SHAP_MAX_DISPLAY = 15

LIME_MAX_DISPLAY = 15


# =============================================================================
# LIME CONFIGURATION
#
# Same settings as previous XAI experiment
# =============================================================================

LIME_BACKGROUND_MAX_SAMPLES = 6000

LIME_NUM_FEATURES = 15

LIME_NUM_SAMPLES = 3000


# =============================================================================
# WHICH CASES TO GENERATE
#
# True means generate that case category
# =============================================================================

GENERATE_CORRECT_CASES = True

GENERATE_MISCLASSIFIED_CASES = True


# =============================================================================
# VISUAL SETTINGS
# =============================================================================

FIGURE_DPI = 400


LIGHTGBM_COLOR = "#2563EB"

XGBOOST_COLOR = "#F97316"


POSITIVE_COLOR = "#10B981"

NEGATIVE_COLOR = "#EF4444"


BASE_COLOR = "#64748B"

FINAL_COLOR = "#7C3AED"

CONNECTOR_COLOR = "#94A3B8"

GRID_COLOR = "#CBD5E1"

TEXT_COLOR = "#0F172A"


plt.rcParams.update(
    {
        "figure.facecolor":
            "white",

        "axes.facecolor":
            "white",

        "axes.edgecolor":
            "#CBD5E1",

        "axes.labelcolor":
            TEXT_COLOR,

        "axes.titlecolor":
            TEXT_COLOR,

        "axes.titleweight":
            "bold",

        "axes.titlesize":
            14,

        "axes.labelsize":
            11,

        "xtick.color":
            "#334155",

        "ytick.color":
            "#334155",

        "font.size":
            10,

        "legend.frameon":
            False,

        "savefig.facecolor":
            "white"
    }
)


# =============================================================================
# PATHS
# =============================================================================

INPUT_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "Final_Model_Test/"
    "Cleaned_oN_Selected_Max.csv"
)


MODEL_ROOT = (
    INPUT_PATH.parent
)


LGB_SOURCE_DIR = (
    MODEL_ROOT
    / "LightGBM_Multiple_Split_Results"
    / "Train60_Test30_CV10"
)


XGB_SOURCE_DIR = (
    MODEL_ROOT
    / "XGBoost_Multiple_Split_Results"
    / "Train60_Test30_CV10"
)


XAI_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "Final_Two_Model_XAI"
)


SELECTED_CASES_PATH = (
    XAI_ROOT
    / "Selected_Local_XAI_Cases.csv"
)


# =============================================================================
# EXISTING SHAP RESULTS
# =============================================================================

LGB_SHAP_DIR = (
    XAI_ROOT
    / "SHAP"
    / "LightGBM"
)


XGB_SHAP_DIR = (
    XAI_ROOT
    / "SHAP"
    / "XGBoost"
)


# =============================================================================
# NEW WATERFALL OUTPUT
# =============================================================================

WATERFALL_ROOT = (
    XAI_ROOT
    / "Waterfall_Diagrams"
)


SHAP_WATERFALL_ROOT = (
    WATERFALL_ROOT
    / "SHAP_Waterfall"
)


LIME_WATERFALL_ROOT = (
    WATERFALL_ROOT
    / "LIME_Waterfall"
)


COMBINED_ROOT = (
    WATERFALL_ROOT
    / "SHAP_LIME_Comparison"
)


for directory in [
    WATERFALL_ROOT,
    SHAP_WATERFALL_ROOT,
    LIME_WATERFALL_ROOT,
    COMBINED_ROOT
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# =============================================================================
# MODEL SPECIFIC OUTPUT
# =============================================================================

for model_name in [
    "LightGBM",
    "XGBoost"
]:

    (
        SHAP_WATERFALL_ROOT
        / model_name
    ).mkdir(
        parents=True,
        exist_ok=True
    )


    (
        LIME_WATERFALL_ROOT
        / model_name
    ).mkdir(
        parents=True,
        exist_ok=True
    )


    (
        COMBINED_ROOT
        / model_name
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# =============================================================================
# HELPERS
# =============================================================================

def normalize_person_id(
    value
):

    text = str(
        value
    ).strip()


    try:

        number = float(
            text
        )


        if (
            np.isfinite(
                number
            )
            and
            number.is_integer()
        ):

            return str(
                int(
                    number
                )
            )


    except Exception:

        pass


    return text


# =============================================================================
# SAFE FILE NAME
# =============================================================================

def safe_filename(
    text
):

    text = str(
        text
    )


    text = re.sub(
        r"[^A-Za-z0-9_.]+",
        "_",
        text
    )


    return text.strip(
        "_"
    )


# =============================================================================
# SHORT FEATURE NAME
# =============================================================================

def short_label(
    text,
    width=48
):

    text = str(
        text
    )


    if len(
        text
    ) <= width:

        return text


    return (
        text[
            :width - 3
        ]
        + "..."
    )


# =============================================================================
# SAVE FIGURE
# =============================================================================

def save_figure(
    figure,
    output_path
):

    output_path = Path(
        output_path
    )


    figure.savefig(
        str(
            output_path
        )
        + ".png",
        dpi=
            FIGURE_DPI,
        bbox_inches=
            "tight"
    )


    figure.savefig(
        str(
            output_path
        )
        + ".svg",
        bbox_inches=
            "tight"
    )


    plt.close(
        figure
    )


# =============================================================================
# CLASS BALANCED SAMPLING
# =============================================================================

def class_balanced_indices(
    labels,
    maximum_samples,
    seed
):

    labels = np.asarray(
        labels
    )


    total_rows = len(
        labels
    )


    if (
        maximum_samples is None
        or
        maximum_samples
        >= total_rows
    ):

        return np.arange(
            total_rows,
            dtype=np.int64
        )


    rng = np.random.default_rng(
        seed
    )


    classes = np.unique(
        labels
    )


    quota = (
        maximum_samples
        // len(
            classes
        )
    )


    remainder = (
        maximum_samples
        % len(
            classes
        )
    )


    selected = []


    for class_position, class_value in enumerate(
        classes
    ):

        available = np.where(
            labels
            == class_value
        )[0]


        current_quota = (
            quota
            +
            (
                1
                if class_position
                < remainder
                else 0
            )
        )


        current_quota = min(
            current_quota,
            len(
                available
            )
        )


        chosen = rng.choice(
            available,
            size=
                current_quota,
            replace=
                False
        )


        selected.extend(
            chosen.tolist()
        )


    selected = np.asarray(
        selected,
        dtype=np.int64
    )


    if len(
        selected
    ) < maximum_samples:

        all_indices = np.arange(
            total_rows
        )


        unused = np.setdiff1d(
            all_indices,
            selected
        )


        additional = min(
            maximum_samples
            - len(
                selected
            ),
            len(
                unused
            )
        )


        if additional > 0:

            extra = rng.choice(
                unused,
                size=
                    additional,
                replace=
                    False
            )


            selected = np.concatenate(
                [
                    selected,
                    extra
                ]
            )


    rng.shuffle(
        selected
    )


    return selected


# =============================================================================
# CHECK REQUIRED FILES
# =============================================================================

required_files = [

    INPUT_PATH,

    SELECTED_CASES_PATH,

    LGB_SOURCE_DIR
    / "best_lightgbm_model.txt",

    LGB_SOURCE_DIR
    / "standard_scaler.joblib",

    LGB_SOURCE_DIR
    / "label_encoder.joblib",

    LGB_SOURCE_DIR
    / "participant_split.json",

    XGB_SOURCE_DIR
    / "xgboost_model.joblib",

    XGB_SOURCE_DIR
    / "standard_scaler.joblib",

    XGB_SOURCE_DIR
    / "label_encoder.joblib",

    XGB_SOURCE_DIR
    / "participant_split.json",

    LGB_SHAP_DIR
    / "shap_values_checkpoint.npz",

    XGB_SHAP_DIR
    / "shap_values_checkpoint.npz"
]


missing_files = [

    str(
        path
    )

    for path
    in required_files

    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(

        "Required files are missing:\n\n"

        + "\n".join(
            missing_files
        )
    )


# =============================================================================
# HEADER
# =============================================================================

print(
    "=" * 100
)


print(
    "SHAP AND LIME WATERFALL DIAGRAM GENERATION"
)


print(
    "=" * 100
)


print(
    "Output:"
)


print(
    WATERFALL_ROOT
)


print(
    "=" * 100
)


# =============================================================================
# LOAD ARTIFACTS
# =============================================================================

lgb_scaler = joblib.load(
    LGB_SOURCE_DIR
    / "standard_scaler.joblib"
)


xgb_scaler = joblib.load(
    XGB_SOURCE_DIR
    / "standard_scaler.joblib"
)


label_encoder = joblib.load(
    LGB_SOURCE_DIR
    / "label_encoder.joblib"
)


xgb_label_encoder = joblib.load(
    XGB_SOURCE_DIR
    / "label_encoder.joblib"
)


if not np.array_equal(
    label_encoder.classes_,
    xgb_label_encoder.classes_
):

    raise RuntimeError(
        "Class encoders are different."
    )


CLASS_NAMES = list(
    label_encoder.classes_
)


NUM_CLASSES = len(
    CLASS_NAMES
)


# =============================================================================
# PARTICIPANT SPLIT
# =============================================================================

with open(
    LGB_SOURCE_DIR
    / "participant_split.json",
    "r",
    encoding="utf8"
) as file:

    participant_split = json.load(
        file
    )


TEST_PERSONS = np.asarray(
    [

        normalize_person_id(
            value
        )

        for value
        in participant_split[
            "testing"
        ]
    ],
    dtype=str
)


TRAIN_PERSONS = np.asarray(
    [

        normalize_person_id(
            value
        )

        for value
        in participant_split[
            "training"
        ]
    ],
    dtype=str
)


# =============================================================================
# LOAD DATASET
# =============================================================================

print()


print(
    "Loading dataset..."
)


df = pd.read_csv(
    INPUT_PATH,
    low_memory=False
)


METADATA_COLUMNS = [

    "PersonSrial",

    "Window",

    "Window_Start",

    "Window_End",

    "Window_Samples"
]


TARGET_COLUMN = "Class"


FEATURE_COLUMNS = [

    column

    for column
    in df.columns

    if column not in METADATA_COLUMNS

    and column
    != TARGET_COLUMN
]


NUMBER_FEATURES = len(
    FEATURE_COLUMNS
)


X_all = (

    df[
        FEATURE_COLUMNS
    ]

    .apply(
        pd.to_numeric,
        errors="coerce"
    )

    .to_numpy(
        dtype=np.float32
    )
)


if not np.isfinite(
    X_all
).all():

    raise RuntimeError(
        "Feature matrix contains invalid values."
    )


y_all = label_encoder.transform(
    df[
        TARGET_COLUMN
    ].astype(
        str
    )
).astype(
    np.int32
)


groups_all = np.asarray(
    [

        normalize_person_id(
            value
        )

        for value
        in df[
            "PersonSrial"
        ].to_numpy()
    ],
    dtype=str
)


# =============================================================================
# TRAIN AND TEST MATRICES
# =============================================================================

train_mask = np.isin(
    groups_all,
    TRAIN_PERSONS
)


test_mask = np.isin(
    groups_all,
    TEST_PERSONS
)


X_train_raw = np.ascontiguousarray(
    X_all[
        train_mask
    ],
    dtype=np.float32
)


y_train = np.ascontiguousarray(
    y_all[
        train_mask
    ],
    dtype=np.int32
)


X_test_raw = np.ascontiguousarray(
    X_all[
        test_mask
    ],
    dtype=np.float32
)


y_test = np.ascontiguousarray(
    y_all[
        test_mask
    ],
    dtype=np.int32
)


g_test = np.asarray(
    groups_all[
        test_mask
    ],
    dtype=str
)


print(
    "Test rows:",
    len(
        X_test_raw
    )
)


print(
    "Features:",
    NUMBER_FEATURES
)


print(
    "Classes:",
    CLASS_NAMES
)


# =============================================================================
# RELEASE LARGE ORIGINAL MATRICES
# =============================================================================

del df

del X_all

del y_all

del groups_all

del train_mask

del test_mask


gc.collect()


# =============================================================================
# LOAD FINAL MODELS
# =============================================================================

lgb_model = lgb.Booster(
    model_file=str(
        LGB_SOURCE_DIR
        / "best_lightgbm_model.txt"
    )
)


xgb_model = joblib.load(
    XGB_SOURCE_DIR
    / "xgboost_model.joblib"
)


# =============================================================================
# MODEL PROBABILITIES
# =============================================================================

X_test_lgb = lgb_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False
)


X_test_xgb = xgb_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False
)


lgb_probability = np.asarray(
    lgb_model.predict(
        X_test_lgb
    ),
    dtype=np.float64
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_test_xgb
    ),
    dtype=np.float64
)


lgb_prediction = np.argmax(
    lgb_probability,
    axis=1
).astype(
    np.int32
)


xgb_prediction = np.argmax(
    xgb_probability,
    axis=1
).astype(
    np.int32
)


# =============================================================================
# LOAD SELECTED CASES
# =============================================================================

selected_cases = pd.read_csv(
    SELECTED_CASES_PATH,
    low_memory=False
)


# =============================================================================
# FILTER DESIRED CASE TYPES
# =============================================================================

case_type_mask = np.zeros(
    len(
        selected_cases
    ),
    dtype=bool
)


if GENERATE_CORRECT_CASES:

    case_type_mask |= (

        selected_cases[
            "Case_Type"
        ]
        == "Correct_High_Confidence"
    )


if GENERATE_MISCLASSIFIED_CASES:

    case_type_mask |= (

        selected_cases[
            "Case_Type"
        ]
        == "Misclassified_High_Confidence"
    )


selected_cases = selected_cases[
    case_type_mask
].copy()


# =============================================================================
# LOAD SHAP CHECKPOINT
# =============================================================================

def load_shap_checkpoint(
    checkpoint_path
):

    checkpoint = np.load(
        checkpoint_path
    )


    values = np.asarray(
        checkpoint[
            "shap_values"
        ],
        dtype=np.float64
    )


    base_values = np.asarray(
        checkpoint[
            "base_values"
        ],
        dtype=np.float64
    )


    indices = np.asarray(
        checkpoint[
            "indices"
        ],
        dtype=np.int64
    )


    return (
        values,
        base_values,
        indices
    )


(
    lgb_shap_values,
    lgb_shap_base,
    lgb_shap_indices

) = load_shap_checkpoint(

    LGB_SHAP_DIR
    / "shap_values_checkpoint.npz"
)


(
    xgb_shap_values,
    xgb_shap_base,
    xgb_shap_indices

) = load_shap_checkpoint(

    XGB_SHAP_DIR
    / "shap_values_checkpoint.npz"
)


print(
    "LightGBM SHAP:",
    lgb_shap_values.shape
)


print(
    "XGBoost SHAP:",
    xgb_shap_values.shape
)


# =============================================================================
# INDEX LOOKUPS
# =============================================================================

lgb_shap_lookup = {

    int(
        sample_index
    ):
        position

    for position, sample_index
    in enumerate(
        lgb_shap_indices
    )
}


xgb_shap_lookup = {

    int(
        sample_index
    ):
        position

    for position, sample_index
    in enumerate(
        xgb_shap_indices
    )
}


# =============================================================================
# CUSTOM SHAP WATERFALL
# =============================================================================

def create_shap_waterfall(
    model_name,
    sample_index,
    true_class,
    predicted_class,
    confidence,
    person_id,
    shap_vector,
    base_value,
    raw_vector,
    output_directory
):

    # =========================================================================
    # SELECT TOP FEATURES
    # =========================================================================

    order = np.argsort(
        np.abs(
            shap_vector
        )
    )[
        ::-1
    ]


    top_indices = order[
        :SHAP_MAX_DISPLAY
    ]


    remaining_indices = order[
        SHAP_MAX_DISPLAY:
    ]


    contributions = []


    for feature_index in top_indices:

        contributions.append(
            {
                "Feature":
                    FEATURE_COLUMNS[
                        feature_index
                    ],

                "Feature_Value":
                    float(
                        raw_vector[
                            feature_index
                        ]
                    ),

                "Contribution":
                    float(
                        shap_vector[
                            feature_index
                        ]
                    ),

                "Type":
                    "Feature"
            }
        )


    # =========================================================================
    # COMBINE REMAINING FEATURES
    # =========================================================================

    if len(
        remaining_indices
    ) > 0:

        other_value = float(
            np.sum(
                shap_vector[
                    remaining_indices
                ]
            )
        )


        contributions.append(
            {
                "Feature":
                    (
                        f"Other "
                        f"{len(remaining_indices)} "
                        f"features"
                    ),

                "Feature_Value":
                    np.nan,

                "Contribution":
                    other_value,

                "Type":
                    "Combined"
            }
        )


    # =========================================================================
    # CONTRIBUTION TABLE
    # =========================================================================

    contribution_table = pd.DataFrame(
        contributions
    )


    # =========================================================================
    # CUMULATIVE VALUES
    # =========================================================================

    current_value = float(
        base_value
    )


    starts = []

    ends = []


    for contribution in contribution_table[
        "Contribution"
    ]:

        starts.append(
            current_value
        )


        current_value += float(
            contribution
        )


        ends.append(
            current_value
        )


    contribution_table[
        "Start_Value"
    ] = starts


    contribution_table[
        "End_Value"
    ] = ends


    final_value = float(
        base_value
        + np.sum(
            shap_vector
        )
    )


    # =========================================================================
    # LABELS
    # =========================================================================

    labels = []


    for _, row in contribution_table.iterrows():

        if row[
            "Type"
        ] == "Feature":

            labels.append(
                (
                    f"{short_label(row['Feature'], 38)}"
                    f" = "
                    f"{row['Feature_Value']:.4g}"
                )
            )


        else:

            labels.append(
                row[
                    "Feature"
                ]
            )


    # =========================================================================
    # FIGURE
    # =========================================================================

    number_rows = len(
        contribution_table
    )


    figure_height = max(
        7.5,
        number_rows
        * 0.55
        + 2.6
    )


    fig, ax = plt.subplots(
        figsize=(
            13,
            figure_height
        )
    )


    y_positions = np.arange(
        number_rows
    )


    # =========================================================================
    # DRAW CONTRIBUTION BARS
    # =========================================================================

    for row_index, row in contribution_table.iterrows():

        contribution = float(
            row[
                "Contribution"
            ]
        )


        start = float(
            row[
                "Start_Value"
            ]
        )


        color = (

            POSITIVE_COLOR

            if contribution >= 0

            else NEGATIVE_COLOR
        )


        ax.barh(
            row_index,
            contribution,
            left=start,
            height=0.62,
            color=color,
            alpha=0.92,
            edgecolor="white",
            linewidth=0.8
        )


        end = float(
            row[
                "End_Value"
            ]
        )


        ax.text(
            end,
            row_index,
            (
                f"  {contribution:+.4f}"
                if contribution >= 0
                else
                f"  {contribution:.4f}"
            ),
            va="center",
            ha=(
                "left"
                if contribution >= 0
                else "right"
            ),
            fontsize=9,
            fontweight="bold",
            color=color
        )


        # =====================================================================
        # CONNECTOR
        # =====================================================================

        if row_index < (
            number_rows
            - 1
        ):

            ax.plot(
                [
                    end,
                    end
                ],
                [
                    row_index
                    + 0.31,
                    row_index
                    + 0.69
                ],
                color=
                    CONNECTOR_COLOR,
                linewidth=
                    1.0,
                linestyle=
                    ":"
            )


    # =========================================================================
    # BASE AND FINAL LINES
    # =========================================================================

    ax.axvline(
        base_value,
        color=
            BASE_COLOR,
        linewidth=
            1.6,
        linestyle=
            "--",
        label=
            (
                f"Base value = "
                f"{base_value:.4f}"
            )
    )


    ax.axvline(
        final_value,
        color=
            FINAL_COLOR,
        linewidth=
            2.0,
        linestyle=
            "-",
        label=
            (
                f"Final SHAP score = "
                f"{final_value:.4f}"
            )
    )


    # =========================================================================
    # AXIS
    # =========================================================================

    ax.set_yticks(
        y_positions
    )


    ax.set_yticklabels(
        labels
    )


    ax.invert_yaxis()


    ax.set_xlabel(
        (
            "Cumulative SHAP contribution "
            "on tree model output scale"
        )
    )


    ax.set_ylabel(
        "Feature"
    )


    ax.set_title(
        (
            f"{model_name} SHAP Waterfall\n"
            f"True: {true_class}    "
            f"Predicted: {predicted_class}    "
            f"Probability: {confidence:.4f}    "
            f"Participant: {person_id}"
        )
    )


    ax.grid(
        axis="x",
        color=
            GRID_COLOR,
        alpha=
            0.35
    )


    ax.legend(
        loc="best"
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


    fig.tight_layout()


    # =========================================================================
    # SAVE
    # =========================================================================

    output_name = safe_filename(
        (
            f"Sample_{sample_index}_"
            f"True_{true_class}_"
            f"Pred_{predicted_class}_"
            f"SHAP_Waterfall"
        )
    )


    save_figure(
        fig,
        output_directory
        / output_name
    )


    contribution_table.to_csv(
        output_directory
        / (
            output_name
            + "_Contributions.csv"
        ),
        index=False
    )


    # =========================================================================
    # ALSO GENERATE NATIVE SHAP WATERFALL
    # =========================================================================

    try:

        explanation = shap.Explanation(
            values=
                shap_vector,
            base_values=
                base_value,
            data=
                raw_vector,
            feature_names=
                FEATURE_COLUMNS
        )


        shap.plots.waterfall(
            explanation,
            max_display=
                SHAP_MAX_DISPLAY,
            show=
                False
        )


        native_figure = plt.gcf()


        native_figure.suptitle(
            (
                f"{model_name} Native SHAP Waterfall\n"
                f"True: {true_class}    "
                f"Predicted: {predicted_class}"
            ),
            fontsize=13,
            fontweight="bold",
            y=1.02
        )


        native_figure.tight_layout()


        save_figure(
            native_figure,
            output_directory
            / (
                output_name
                + "_Native"
            )
        )


    except Exception as error:

        print(
            "Native SHAP waterfall warning:",
            error
        )


    return {
        "Baseline":
            float(
                base_value
            ),

        "Final_SHAP_Score":
            final_value,

        "Top_Feature":
            contribution_table.iloc[
                0
            ][
                "Feature"
            ]
    }


# =============================================================================
# PREPARE LIME BACKGROUND
# =============================================================================

lime_background_indices = class_balanced_indices(
    y_train,
    LIME_BACKGROUND_MAX_SAMPLES,
    SEED + 500
)


X_lime_background = np.ascontiguousarray(
    X_train_raw[
        lime_background_indices
    ],
    dtype=np.float32
)


# =============================================================================
# MODEL PREDICTION WRAPPERS FOR LIME
# =============================================================================

def predict_lgb_raw(
    raw_features
):

    raw_features = np.asarray(
        raw_features,
        dtype=np.float32
    )


    scaled = lgb_scaler.transform(
        raw_features
    ).astype(
        np.float32,
        copy=False
    )


    return np.asarray(
        lgb_model.predict(
            scaled
        ),
        dtype=np.float64
    )


def predict_xgb_raw(
    raw_features
):

    raw_features = np.asarray(
        raw_features,
        dtype=np.float32
    )


    scaled = xgb_scaler.transform(
        raw_features
    ).astype(
        np.float32,
        copy=False
    )


    return np.asarray(
        xgb_model.predict_proba(
            scaled
        ),
        dtype=np.float64
    )


# =============================================================================
# CREATE LIME EXPLAINERS
#
# Same seeds as original XAI script
# =============================================================================

lgb_lime_explainer = LimeTabularExplainer(

    training_data=
        X_lime_background,

    feature_names=
        FEATURE_COLUMNS,

    class_names=
        CLASS_NAMES,

    mode=
        "classification",

    discretize_continuous=
        True,

    discretizer=
        "quartile",

    sample_around_instance=
        True,

    random_state=
        SEED + 100
)


xgb_lime_explainer = LimeTabularExplainer(

    training_data=
        X_lime_background,

    feature_names=
        FEATURE_COLUMNS,

    class_names=
        CLASS_NAMES,

    mode=
        "classification",

    discretize_continuous=
        True,

    discretizer=
        "quartile",

    sample_around_instance=
        True,

    random_state=
        SEED + 200
)


# =============================================================================
# CUSTOM LIME WATERFALL
# =============================================================================

def create_lime_waterfall(
    model_name,
    explainer,
    prediction_function,
    sample_index,
    true_class,
    predicted_class_index,
    predicted_class,
    confidence,
    person_id,
    output_directory
):

    print(
        (
            f"LIME waterfall | "
            f"{model_name} | "
            f"sample {sample_index}"
        )
    )


    # =========================================================================
    # CREATE LOCAL EXPLANATION
    # =========================================================================

    explanation = explainer.explain_instance(

        X_test_raw[
            sample_index
        ],

        prediction_function,

        labels=[
            predicted_class_index
        ],

        num_features=
            LIME_NUM_FEATURES,

        num_samples=
            LIME_NUM_SAMPLES
    )


    # =========================================================================
    # INTERCEPT
    # =========================================================================

    intercept_object = explanation.intercept


    if isinstance(
        intercept_object,
        dict
    ):

        intercept = float(
            intercept_object[
                predicted_class_index
            ]
        )


    else:

        intercept_array = np.asarray(
            intercept_object
        ).reshape(
            -1
        )


        if len(
            intercept_array
        ) == NUM_CLASSES:

            intercept = float(
                intercept_array[
                    predicted_class_index
                ]
            )


        else:

            intercept = float(
                intercept_array[
                    0
                ]
            )


    # =========================================================================
    # LIME FEATURE CONTRIBUTIONS
    # =========================================================================

    contribution_list = explanation.as_list(
        label=
            predicted_class_index
    )


    lime_table = pd.DataFrame(
        {

            "Feature_Condition":
                [
                    item[
                        0
                    ]

                    for item
                    in contribution_list
                ],

            "Contribution":
                [
                    float(
                        item[
                            1
                        ]
                    )

                    for item
                    in contribution_list
                ]
        }
    )


    lime_table[
        "Absolute_Contribution"
    ] = np.abs(
        lime_table[
            "Contribution"
        ]
    )


    lime_table = (

        lime_table

        .sort_values(
            "Absolute_Contribution",
            ascending=False
        )

        .head(
            LIME_MAX_DISPLAY
        )

        .reset_index(
            drop=True
        )
    )


    # =========================================================================
    # CALCULATE CUMULATIVE SURROGATE SCORE
    # =========================================================================

    current_value = intercept


    starts = []

    ends = []


    for contribution in lime_table[
        "Contribution"
    ]:

        starts.append(
            current_value
        )


        current_value += float(
            contribution
        )


        ends.append(
            current_value
        )


    lime_table[
        "Start_Value"
    ] = starts


    lime_table[
        "End_Value"
    ] = ends


    calculated_surrogate_score = float(
        current_value
    )


    # =========================================================================
    # LIME REPORTED LOCAL PREDICTION
    # =========================================================================

    try:

        reported_local_prediction = float(
            np.asarray(
                explanation.local_pred
            ).reshape(
                -1
            )[
                0
            ]
        )


    except Exception:

        reported_local_prediction = np.nan


    # =========================================================================
    # PLOT
    # =========================================================================

    number_rows = len(
        lime_table
    )


    figure_height = max(
        7.5,
        number_rows
        * 0.56
        + 2.8
    )


    fig, ax = plt.subplots(
        figsize=(
            13,
            figure_height
        )
    )


    y_positions = np.arange(
        number_rows
    )


    for row_index, row in lime_table.iterrows():

        contribution = float(
            row[
                "Contribution"
            ]
        )


        start = float(
            row[
                "Start_Value"
            ]
        )


        end = float(
            row[
                "End_Value"
            ]
        )


        color = (

            POSITIVE_COLOR

            if contribution >= 0

            else NEGATIVE_COLOR
        )


        ax.barh(
            row_index,
            contribution,
            left=start,
            height=0.62,
            color=color,
            alpha=0.92,
            edgecolor="white",
            linewidth=0.8
        )


        ax.text(
            end,
            row_index,
            (
                f"  {contribution:+.4f}"
                if contribution >= 0
                else
                f"  {contribution:.4f}"
            ),
            va="center",
            ha=(
                "left"
                if contribution >= 0
                else "right"
            ),
            fontsize=9,
            fontweight="bold",
            color=color
        )


        if row_index < (
            number_rows
            - 1
        ):

            ax.plot(
                [
                    end,
                    end
                ],
                [
                    row_index
                    + 0.31,
                    row_index
                    + 0.69
                ],
                color=
                    CONNECTOR_COLOR,
                linestyle=
                    ":",
                linewidth=
                    1.0
            )


    # =========================================================================
    # INTERCEPT
    # =========================================================================

    ax.axvline(
        intercept,
        color=
            BASE_COLOR,
        linestyle=
            "--",
        linewidth=
            1.7,
        label=
            (
                f"LIME intercept = "
                f"{intercept:.4f}"
            )
    )


    # =========================================================================
    # FINAL SURROGATE VALUE
    # =========================================================================

    ax.axvline(
        calculated_surrogate_score,
        color=
            FINAL_COLOR,
        linestyle=
            "-",
        linewidth=
            2.0,
        label=
            (
                f"Local surrogate = "
                f"{calculated_surrogate_score:.4f}"
            )
    )


    # =========================================================================
    # LABELS
    # =========================================================================

    ax.set_yticks(
        y_positions
    )


    ax.set_yticklabels(
        [

            short_label(
                value,
                width=60
            )

            for value
            in lime_table[
                "Feature_Condition"
            ]
        ]
    )


    ax.invert_yaxis()


    ax.set_xlabel(
        "Cumulative LIME local surrogate score"
    )


    ax.set_ylabel(
        "Feature condition"
    )


    ax.set_title(
        (
            f"{model_name} LIME Waterfall\n"
            f"True: {true_class}    "
            f"Predicted: {predicted_class}    "
            f"Model probability: {confidence:.4f}    "
            f"Participant: {person_id}"
        )
    )


    ax.grid(
        axis="x",
        color=
            GRID_COLOR,
        alpha=
            0.35
    )


    ax.legend(
        loc="best"
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


    fig.tight_layout()


    output_name = safe_filename(
        (
            f"Sample_{sample_index}_"
            f"True_{true_class}_"
            f"Pred_{predicted_class}_"
            f"LIME_Waterfall"
        )
    )


    save_figure(
        fig,
        output_directory
        / output_name
    )


    # =========================================================================
    # SAVE CONTRIBUTIONS
    # =========================================================================

    lime_table.to_csv(
        output_directory
        / (
            output_name
            + "_Contributions.csv"
        ),
        index=False
    )


    # =========================================================================
    # SAVE METADATA
    # =========================================================================

    metadata = {

        "Model":
            model_name,

        "Sample_Index":
            int(
                sample_index
            ),

        "Participant":
            str(
                person_id
            ),

        "True_Class":
            true_class,

        "Predicted_Class":
            predicted_class,

        "Model_Probability":
            float(
                confidence
            ),

        "LIME_Intercept":
            float(
                intercept
            ),

        "Calculated_Local_Surrogate":
            float(
                calculated_surrogate_score
            ),

        "LIME_Reported_Local_Prediction":
            (
                float(
                    reported_local_prediction
                )

                if np.isfinite(
                    reported_local_prediction
                )

                else None
            )
    }


    with open(
        output_directory
        / (
            output_name
            + "_Metadata.json"
        ),
        "w",
        encoding="utf8"
    ) as file:

        json.dump(
            metadata,
            file,
            indent=4
        )


    return {

        "LIME_Intercept":
            intercept,

        "LIME_Surrogate":
            calculated_surrogate_score,

        "LIME_Reported_Local_Prediction":
            reported_local_prediction,

        "Top_LIME_Feature":
            lime_table.iloc[
                0
            ][
                "Feature_Condition"
            ]
    }


# =============================================================================
# RUN WATERFALL GENERATION
# =============================================================================

summary_records = []


for _, case in selected_cases.iterrows():

    model_name = str(
        case[
            "Model"
        ]
    )


    sample_index = int(
        case[
            "Sample_Index"
        ]
    )


    case_type = str(
        case[
            "Case_Type"
        ]
    )


    true_class = str(
        case[
            "True_Class"
        ]
    )


    predicted_class = str(
        case[
            "Predicted_Class"
        ]
    )


    predicted_class_index = int(
        case[
            "Predicted_Class_Index"
        ]
    )


    person_id = str(
        case[
            "PersonSrial"
        ]
    )


    confidence = float(
        case[
            "Prediction_Confidence"
        ]
    )


    raw_vector = X_test_raw[
        sample_index
    ]


    print()


    print(
        "=" * 100
    )


    print(
        (
            f"{model_name} | "
            f"{case_type} | "
            f"sample {sample_index}"
        )
    )


    print(
        "=" * 100
    )


    # =========================================================================
    # LIGHTGBM
    # =========================================================================

    if model_name == "LightGBM":

        if sample_index not in lgb_shap_lookup:

            print(
                "SHAP sample not available:",
                sample_index
            )


            continue


        shap_position = lgb_shap_lookup[
            sample_index
        ]


        shap_vector = lgb_shap_values[
            shap_position,
            :,
            predicted_class_index
        ]


        base_value = float(
            lgb_shap_base[
                predicted_class_index
            ]
        )


        shap_output_dir = (
            SHAP_WATERFALL_ROOT
            / "LightGBM"
        )


        lime_output_dir = (
            LIME_WATERFALL_ROOT
            / "LightGBM"
        )


        shap_result = create_shap_waterfall(

            model_name=
                model_name,

            sample_index=
                sample_index,

            true_class=
                true_class,

            predicted_class=
                predicted_class,

            confidence=
                confidence,

            person_id=
                person_id,

            shap_vector=
                shap_vector,

            base_value=
                base_value,

            raw_vector=
                raw_vector,

            output_directory=
                shap_output_dir
        )


        lime_result = create_lime_waterfall(

            model_name=
                model_name,

            explainer=
                lgb_lime_explainer,

            prediction_function=
                predict_lgb_raw,

            sample_index=
                sample_index,

            true_class=
                true_class,

            predicted_class_index=
                predicted_class_index,

            predicted_class=
                predicted_class,

            confidence=
                confidence,

            person_id=
                person_id,

            output_directory=
                lime_output_dir
        )


    # =========================================================================
    # XGBOOST
    # =========================================================================

    elif model_name == "XGBoost":

        if sample_index not in xgb_shap_lookup:

            print(
                "SHAP sample not available:",
                sample_index
            )


            continue


        shap_position = xgb_shap_lookup[
            sample_index
        ]


        shap_vector = xgb_shap_values[
            shap_position,
            :,
            predicted_class_index
        ]


        base_value = float(
            xgb_shap_base[
                predicted_class_index
            ]
        )


        shap_output_dir = (
            SHAP_WATERFALL_ROOT
            / "XGBoost"
        )


        lime_output_dir = (
            LIME_WATERFALL_ROOT
            / "XGBoost"
        )


        shap_result = create_shap_waterfall(

            model_name=
                model_name,

            sample_index=
                sample_index,

            true_class=
                true_class,

            predicted_class=
                predicted_class,

            confidence=
                confidence,

            person_id=
                person_id,

            shap_vector=
                shap_vector,

            base_value=
                base_value,

            raw_vector=
                raw_vector,

            output_directory=
                shap_output_dir
        )


        lime_result = create_lime_waterfall(

            model_name=
                model_name,

            explainer=
                xgb_lime_explainer,

            prediction_function=
                predict_xgb_raw,

            sample_index=
                sample_index,

            true_class=
                true_class,

            predicted_class_index=
                predicted_class_index,

            predicted_class=
                predicted_class,

            confidence=
                confidence,

            person_id=
                person_id,

            output_directory=
                lime_output_dir
        )


    else:

        print(
            "Unknown model:",
            model_name
        )


        continue


    # =========================================================================
    # SUMMARY RECORD
    # =========================================================================

    summary_records.append(
        {

            "Model":
                model_name,

            "Sample_Index":
                sample_index,

            "PersonSrial":
                person_id,

            "Case_Type":
                case_type,

            "True_Class":
                true_class,

            "Predicted_Class":
                predicted_class,

            "Prediction_Confidence":
                confidence,

            "SHAP_Base_Value":
                shap_result[
                    "Baseline"
                ],

            "SHAP_Final_Score":
                shap_result[
                    "Final_SHAP_Score"
                ],

            "SHAP_Top_Feature":
                shap_result[
                    "Top_Feature"
                ],

            "LIME_Intercept":
                lime_result[
                    "LIME_Intercept"
                ],

            "LIME_Surrogate":
                lime_result[
                    "LIME_Surrogate"
                ],

            "LIME_Reported_Local_Prediction":
                lime_result[
                    "LIME_Reported_Local_Prediction"
                ],

            "LIME_Top_Feature":
                lime_result[
                    "Top_LIME_Feature"
                ]
        }
    )


    gc.collect()


# =============================================================================
# SAVE SUMMARY
# =============================================================================

summary_table = pd.DataFrame(
    summary_records
)


summary_table.to_csv(
    WATERFALL_ROOT
    / "Waterfall_Diagram_Summary.csv",
    index=False
)


# =============================================================================
# CREATE COMPACT MODEL SUMMARY
# =============================================================================

if len(
    summary_table
) > 0:

    model_summary = (

        summary_table

        .groupby(
            [
                "Model",
                "Case_Type"
            ],
            as_index=False
        )

        .agg(

            Number_of_Waterfalls=(
                "Sample_Index",
                "count"
            ),

            Mean_Prediction_Confidence=(
                "Prediction_Confidence",
                "mean"
            )
        )
    )


    model_summary.to_csv(
        WATERFALL_ROOT
        / "Waterfall_Model_Summary.csv",
        index=False
    )


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print()


print(
    "#" * 100
)


print(
    "SHAP AND LIME WATERFALL DIAGRAMS COMPLETED"
)


print(
    "#" * 100
)


print()


print(
    "Total cases:",
    len(
        summary_table
    )
)


print()


print(
    "SHAP waterfall results:"
)


print(
    SHAP_WATERFALL_ROOT
)


print()


print(
    "LIME waterfall results:"
)


print(
    LIME_WATERFALL_ROOT
)


print()


print(
    "Summary:"
)


print(
    WATERFALL_ROOT
    / "Waterfall_Diagram_Summary.csv"
)


print()


print(
    "ALL WATERFALL RESULTS SAVED IN:"
)


print(
    WATERFALL_ROOT
)


print(
    "#" * 100
)

/home/rakib/Desktop/ml_apply/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP AND LIME WATERFALL DIAGRAM GENERATION
Output:
/home/rakib/Desktop/ml_apply/Final_Two_Model_XAI/Waterfall_Diagrams

Loading dataset...
Test rows: 45678
Features: 232
Classes: ['class_lecture', 'music', 'normal', 'reading']
LightGBM SHAP: (6013, 232, 4)
XGBoost SHAP: (6013, 232, 4)

LightGBM | Correct_High_Confidence | sample 35735
LIME waterfall | LightGBM | sample 35735

LightGBM | Misclassified_High_Confidence | sample 10485
LIME waterfall | LightGBM | sample 10485

LightGBM | Correct_High_Confidence | sample 34996
LIME waterfall | LightGBM | sample 34996

LightGBM | Misclassified_High_Confidence | sample 36636
LIME waterfall | LightGBM | sample 36636

LightGBM | Correct_High_Confidence | sample 44943
LIME waterfall | LightGBM | sample 44943

LightGBM | Misclassified_High_Confidence | sample 18732
LIME waterfall | LightGBM | sample 18732

LightGBM | Correct_High_Confidence | sample 32818
LIME waterfall | LightGBM | sample 32818

LightGBM | Misclassified_High_Confidence | sample 6

**END OF ALL CODING**